# Networking Fundamentals — an active-learning lab in C++

This notebook teaches networking from first principles: TCP/IP, the OSI 7-layer model, and the
devices in between (NIC, modem, switch, router, access point, the home "router" box).
You will learn by **doing**: short explanations, brief questions, and C++ exercises that
operate on **real bytes** laid out exactly as they travel on a real network.

| Part | Topic | Time |
|---|---|---|
| 0 | Using this notebook | 10 min |
| 1 | Bytes and bits primer | 35 min |
| 2 | What a network is | 30 min |
| 3 | Layering | 25 min |
| 4 | Physical layer and hardware | 30 min |
| 5 | Link layer: Ethernet | 35 min |
| 6 | Network layer: IPv4 | 45 min |
| 7 | Home and ISP infrastructure services | 35 min |
| 8 | Transport layer: UDP and TCP | 45 min |
| 9 | Sockets in practice (loopback only) | 35 min |
| 10 | Capstone: dissect a real frame | 30 min |

Total: about 6 hours.

# Part 0 — Using this notebook

## 0.1 How this notebook works

A notebook is a list of **cells**. *Markdown* cells hold text; *code* cells hold C++ that the
**kernel** (the program running your code, here *xeus-cling*, a C++ interpreter) executes
when you press **Shift+Enter**. Cells run in the order *you* run them, and everything defined
in one cell stays visible to later cells.

Every concept follows the same rhythm:

```
explanation (≤ ~150 words, diagram if useful)
    └─► 1–2 short questions ──► collapsed solution after each
    └─► coding exercise ──► test cell ──► collapsed solution
```

* **Questions** are tagged *Why?*, *Predict the output*, *Compute*, or *Code reading*.
  Answer in your head or on paper **before** expanding the solution.
* **Exercises** give you a *stub*: a function that compiles but returns a placeholder
  (`return 0;`, `return {};`). Replace the body, run the cell, then run the test cell below it.
* **Solutions** are collapsed. Click **▶ Solution** to expand one.

**Q0.1** *(Concept)* — You run the notebook top to bottom without solving anything. Should you expect errors, ❌ marks, or both?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Only ❌ marks. Every stub compiles and returns a placeholder value, so every cell runs;
the tests simply report that the placeholder is wrong. A *cell error* (red traceback)
always means something else went wrong, typically the redefinition problem covered in §0.3.

</details>

## 0.2 The setup cell

The next cell must run **first**. It contains:

* every `#include` used anywhere in the notebook;
* `CHECK(expr)` prints ✅ or ❌ followed by the expression's text, and counts passes and failures;
* `CHECK_EQ(a, b)` does the same for `a == b`, and on failure prints both values ("got" and "expected");
* `CHECK_NEAR(a, b)` compares two floating-point numbers with a small relative tolerance;
* `hexdump(p, n)` prints raw bytes (see §0.4);
* `print_score()` and `reset_score()`.

None of these helpers ever aborts the kernel: a failed check prints and moves on.

In [1]:
// ============================ SETUP CELL ==================================
// Run this cell once, first. It contains every #include used anywhere in the
// notebook plus the test helpers (CHECK, CHECK_EQ, CHECK_NEAR) and hexdump().
#include <iostream>
#include <iomanip>
#include <cstdint>
#include <cstring>
#include <string>
#include <vector>
#include <array>
#include <map>
#include <unordered_map>
#include <sstream>
#include <bitset>
#include <optional>
#include <algorithm>
#include <thread>
#include <chrono>
#include <set>
#include <cmath>
#include <type_traits>
#include <csignal>
#include <sys/socket.h>
#include <netinet/in.h>
#include <arpa/inet.h>
#include <unistd.h>
#include <fcntl.h>
#include <sys/time.h>
#include <cerrno>

#ifndef MSG_NOSIGNAL          // Linux has it; macOS does not (SIGPIPE is ignored below anyway)
#define MSG_NOSIGNAL 0
#endif

namespace nbtest {

// Writing to a TCP connection the peer has closed raises SIGPIPE, which would kill the kernel.
// Ignore it process-wide: send() then simply fails with errno EPIPE.
const bool sigpipe_ignored = (std::signal(SIGPIPE, SIG_IGN) != SIG_ERR);

int g_pass = 0;   // number of CHECKs that passed so far
int g_fail = 0;   // number of CHECKs that failed so far

// ---- turning a value into readable text (used when a CHECK_EQ fails) ----
template <typename T, typename = void>
struct is_streamable : std::false_type {};
template <typename T>
struct is_streamable<T, std::void_t<decltype(std::declval<std::ostream&>() << std::declval<const T&>())>>
    : std::true_type {};

template <typename T>
std::string show(const T& v) {
    std::ostringstream os;
    if constexpr (std::is_same_v<T, bool>) {
        os << (v ? "true" : "false");
    } else if constexpr (std::is_enum_v<T>) {
        os << "enum value " << static_cast<long long>(v);
    } else if constexpr (std::is_integral_v<T>) {
        // print integers in decimal AND hex: byte-order bugs jump out in hex
        os << +v << " (0x" << std::hex << +v << std::dec << ")";
    } else if constexpr (std::is_convertible_v<T, std::string>) {
        os << '"' << std::string(v) << '"';
    } else if constexpr (is_streamable<T>::value) {
        os << v;
    } else {
        os << "<value>";
    }
    return os.str();
}

template <typename U>
std::string show(const std::vector<U>& v) {
    std::ostringstream os;
    os << "{";
    for (size_t i = 0; i < v.size(); ++i) {
        if (i) os << ", ";
        if constexpr (std::is_same_v<U, uint8_t>)
            os << "0x" << std::hex << std::setw(2) << std::setfill('0') << unsigned(v[i]) << std::dec;
        else
            os << show(v[i]);
    }
    os << "}";
    return os.str();
}

// ---- the checks themselves: they print, count, and NEVER abort ----
bool check_impl(bool ok, const char* expr) {
    if (ok) { ++g_pass; std::cout << "✅ " << expr << "\n"; }
    else    { ++g_fail; std::cout << "❌ " << expr << "\n"; }
    return ok;
}

template <typename A, typename B>
bool check_eq_impl(const A& a, const B& b, const char* expr) {
    bool ok = (a == b);
    check_impl(ok, expr);
    if (!ok) std::cout << "     got:      " << show(a) << "\n     expected: " << show(b) << "\n";
    return ok;
}

bool check_near_impl(double a, double b, const char* expr) {
    // relative tolerance 1e-6 (plus a tiny absolute tolerance for values near 0)
    bool ok = std::fabs(a - b) <= 1e-6 * std::max(std::fabs(a), std::fabs(b)) + 1e-15;
    check_impl(ok, expr);
    if (!ok) std::cout << "     got:      " << a << "\n     expected: " << b << "\n";
    return ok;
}

void print_score() {
    std::cout << "Score so far: " << g_pass << " passed, " << g_fail << " failed"
              << " (re-running a test cell counts its CHECKs again)\n";
}

void reset_score() { g_pass = 0; g_fail = 0; }

// ---- hexdump: 16 bytes per line: offset | hex bytes | printable ASCII ----
void hexdump(const uint8_t* p, size_t n) {
    for (size_t off = 0; off < n; off += 16) {
        std::cout << std::hex << std::setfill('0') << std::setw(4) << off << "  ";
        for (size_t i = 0; i < 16; ++i) {
            if (off + i < n) std::cout << std::setw(2) << unsigned(p[off + i]) << ' ';
            else             std::cout << "   ";
            if (i == 7) std::cout << ' ';
        }
        std::cout << " |";
        for (size_t i = 0; i < 16 && off + i < n; ++i) {
            uint8_t c = p[off + i];
            std::cout << ((c >= 32 && c < 127) ? char(c) : '.');
        }
        std::cout << "|\n";
    }
    std::cout << std::dec << std::setfill(' ');
}

void hexdump(const std::vector<uint8_t>& v) { hexdump(v.data(), v.size()); }

} // namespace nbtest

using namespace nbtest;

#define CHECK(expr)          nbtest::check_impl(static_cast<bool>(expr), #expr)
#define CHECK_EQ(a, b)       nbtest::check_eq_impl((a), (b), #a " == " #b)
#define CHECK_NEAR(a, b)     nbtest::check_near_impl((a), (b), #a " ≈ " #b)

Here is what checks look like. One of them fails **on purpose**: read how the failure is reported.

In [2]:
{
    int answer = 6 * 7;
    CHECK(answer > 40);
    CHECK_EQ(answer, 42);
    CHECK_EQ(answer, 43);          // fails on purpose: note "got" vs "expected"
    CHECK_NEAR(0.1 + 0.2, 0.3);    // exact == would fail for doubles; NEAR tolerates rounding
}

✅ answer > 40
✅ answer == 42
❌ answer == 43
     got:      42 (0x2a)
     expected: 43 (0x2b)
✅ 0.1 + 0.2 ≈ 0.3


In [3]:
print_score();
reset_score();   // start from zero for the real exercises

Score so far: 3 passed, 1 failed (re-running a test cell counts its CHECKs again)


## 0.3 xeus-cling quirks (read this once, it will save you time)

xeus-cling interprets C++ cell by cell, which changes a few rules:

1. **No `main()`.** Statements at the top level of a cell run immediately.
2. **A cell either *defines* or *runs*, never both.** A cell that starts with `namespace`,
   `struct`, or a function definition may contain only definitions. A cell that starts with a
   statement (`CHECK(...)`, `std::cout << ...`, `{`) may contain only statements. This is why
   every exercise has a *definition* cell followed by a separate *test* cell.
3. **Redefinition errors.** Defining the same function or struct twice is an error, and re-running
   a definition cell does exactly that. To limit the damage, each exercise lives in its own
   namespace, such as `namespace ex5_3 { ... }`.

> **If you get a redefinition error, restart the kernel and use *Run → Run All Above* (JupyterLab)
> or *Run Above* (VS Code) from the cell you are editing, then run your cell again.**

Test cells are wrapped in `{ ... }`: their local variables vanish at the closing brace, so test
cells *can* be re-run freely.

**Q0.2** *(Why?)* — You fix a bug inside `namespace ex1_6 { ... }` and re-run the cell. Cling reports
`redefinition of 'read_be16'`. Why, and what do you do?

<details>
<summary><b>Solution</b> (click to expand)</summary>

The first run already defined `ex1_6::read_be16` in the interpreter's memory, and C++ forbids
two definitions of one function (the *One Definition Rule*). Cling cannot "replace" a definition
the way Python rebinds a name. **Fix:** restart the kernel, then *Run All Above* and run your
edited cell. Your edited source is kept, because only the interpreter's memory is reset.

*Misconception:* renaming the namespace to `ex1_6b` "fixes" it, but then the test cell, which calls
`ex1_6::read_be16`, still sees the old, buggy version.

</details>

## 0.4 Looking at raw bytes with `hexdump`

A **byte** is 8 bits and holds a value from 0 to 255. Networking code constantly inspects raw
bytes, and decimal is a poor fit: a byte is written far more compactly in **hexadecimal**
(base 16, digits `0–9` then `a–f` for 10–15). One byte is always exactly two hex digits,
`00` to `ff`, and C++ writes a hex literal with the prefix `0x` (e.g. `0x41` = 4·16 + 1 = 65).
Part 1 practises this properly.

`hexdump` prints 16 bytes per line:

```
0000  48 65 6c 6c 6f 2c 20 6e  65 74 77 6f 72 6b 21 0a  |Hello, network!.|
^^^^  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^  ^^^^^^^^^^^^^^^^^^
offset   16 bytes in hex (extra gap after the 8th)         printable chars
```

The *offset* (in hex) is the position of the line's first byte. Text is stored as bytes using
**ASCII**, a table mapping characters to numbers (`'H'` = 0x48, `'e'` = 0x65, newline = 0x0a).
Non-printable bytes appear as `.` on the right.

In [4]:
{
    const char* msg = "Hello, network!\n";
    hexdump(reinterpret_cast<const uint8_t*>(msg), std::strlen(msg));
}

0000  48 65 6c 6c 6f 2c 20 6e  65 74 77 6f 72 6b 21 0a  |Hello, network!.|


**Q0.3** *(Predict the output)* — The string `"Hi!"` is dumped with `hexdump`. `'H'` is 0x48, `'i'` is 0x69,
and `'!'` is 0x21. What does the single output line look like?

<details>
<summary><b>Solution</b> (click to expand)</summary>

```
0000  48 69 21                                          |Hi!|
```
Three bytes, offset `0000`, the hex bytes in memory order, then blank padding up to 16 columns,
and the printable characters between the bars. The dump shows bytes in the order they sit
in memory. That order matters a lot in Part 1: for multi-byte numbers it is **not** always the order you
would write the digits.

</details>

### 🔁 Recap

- Run the **setup cell** first; every later cell depends on it.
- `CHECK`, `CHECK_EQ`, and `CHECK_NEAR` print ✅/❌, count results, and never abort.
- A cell either defines things or runs statements; exercises live in unique namespaces.
- Redefinition error → restart the kernel → *Run All Above*.
- `hexdump` shows bytes as offset, hex, and ASCII columns.

**You should now be able to explain…**

- [ ] why re-running a definition cell fails in cling, and how to recover;
- [ ] what each of the three columns of a hexdump line means;
- [ ] why a byte is written as exactly two hex digits.

# Part 1 — Bytes and bits primer

Everything a network carries is a sequence of **bytes**. Before touching any protocol you need
to be fluent at reading and building bytes: fixed-width integers, hex, bit masks, and byte order.

## 1.1 Fixed-width integers

A **bit** is 0 or 1; a **byte** is 8 bits. C++'s `int` has a platform-dependent size, which is
useless for describing a protocol field "exactly 16 bits wide". The header `<cstdint>` provides
exact widths:

| type | bits | range |
|---|---|---|
| `uint8_t`  | 8  | 0 … 255 |
| `uint16_t` | 16 | 0 … 65 535 |
| `uint32_t` | 32 | 0 … 4 294 967 295 |
| `uint64_t` | 64 | 0 … 2⁶⁴ − 1 |

An $n$-bit unsigned type holds $0 \dots 2^n - 1$, and its arithmetic **wraps** modulo $2^n$
(255 + 1 becomes 0 in a `uint8_t`).

**Pitfall:** `uint8_t` is a character type, so `std::cout << b` prints a *character*.
Write `+b` (unary plus promotes it to `int`) or `unsigned(b)` to print the number.

**Q1.1** *(Predict the output)* — `uint8_t x = 250; x += 10; std::cout << +x;` — what is printed?

<details>
<summary><b>Solution</b> (click to expand)</summary>

**4**. 250 + 10 = 260, which does not fit in 8 bits; the stored value is
260 mod 256 = 4. Unsigned overflow is *well defined* in C++ (it wraps), whereas signed
overflow is undefined behaviour. That is one reason protocol code uses unsigned types everywhere.

</details>

**Q1.2** *(Predict the output)* — `uint8_t b = 65; std::cout << b << " " << +b;` — what is printed?

<details>
<summary><b>Solution</b> (click to expand)</summary>

**`A 65`**. `std::cout << b` treats a `uint8_t` as a `char`, and 65 is ASCII `'A'`. `+b`
promotes it to `int` and prints the number. This bug is common when printing header bytes.

</details>

### ✍️ Exercise 1.1 — The largest value of an n-bit field

Protocol fields often have odd widths (4, 13, or 20 bits). Write `max_value(bits)` returning
$2^{bits} - 1$ for `bits` in 1…32, as a `uint32_t`.

*Trap:* in C++, shifting a 32-bit value by 32 or more positions is **undefined behaviour**,
so `(1u << 32) - 1` is not allowed. Handle `bits == 32` separately.

In [5]:
namespace ex1_1 {
uint32_t max_value(int bits) {
    if (bits >= 32) return 0xFFFFFFFFu;   // all 32 bits set; avoids the UB shift
    return (1u << bits) - 1;              // e.g. bits=4: 0b10000 - 1 = 0b01111
}
}

**Test for Exercise 1.1** — run the next cell.

In [6]:
{
    CHECK_EQ(ex1_1::max_value(1), 1u);
    CHECK_EQ(ex1_1::max_value(4), 15u);
    CHECK_EQ(ex1_1::max_value(8), 255u);
    CHECK_EQ(ex1_1::max_value(16), 65535u);
    CHECK_EQ(ex1_1::max_value(32), 4294967295u);
}

✅ ex1_1::max_value(1) == 1u
✅ ex1_1::max_value(4) == 15u
✅ ex1_1::max_value(8) == 255u
✅ ex1_1::max_value(16) == 65535u
✅ ex1_1::max_value(32) == 4294967295u


<details>
<summary><b>Solution</b> (click to expand)</summary>

$2^n$ is `1u << n` (a 1 followed by $n$ zeros); subtracting 1 turns it into $n$ ones.
For $n = 32$ the shift is undefined behaviour: on x86 the hardware masks the shift count to
`32 & 31 = 0`, so it often *silently* yields `1 - 1 = 0`. The same trap appears when you build a
`/0` subnet mask in Part 6.

```cpp
namespace ex1_1 {
uint32_t max_value(int bits) {
    if (bits >= 32) return 0xFFFFFFFFu;   // all 32 bits set; avoids the UB shift
    return (1u << bits) - 1;              // e.g. bits=4: 0b10000 - 1 = 0b01111
}
}
```

</details>

## 1.2 Hexadecimal and binary notation

Hex (base 16) is the lingua franca of packet dumps because **one hex digit is exactly 4 bits**
(a *nibble*), so a byte is two digits and the bit pattern can be read directly:

```
hex digit:  0    1    2   ...  9    a    b    c    d    e    f
bits:      0000 0001 0010 ... 1001 1010 1011 1100 1101 1110 1111

0xC0A8  =  1100 0000 1010 1000  =  12·16³ + 0·16² + 10·16 + 8
```

C++ literals: `0x2f` (hex), `0b0010'1111` (binary; `'` is an optional digit separator).
To print hex: `std::cout << std::hex << std::setw(2) << std::setfill('0') << +b;`.
`std::hex` and `setfill` are *sticky* (they stay in effect), so restore with `std::dec`;
`setw` applies only to the next item.

**Q1.3** *(Compute)* — Convert `0xC0A8` to decimal, and `0b1010'0101` to hex.

<details>
<summary><b>Solution</b> (click to expand)</summary>

* `0xC0A8` = 0xC0·256 + 0xA8 = 192·256 + 168 = **49320**.
* `0b1010'0101`: split into nibbles, `1010` = a and `0101` = 5, giving **0xa5**.

Splitting a byte into nibbles is the fastest way to convert between binary and hex; never go through decimal.

</details>

### ✍️ Exercise 1.2 — Format a byte as two hex digits

Write `to_hex8(b)` returning exactly two **lowercase** hex digits, e.g. `0x0a` → `"0a"`.
Use `std::ostringstream` (a stream that writes into a string, from `<sstream>`) with the
manipulators above, and call `.str()` on it to get the result.

In [7]:
namespace ex1_2 {
std::string to_hex8(uint8_t b) {
    std::ostringstream os;
    // unsigned(b): print the number, not the character; setw(2)+setfill('0'): keep leading 0
    os << std::hex << std::setw(2) << std::setfill('0') << unsigned(b);
    return os.str();
}
}

**Test for Exercise 1.2** — run the next cell.

In [8]:
{
    CHECK_EQ(ex1_2::to_hex8(0x0a), "0a");
    CHECK_EQ(ex1_2::to_hex8(0xff), "ff");
    CHECK_EQ(ex1_2::to_hex8(0x00), "00");
    CHECK_EQ(ex1_2::to_hex8(65), "41");
}

✅ ex1_2::to_hex8(0x0a) == "0a"
✅ ex1_2::to_hex8(0xff) == "ff"
✅ ex1_2::to_hex8(0x00) == "00"
✅ ex1_2::to_hex8(65) == "41"


<details>
<summary><b>Solution</b> (click to expand)</summary>

Two classic bugs: forgetting `setw(2)`/`setfill('0')` (giving `"a"` instead of `"0a"`) and streaming
the `uint8_t` directly (giving the *character* with code 0x41, i.e. `"A"`, instead of `"41"`).
The stream is local, so its sticky `std::hex` does not leak into `std::cout`.

```cpp
namespace ex1_2 {
std::string to_hex8(uint8_t b) {
    std::ostringstream os;
    // unsigned(b): print the number, not the character; setw(2)+setfill('0'): keep leading 0
    os << std::hex << std::setw(2) << std::setfill('0') << unsigned(b);
    return os.str();
}
}
```

</details>

## 1.3 Bitwise operators and masks

Bits are numbered from **bit 0 = least significant** (value 1) up to bit 7 (value 128) in a byte.

| op | meaning | example (8-bit) |
|---|---|---|
| `a & b` | AND: keep bits set in both | `1100'1010 & 0000'1111 = 0000'1010` |
| `a \| b` | OR: set bits from either | `1100'0000 \| 0000'0011 = 1100'0011` |
| `a ^ b` | XOR: bits that differ | `1111'0000 ^ 1010'1010 = 0101'1010` |
| `~a` | NOT: flip every bit | `~0000'1111 = 1111'0000` |
| `a << n`, `a >> n` | shift left/right by n | `0000'0001 << 3 = 0000'1000` |

A **mask** is a constant selecting bits: `1u << i` selects bit *i*.
Test: `(x >> i) & 1`. Set: `x | (1u << i)`. Clear: `x & ~(1u << i)`.

**Pitfall (integer promotion):** C++ converts operands narrower than `int` to `int` before any
operator, so `~uint8_t(0x0F)` is the `int` `0xFFFFFFF0`. Cast results back: `uint8_t(~x)`.

**Q1.4** *(Predict the output)* — `std::cout << std::hex << (0xF0 & 0x3C) << " " << (0xF0 | 0x0F) << " " << (0x5A ^ 0xFF);`

<details>
<summary><b>Solution</b> (click to expand)</summary>

**`30 ff a5`**

* `1111'0000 & 0011'1100 = 0011'0000` (only the overlap survives)
* `1111'0000 | 0000'1111 = 1111'1111`
* `0101'1010 ^ 1111'1111 = 1010'0101`: XOR with all ones flips every bit (like `~`, but without promotion surprises in 8 bits).

</details>

### ✍️ Exercise 1.3 — Get, set and clear one bit

Implement the three helpers for bit index `i` in 0…7. Return `uint8_t` (cast!).

In [9]:
namespace ex1_3 {
bool get_bit(uint8_t x, int i)      { return (x >> i) & 1u; }
uint8_t set_bit(uint8_t x, int i)   { return uint8_t(x | (1u << i)); }
uint8_t clear_bit(uint8_t x, int i) { return uint8_t(x & ~(1u << i)); }
}

**Test for Exercise 1.3** — run the next cell.

In [10]:
{
    CHECK_EQ(ex1_3::get_bit(0b0000'0100, 2), true);
    CHECK_EQ(ex1_3::get_bit(0b0000'0100, 3), false);
    CHECK_EQ(ex1_3::get_bit(0x80, 7), true);
    CHECK_EQ(ex1_3::set_bit(0x00, 7), 0x80);
    CHECK_EQ(ex1_3::set_bit(0x01, 0), 0x01);     // setting a set bit changes nothing
    CHECK_EQ(ex1_3::clear_bit(0xFF, 0), 0xFE);
    CHECK_EQ(ex1_3::clear_bit(0x12, 4), 0x02);
}

✅ ex1_3::get_bit(0b0000'0100, 2) == true
✅ ex1_3::get_bit(0b0000'0100, 3) == false
✅ ex1_3::get_bit(0x80, 7) == true
✅ ex1_3::set_bit(0x00, 7) == 0x80
✅ ex1_3::set_bit(0x01, 0) == 0x01
✅ ex1_3::clear_bit(0xFF, 0) == 0xFE
✅ ex1_3::clear_bit(0x12, 4) == 0x02


<details>
<summary><b>Solution</b> (click to expand)</summary>

`~(1u << i)` is a mask with every bit set *except* bit *i*; AND-ing with it clears exactly that
bit. The `uint8_t(...)` casts matter: without them the expression is an `unsigned int` whose
upper 24 bits are garbage from the promoted `~`, which the implicit conversion on `return` would
truncate, but only by luck of the return type.

```cpp
namespace ex1_3 {
bool get_bit(uint8_t x, int i)      { return (x >> i) & 1u; }
uint8_t set_bit(uint8_t x, int i)   { return uint8_t(x | (1u << i)); }
uint8_t clear_bit(uint8_t x, int i) { return uint8_t(x & ~(1u << i)); }
}
```

</details>

## 1.4 Extracting a multi-bit field

Protocol headers pack several small numbers into one byte. Example: a byte `0x45` holding two
4-bit fields, *A* in the high nibble and *B* in the low nibble:

```
  bit:   7 6 5 4 | 3 2 1 0
0x45 =   0 1 0 0 | 0 1 0 1
         A = 4   | B = 5
```

General recipe for a field of `width` bits starting at bit `shift`:

$$\text{field} = (b \gg \text{shift}) \;\&\; (2^{\text{width}} - 1)$$

Shift the field down to bit 0, then mask off everything above it. Packing is the reverse:
`(A << 4) | B`.

**Q1.5** *(Compute)* — For `b = 0xB6`, what is the 3-bit field starting at bit 2 (bits 2, 3, 4)?

<details>
<summary><b>Solution</b> (click to expand)</summary>

`0xB6 = 10110110`. Shift right by 2: `00101101`. Mask with `0b111`: `101` = **5**.

A common error is to mask first and then shift with the wrong mask (`b & 0b111` selects bits 0–2, not 2–4).
If you mask first, the mask must be pre-shifted: `(b & (0b111 << 2)) >> 2`.

</details>

### ✍️ Exercise 1.4 — Get a field and pack two nibbles

* `get_field(b, shift, width)`: the recipe above (width in 1…8).
* `pack_nibbles(hi, lo)`: returns the byte with `hi` in bits 7–4 and `lo` in bits 3–0
  (assume both are ≤ 15).

In [11]:
namespace ex1_4 {
uint8_t get_field(uint8_t b, int shift, int width) {
    uint8_t mask = uint8_t((1u << width) - 1);   // width ones, e.g. width 3 -> 0b111
    return uint8_t((b >> shift) & mask);
}
uint8_t pack_nibbles(uint8_t hi, uint8_t lo) {
    return uint8_t((hi << 4) | (lo & 0x0F));
}
}

**Test for Exercise 1.4** — run the next cell.

In [12]:
{
    CHECK_EQ(ex1_4::get_field(0x45, 4, 4), 4);   // high nibble
    CHECK_EQ(ex1_4::get_field(0x45, 0, 4), 5);   // low nibble
    CHECK_EQ(ex1_4::get_field(0xB6, 2, 3), 5);
    CHECK_EQ(ex1_4::get_field(0x12, 1, 1), 1);   // bit 1 of 0001'0010
    CHECK_EQ(ex1_4::get_field(0x12, 0, 1), 0);
    CHECK_EQ(ex1_4::pack_nibbles(4, 5), 0x45);
    CHECK_EQ(ex1_4::pack_nibbles(0xF, 0x0), 0xF0);
}

✅ ex1_4::get_field(0x45, 4, 4) == 4
✅ ex1_4::get_field(0x45, 0, 4) == 5
✅ ex1_4::get_field(0xB6, 2, 3) == 5
✅ ex1_4::get_field(0x12, 1, 1) == 1
✅ ex1_4::get_field(0x12, 0, 1) == 0
✅ ex1_4::pack_nibbles(4, 5) == 0x45
✅ ex1_4::pack_nibbles(0xF, 0x0) == 0xF0


<details>
<summary><b>Solution</b> (click to expand)</summary>

Masking with `lo & 0x0F` in `pack_nibbles` is defensive: if a caller passes a value > 15, its
extra bits would otherwise overwrite the high nibble. Note that `1u << width` with `width = 8` is
fine here (`1u` is 32 bits wide), unlike the `<< 32` trap from §1.1.

```cpp
namespace ex1_4 {
uint8_t get_field(uint8_t b, int shift, int width) {
    uint8_t mask = uint8_t((1u << width) - 1);   // width ones, e.g. width 3 -> 0b111
    return uint8_t((b >> shift) & mask);
}
uint8_t pack_nibbles(uint8_t hi, uint8_t lo) {
    return uint8_t((hi << 4) | (lo & 0x0F));
}
}
```

</details>

## 1.5 Endianness: how multi-byte integers sit in memory

Memory is addressed per byte, so a `uint32_t` occupies 4 consecutive addresses. *Which byte goes
first* is a CPU convention called **endianness**:

```
value 0x0A0B0C0D     address:  +0   +1   +2   +3
big-endian    (BE)             0A   0B   0C   0D   most significant byte first
little-endian (LE)             0D   0C   0B   0A   least significant byte first
```

x86-64 and ARM (as configured on phones, Macs and servers) are **little-endian**. Neither order
is "correct"; they are simply conventions. `std::memcpy(dst, src, n)` (from `<cstring>`) copies
`n` raw bytes, which lets you look at the in-memory representation of a value:

In [13]:
{
    uint32_t v = 0x0A0B0C0D;
    uint8_t bytes[4];
    std::memcpy(bytes, &v, 4);       // copy the 4 bytes exactly as they sit in memory
    hexdump(bytes, 4);
}

0000  0d 0c 0b 0a                                       |....|


**Q1.6** *(Predict the output)* — On your (little-endian) machine: `uint16_t v = 0x1234; uint8_t buf[2]; std::memcpy(buf, &v, 2);
std::cout << std::hex << +buf[0];` — what is printed?

<details>
<summary><b>Solution</b> (click to expand)</summary>

**`34`**. Little-endian stores the least significant byte (0x34) at the lowest address, so
`buf = {0x34, 0x12}`. On a big-endian CPU it would print `12`. A hexdump of an integer taken from
memory therefore looks "reversed" on x86.

</details>

### ✍️ Exercise 1.5 — Detect the host's endianness

Write `host_is_little_endian()` using `memcpy` on a `uint16_t` whose value is 1: inspect which
byte ends up non-zero.

In [14]:
namespace ex1_5 {
bool host_is_little_endian() {
    uint16_t one = 1;               // bytes are {01,00} on LE, {00,01} on BE
    uint8_t first;
    std::memcpy(&first, &one, 1);   // copy just the byte at the lowest address
    return first == 1;
}
}

**Test for Exercise 1.5** — run the next cell.

In [15]:
{
    // The compiler knows the answer too; we compare against its built-in macro.
    bool expected = (__BYTE_ORDER__ == __ORDER_LITTLE_ENDIAN__);
    CHECK_EQ(ex1_5::host_is_little_endian(), expected);
}

✅ ex1_5::host_is_little_endian() == expected


<details>
<summary><b>Solution</b> (click to expand)</summary>

The byte at the lowest address is the least significant one exactly when the host is little-endian.
`memcpy` is the portable way to reinterpret bytes; casting pointers to unrelated types
(`*(uint8_t*)&one` is allowed, but `*(uint32_t*)some_bytes` is not in general) runs into
C++'s aliasing and alignment rules.

```cpp
namespace ex1_5 {
bool host_is_little_endian() {
    uint16_t one = 1;               // bytes are {01,00} on LE, {00,01} on BE
    uint8_t first;
    std::memcpy(&first, &one, 1);   // copy just the byte at the lowest address
    return first == 1;
}
}
```

</details>

## 1.6 Network byte order and the portable way to read it

Two machines exchanging a 16-bit number must agree on byte order, or an LE sender and a BE
receiver disagree about every value. Internet protocols fix the convention: multi-byte header
fields are transmitted **big-endian**, which is therefore called **network byte order**.

```
bytes on the wire (in order):   0x12   0x34
value:                          0x1234   =  (p[0] << 8) | p[1]
```

Assembling the value from individual bytes with shifts is **portable**: it gives the same
answer on any CPU and never performs a misaligned multi-byte load. Do *not* write
`*(uint16_t*)p`: on LE it gives 0x3412, and if `p` is not 2-byte aligned it is undefined behaviour.

**Pitfall:** `p[0] << 24` promotes `p[0]` to a *signed* `int`; for `p[0] ≥ 0x80` the result
overflows it. Write `uint32_t(p[0]) << 24`.

**Q1.7** *(Predict the output)* — Two bytes arrive: `0x01 0xBB`. What 16-bit value do they represent in network byte order?
What would a buggy little-endian read (`memcpy` into a `uint16_t` on x86) produce?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Network order (big-endian): `0x01BB` = **443**. The buggy LE read gives `0xBB01` =
**47873**. Byte-order bugs rarely crash; they silently produce plausible-looking wrong numbers.

</details>

### ✍️ Exercise 1.6 — read_be16, read_be32, write_be16, write_be32

Implement the four big-endian helpers. You will reuse them in almost every later part.
`write_*` store `v` into `p[0..1]` / `p[0..3]` in network byte order.

In [16]:
namespace ex1_6 {
uint16_t read_be16(const uint8_t* p) {
    return uint16_t((uint16_t(p[0]) << 8) | p[1]);
}
uint32_t read_be32(const uint8_t* p) {
    return (uint32_t(p[0]) << 24) | (uint32_t(p[1]) << 16) |
           (uint32_t(p[2]) << 8)  |  uint32_t(p[3]);
}
void write_be16(uint8_t* p, uint16_t v) {
    p[0] = uint8_t(v >> 8);      // most significant byte first
    p[1] = uint8_t(v & 0xFF);
}
void write_be32(uint8_t* p, uint32_t v) {
    p[0] = uint8_t(v >> 24);
    p[1] = uint8_t(v >> 16);     // the uint8_t cast keeps only the low 8 bits
    p[2] = uint8_t(v >> 8);
    p[3] = uint8_t(v);
}
}

**Test for Exercise 1.6** — run the next cell.

In [17]:
{
    const uint8_t in[] = {0x12, 0x34, 0x56, 0x78, 0xff, 0xff, 0xff, 0xfe};
    CHECK_EQ(ex1_6::read_be16(in), 0x1234);
    CHECK_EQ(ex1_6::read_be16(in + 2), 0x5678);
    CHECK_EQ(ex1_6::read_be32(in), 0x12345678u);
    CHECK_EQ(ex1_6::read_be32(in + 4), 0xfffffffeu);   // top bit set: catches signed-shift bugs

    uint8_t out[6] = {0, 0, 0, 0, 0, 0};
    ex1_6::write_be16(out, 0xABCD);
    ex1_6::write_be32(out + 2, 0xC0A8010Au);
    const uint8_t want[] = {0xab, 0xcd, 0xc0, 0xa8, 0x01, 0x0a};
    CHECK(std::memcmp(out, want, 6) == 0);
    hexdump(out, 6);
    CHECK_EQ(ex1_6::read_be32(out + 2), 0xC0A8010Au);   // round trip
}

✅ ex1_6::read_be16(in) == 0x1234
✅ ex1_6::read_be16(in + 2) == 0x5678
✅ ex1_6::read_be32(in) == 0x12345678u
✅ ex1_6::read_be32(in + 4) == 0xfffffffeu
✅ std::memcmp(out, want, 6) == 0
0000  ab cd c0 a8 01 0a                                 |......|
✅ ex1_6::read_be32(out + 2) == 0xC0A8010Au


<details>
<summary><b>Solution</b> (click to expand)</summary>

Each byte is widened to the result type **before** shifting, which avoids the signed-`int`
overflow at `<< 24`. `write_*` go the other way: shift the wanted byte down to bits 0–7 and truncate
with a cast. These functions never ask what the host's endianness is, which is why they are portable.

```cpp
namespace ex1_6 {
uint16_t read_be16(const uint8_t* p) {
    return uint16_t((uint16_t(p[0]) << 8) | p[1]);
}
uint32_t read_be32(const uint8_t* p) {
    return (uint32_t(p[0]) << 24) | (uint32_t(p[1]) << 16) |
           (uint32_t(p[2]) << 8)  |  uint32_t(p[3]);
}
void write_be16(uint8_t* p, uint16_t v) {
    p[0] = uint8_t(v >> 8);      // most significant byte first
    p[1] = uint8_t(v & 0xFF);
}
void write_be32(uint8_t* p, uint32_t v) {
    p[0] = uint8_t(v >> 24);
    p[1] = uint8_t(v >> 16);     // the uint8_t cast keeps only the low 8 bits
    p[2] = uint8_t(v >> 8);
    p[3] = uint8_t(v);
}
}
```

</details>

## 1.7 `htons`, `ntohs`, `htonl`, `ntohl`

The POSIX header `<arpa/inet.h>` provides the classic conversion functions. Read the names as
**h**ost **to** **n**etwork **s**hort (16-bit) / **l**ong (32-bit), and the reverse:

```
htons, ntohs : uint16_t -> uint16_t      htonl, ntohl : uint32_t -> uint32_t
little-endian host: swap the bytes       big-endian host: return the value unchanged
```

They operate on *values*, not byte pointers: `htons(0x1234)` returns the number whose in-memory
bytes are `12 34` on this host. You need them wherever an API wants a network-order *value*
inside a struct; the socket API in Part 9 is the main example. For parsing bytes, `read_be16` and
`memcpy` + `ntohs` are equivalent.

**Q1.8** *(Predict the output)* — On x86: `std::cout << std::hex << htons(0x1234) << " " << htonl(0x0A000001);`

<details>
<summary><b>Solution</b> (click to expand)</summary>

**`3412 100000a`**. On a little-endian host both functions reverse
the bytes. Printing the result as a *number* on the same host therefore shows the swapped value; when stored
to memory, however, its bytes are in network order (`12 34`).

</details>

**Q1.9** *(Why?)* — On a big-endian host `htonl` does nothing. Why should portable code still call it,
and does a single `uint8_t` field ever need conversion?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Code must compile and behave identically on both kinds of host. Calling `htonl` documents the intent and
becomes a no-op where no swap is needed, so it costs nothing. A single byte has **no** byte order:
endianness is about the order of bytes *within* a multi-byte value. (The order of *bits* on a
cable is a physical-layer matter, handled by hardware and invisible to software.)

</details>

### ✍️ Exercise 1.7 — Swap bytes yourself and verify against htons/htonl

* `swap16(v)`, `swap32(v)`: reverse the byte order of a value using shifts and masks.
* `read_be16_via_ntohs(p)`: read 2 bytes with `memcpy` into a `uint16_t`, then apply `ntohs`.

On a little-endian host `swap16` must agree with `htons`.

In [18]:
namespace ex1_7 {
uint16_t swap16(uint16_t v) {
    return uint16_t((v >> 8) | (v << 8));   // truncation to 16 bits drops the overflow
}
uint32_t swap32(uint32_t v) {
    return  (v >> 24)                  // byte 3 -> byte 0
         | ((v >> 8)  & 0x0000FF00u)   // byte 2 -> byte 1
         | ((v << 8)  & 0x00FF0000u)   // byte 1 -> byte 2
         |  (v << 24);                 // byte 0 -> byte 3
}
uint16_t read_be16_via_ntohs(const uint8_t* p) {
    uint16_t raw;
    std::memcpy(&raw, p, 2);   // raw now holds the wire bytes in memory order
    return ntohs(raw);         // reinterpret them as network order
}
}

**Test for Exercise 1.7** — run the next cell.

In [19]:
{
    CHECK_EQ(ex1_7::swap16(0x1234), 0x3412);
    CHECK_EQ(ex1_7::swap32(0x0A0B0C0Du), 0x0D0C0B0Au);
    CHECK_EQ(ex1_7::swap32(ex1_7::swap32(0xDEADBEEFu)), 0xDEADBEEFu);
    bool le = (__BYTE_ORDER__ == __ORDER_LITTLE_ENDIAN__);
    if (le) {
        CHECK_EQ(ex1_7::swap16(0xBEEF), htons(0xBEEF));
        CHECK_EQ(ex1_7::swap32(0x0A000001u), htonl(0x0A000001u));
    }
    const uint8_t wire[] = {0x01, 0xbb, 0x80, 0x00};
    CHECK_EQ(ex1_7::read_be16_via_ntohs(wire), 443);
    CHECK_EQ(ex1_7::read_be16_via_ntohs(wire + 2), 0x8000);
}

✅ ex1_7::swap16(0x1234) == 0x3412
✅ ex1_7::swap32(0x0A0B0C0Du) == 0x0D0C0B0Au
✅ ex1_7::swap32(ex1_7::swap32(0xDEADBEEFu)) == 0xDEADBEEFu
✅ ex1_7::swap16(0xBEEF) == htons(0xBEEF)
✅ ex1_7::swap32(0x0A000001u) == htonl(0x0A000001u)
✅ ex1_7::read_be16_via_ntohs(wire) == 443
✅ ex1_7::read_be16_via_ntohs(wire + 2) == 0x8000


<details>
<summary><b>Solution</b> (click to expand)</summary>

`swap32` moves each byte to its mirror position; the masks discard the bytes that slid into the wrong lane.
Compilers recognise this pattern and emit a single `bswap` instruction, which is essentially what
`htonl` is on x86. `memcpy` + `ntohs` works because `memcpy` preserves byte order and `ntohs`
then performs exactly the swap that the host's endianness requires (none on BE).

```cpp
namespace ex1_7 {
uint16_t swap16(uint16_t v) {
    return uint16_t((v >> 8) | (v << 8));   // truncation to 16 bits drops the overflow
}
uint32_t swap32(uint32_t v) {
    return  (v >> 24)                  // byte 3 -> byte 0
         | ((v >> 8)  & 0x0000FF00u)   // byte 2 -> byte 1
         | ((v << 8)  & 0x00FF0000u)   // byte 1 -> byte 2
         |  (v << 24);                 // byte 0 -> byte 3
}
uint16_t read_be16_via_ntohs(const uint8_t* p) {
    uint16_t raw;
    std::memcpy(&raw, p, 2);   // raw now holds the wire bytes in memory order
    return ntohs(raw);         // reinterpret them as network order
}
}
```

</details>

### 🔁 Recap

- Use exact-width unsigned types (`uint8_t`…`uint32_t`); they wrap modulo $2^n$. Print a `uint8_t` with `+b`.
- One hex digit = one nibble = 4 bits; one byte = two hex digits.
- Field extraction: `(b >> shift) & ((1u << width) - 1)`. Beware shifts ≥ the type's width and integer promotion.
- Endianness is the byte order of multi-byte values in memory; x86 and ARM are little-endian.
- Network byte order is big-endian. Read it portably with shifts (`read_be16/32`), or with `memcpy` + `ntohs/ntohl`.

**You should now be able to explain…**

- [ ] why `std::cout << uint8_t(65)` prints `A`;
- [ ] how to extract any bit field from a byte, and why `1u << 32` is a bug;
- [ ] what `htons(0x1234)` returns on x86, and why;
- [ ] why `*(uint16_t*)p` is wrong for parsing packet bytes.

# Part 2 — What a network is

## 2.1 Hosts, links, and packets

* A **host** (or *end system*) is a computer that runs applications: a laptop, a phone, a server.
* A **link** is a physical connection that carries bits between two devices: a cable, a fibre, or a radio channel.
* A **node** is any device attached to links: a host, or a *packet switch*. A packet switch is
  a device that receives data on one link and forwards it on another; Part 4 distinguishes its two main kinds.
* A **packet** is a bounded chunk of bytes sent as one unit. It has a **header** (control
  information: addresses, lengths, types) followed by a **payload** (the data being carried).

```
[host A]===link===(packet switch)===link===(packet switch)===link===[host B]

one packet:   +-----------+---------------------------+
              |  header   |          payload          |
              +-----------+---------------------------+
```

A long message (a 1 GB file) is split into many packets, and each is forwarded independently.

**Q2.1** *(Why?)* — Why split a 1 GB file into ~1500-byte packets instead of sending it as one giant unit?

<details>
<summary><b>Solution</b> (click to expand)</summary>

* **Sharing:** a link carrying one giant unit is monopolised for its whole duration; with small packets
  many conversations interleave on the same link.
* **Error recovery:** a corrupted bit ruins only one small packet, which is resent, rather than the whole gigabyte.
* **Buffering and pipelining:** a switch must store a whole unit before forwarding it (see §2.6), so
  small packets need small buffers and let successive links work in parallel.

*Misconception:* packets exist "because cables are short". Packetisation is about sharing and failure
granularity, not distance.

</details>

## 2.2 Circuit switching vs packet switching

**Circuit switching** (the classic telephone network) *reserves* a fixed share of capacity on
every link along the path for the whole call. That share is guaranteed, but it is wasted whenever
the user is silent.

**Packet switching** (the Internet) reserves nothing. Packets from all users share each link on
demand; a switch that receives a packet while the link is busy **stores** it in a buffer
(a *queue*) and **forwards** it later. This is *statistical multiplexing*: it exploits the fact
that users are rarely all active at once.

Example: a 10 Mb/s link, where each user needs 1 Mb/s while active and is active 10% of the time.
Circuit switching admits exactly 10 users. With packet switching and $n$ users, trouble (demand
above 10 Mb/s) occurs only when more than 10 are active simultaneously:

$$P(X > 10), \quad X \sim \mathrm{Binomial}(n, 0.1).$$

### ✍️ Exercise 2.2 — How rare is overload?

Implement `prob_more_than(n, p, k)` $= P(X > k) = \sum_{i=k+1}^{n} \binom{n}{i} p^i (1-p)^{n-i}$.
Write a helper `choose(n, i)` that computes $\binom{n}{i}$ in `double` with the product
$\prod_{j=1}^{i} \frac{n - i + j}{j}$ (this avoids huge factorials). `std::pow` is in `<cmath>`.

In [20]:
namespace ex2_2 {
double choose(int n, int i) {
    double c = 1.0;
    for (int j = 1; j <= i; ++j) c = c * (n - i + j) / j;   // stays an integer at every step
    return c;
}
double prob_more_than(int n, double p, int k) {
    double total = 0.0;
    for (int i = k + 1; i <= n; ++i)
        total += choose(n, i) * std::pow(p, i) * std::pow(1.0 - p, n - i);
    return total;
}
}

**Test for Exercise 2.2** — run the next cell.

In [21]:
{
    CHECK_NEAR(ex2_2::choose(35, 11), 417225900.0);
    CHECK_NEAR(ex2_2::prob_more_than(10, 0.5, 5), 0.376953125);
    double p35 = ex2_2::prob_more_than(35, 0.1, 10);
    CHECK_NEAR(p35, 0.00042429759545082625);
    std::cout << "P(overload) with 35 users = " << p35 << "\n";
}

✅ ex2_2::choose(35, 11) ≈ 417225900.0
✅ ex2_2::prob_more_than(10, 0.5, 5) ≈ 0.376953125
✅ p35 ≈ 0.00042429759545082625
P(overload) with 35 users = 0.000424298


<details>
<summary><b>Solution</b> (click to expand)</summary>

With 35 users the link is overloaded only with probability ≈ 0.0004. Packet switching therefore serves
**3.5×** as many users as circuit switching at nearly the same quality, and during that rare overload
the excess is buffered rather than refused. The price is that delay is no longer guaranteed; that is
the subject of §2.5.

```cpp
namespace ex2_2 {
double choose(int n, int i) {
    double c = 1.0;
    for (int j = 1; j <= i; ++j) c = c * (n - i + j) / j;   // stays an integer at every step
    return c;
}
double prob_more_than(int n, double p, int k) {
    double total = 0.0;
    for (int i = k + 1; i <= n; ++i)
        total += choose(n, i) * std::pow(p, i) * std::pow(1.0 - p, n - i);
    return total;
}
}
```

</details>

## 2.3 Bandwidth vs latency (and units)

* **Bandwidth** (transmission rate) $R$ is the number of bits per second a link can push onto
  the wire: *how wide the pipe is*.
* **Latency** is how long a bit takes to get from A to B: *how long the pipe is*.

They are independent: a satellite link can have huge bandwidth *and* huge latency.

**Units, the classic trap:**
* `b` = bit and `B` = byte, so 100 **Mb/s** (megabits per second) = 12.5 **MB/s** (megabytes per second).
* Data *rates* use decimal prefixes: k = 10³, M = 10⁶, G = 10⁹. Memory and file sizes are often
  binary: KiB = 1024 B, MiB = 1024² B.

**Q2.2** *(Compute)* — Your ISP sells "500 Mb/s". Ignoring overhead, how long does a 2 GB (2·10⁹ bytes) download take at best?

<details>
<summary><b>Solution</b> (click to expand)</summary>

$\dfrac{2\cdot10^9 \cdot 8\ \text{bits}}{500\cdot10^6\ \text{bits/s}} = $ **32 s**.

Forgetting the factor 8 (bytes → bits) gives 4 s, which is 8× too optimistic. Always convert everything to bits
*or* everything to bytes before dividing.

</details>

## 2.4 Transmission delay and propagation delay

Sending one packet of $L$ bits over one link of rate $R$ bits/s and length $d$ metres:

* **Transmission delay** $d_\text{trans} = L / R$: the time to *push all $L$ bits onto the link*,
  one after another.
* **Propagation delay** $d_\text{prop} = d / s$: the time for *one bit* to travel the length of the
  link at signal speed $s$. In copper and fibre $s \approx 2\cdot10^8$ m/s, about ⅔ of the speed of light.

```
time ─►
sender:    |<- L/R: first bit … last bit leave ->|
receiver:              |<- d/s ->|<- L/R: first bit … last bit arrive ->|
```

The two are easy to confuse. $L/R$ depends on the packet size and the link rate, not on distance;
$d/s$ depends on distance, not on the packet or the rate.

### ✍️ Exercise 2.4 — Transmission and propagation delay

Implement both functions (SI units: bits, bits/s, metres, metres/s; result in seconds).

In [22]:
namespace ex2_4 {
double transmission_delay(double L_bits, double R_bps) { return L_bits / R_bps; }
double propagation_delay(double d_m, double s_mps)     { return d_m / s_mps; }
}

**Test for Exercise 2.4** — run the next cell.

In [23]:
{
    // 1500-byte packet on a 1 Gb/s link that is 3000 km long
    CHECK_NEAR(ex2_4::transmission_delay(1500 * 8, 1e9), 1.2e-05);
    CHECK_NEAR(ex2_4::propagation_delay(3000e3, 2e8), 0.015);
    // same packet on a 10 Mb/s link
    CHECK_NEAR(ex2_4::transmission_delay(1500 * 8, 10e6), 0.0012);
}

✅ ex2_4::transmission_delay(1500 * 8, 1e9) ≈ 1.2e-05
✅ ex2_4::propagation_delay(3000e3, 2e8) ≈ 0.015
✅ ex2_4::transmission_delay(1500 * 8, 10e6) ≈ 0.0012


<details>
<summary><b>Solution</b> (click to expand)</summary>

Trivial formulas; the value of the exercise lies in the magnitudes. On a fast long-haul link propagation
dominates by three orders of magnitude, whereas on a slow access link transmission can dominate.

```cpp
namespace ex2_4 {
double transmission_delay(double L_bits, double R_bps) { return L_bits / R_bps; }
double propagation_delay(double d_m, double s_mps)     { return d_m / s_mps; }
}
```

</details>

**Q2.3** *(Compute)* — For that 1500-byte packet over 3000 km of fibre at 1 Gb/s, which delay dominates, and by what factor?
Would upgrading to 10 Gb/s help much?

<details>
<summary><b>Solution</b> (click to expand)</summary>

$d_\text{trans} = 12\ \mu s$ and $d_\text{prop} = 15\ \text{ms}$: propagation is
**1250×** larger. At 10 Gb/s, $d_\text{trans}$ drops to 1.2 µs, but the total barely
moves (15.0120 ms → 15.0012 ms).

*Misconception:* "a faster link means lower latency". Bandwidth only shrinks $L/R$; nothing shrinks $d/s$ except
moving the endpoints closer, which is why content is cached near users.

</details>

## 2.5 Queuing and processing delay

Each switch along the path adds two more delays:

* **Processing delay** $d_\text{proc}$: examining the header and choosing the output link
  (nanoseconds to microseconds in hardware).
* **Queuing delay** $d_\text{queue}$: waiting in the output buffer while earlier packets are
  transmitted. It varies from packet to packet.

$$d_\text{nodal} = d_\text{proc} + d_\text{queue} + d_\text{trans} + d_\text{prop}$$

Queuing is governed by the **traffic intensity** $I = La/R$, where $a$ is the average arrival
rate in packets/s. As $I \to 1$ the average queuing delay grows without bound, and for $I > 1$
the queue grows forever. Buffers are finite, so a packet arriving at a full buffer is **dropped**:
this is **packet loss**.

A simple worst case is a *burst*: $N$ packets arrive at an empty link at the same instant.
Packet $k$ (counting from 0) waits for the $k$ packets ahead of it, i.e. $k \cdot L/R$.

### ✍️ Exercise 2.5 — Average queuing delay of a burst

Implement `avg_queuing_delay_burst(N, L_bits, R_bps)`: the average of $k \cdot L/R$ over $k = 0 \dots N-1$.
Use a loop (do not look up a closed form yet).

In [24]:
namespace ex2_5 {
double avg_queuing_delay_burst(int N, double L_bits, double R_bps) {
    double total = 0.0;
    for (int k = 0; k < N; ++k) total += k * L_bits / R_bps;   // packet k waits for k packets
    return total / N;
}
}

**Test for Exercise 2.5** — run the next cell.

In [25]:
{
    CHECK_NEAR(ex2_5::avg_queuing_delay_burst(1, 12000, 1e8), 0.0);
    CHECK_NEAR(ex2_5::avg_queuing_delay_burst(5, 12000, 100000000.0), 0.00024);
}

✅ ex2_5::avg_queuing_delay_burst(1, 12000, 1e8) ≈ 0.0
✅ ex2_5::avg_queuing_delay_burst(5, 12000, 100000000.0) ≈ 0.00024


<details>
<summary><b>Solution</b> (click to expand)</summary>

Summing $k$ from 0 to $N-1$ gives $\frac{N(N-1)}{2}$, so the average is $\frac{N-1}{2}\cdot\frac{L}{R}$; for
5 packets of 1500 B at 100 Mb/s that is 240 µs. The first packet waits nothing and the last waits longest,
so queuing delay differs from packet to packet even inside one burst. This variation is called **jitter**.

```cpp
namespace ex2_5 {
double avg_queuing_delay_burst(int N, double L_bits, double R_bps) {
    double total = 0.0;
    for (int k = 0; k < N; ++k) total += k * L_bits / R_bps;   // packet k waits for k packets
    return total / N;
}
}
```

</details>

**Q2.4** *(Why?)* — A vendor proposes giant router buffers "so that no packet is ever dropped". Why is this a bad idea?
Also compute $I$ for $L$ = 1500 B, $a$ = 8000 packets/s, $R$ = 100 Mb/s.

<details>
<summary><b>Solution</b> (click to expand)</summary>

$I = \dfrac{12000 \cdot 8000}{10^8} = $ **0.96**, a busy link where queues form often.

Huge buffers turn loss into **delay**: when $I$ approaches 1, a full giant buffer can hold hundreds of
milliseconds of packets, so interactive traffic (calls, games) suffers. This is known as *bufferbloat*.
Loss is also a useful **signal**: well-designed senders slow down when they notice loss, so hiding
loss postpones their reaction. *Misconception:* "packet loss always means a broken link". Most loss on
the Internet is simply full buffers.

</details>

## 2.6 Store-and-forward over several links

A packet switch must receive the **entire** packet before it forwards it: it has to read the
whole header, and possibly check the entire packet for errors. This is **store-and-forward**. Over
$N$ identical links with no queuing and no processing delay, one packet therefore takes

$$d_\text{end-to-end} = N\left(\frac{L}{R} + d_\text{prop}\right).$$

With $P$ packets sent back to back (ignore $d_\text{prop}$), the links work like a pipeline:

```
time slot (each = L/R):   1    2    3    4    5    6
link 1 (A -> S1):        p1   p2   p3   p4
link 2 (S1 -> S2):            p1   p2   p3   p4
link 3 (S2 -> B):                  p1   p2   p3   p4      total = (N + P - 1) · L/R
```

### ✍️ Exercise 2.6 — End-to-end delay

* `e2e_one(N, L_bits, R_bps, dprop_per_link)`: one packet over $N$ links.
* `e2e_many(P, N, L_bits, R_bps)`: $P$ back-to-back packets, ignoring propagation.

In [26]:
namespace ex2_6 {
double e2e_one(int N, double L_bits, double R_bps, double dprop) {
    return N * (L_bits / R_bps + dprop);         // each hop: store fully, then send
}
double e2e_many(int P, int N, double L_bits, double R_bps) {
    return (N + P - 1) * (L_bits / R_bps);        // pipeline: fill (N) + drain (P - 1)
}
}

**Test for Exercise 2.6** — run the next cell.

In [27]:
{
    CHECK_NEAR(ex2_6::e2e_one(3, 12000, 10000000.0, 0.001), 0.006599999999999999);
    CHECK_NEAR(ex2_6::e2e_many(4, 3, 12000, 10000000.0), 0.0072);
    CHECK_NEAR(ex2_6::e2e_many(1, 1, 12000, 10000000.0), 0.0012);   // one packet, one link = L/R
}

✅ ex2_6::e2e_one(3, 12000, 10000000.0, 0.001) ≈ 0.006599999999999999
✅ ex2_6::e2e_many(4, 3, 12000, 10000000.0) ≈ 0.0072
✅ ex2_6::e2e_many(1, 1, 12000, 10000000.0) ≈ 0.0012


<details>
<summary><b>Solution</b> (click to expand)</summary>

For 3 links at 10 Mb/s with 1 ms of propagation each, one 1500 B packet takes 6.6 ms. Four packets
take $(3+4-1)\cdot 1.2$ ms = 7.2 ms, **not** $4 \times 3 \times 1.2$ ms, because different links
transmit different packets at the same time.

```cpp
namespace ex2_6 {
double e2e_one(int N, double L_bits, double R_bps, double dprop) {
    return N * (L_bits / R_bps + dprop);         // each hop: store fully, then send
}
double e2e_many(int P, int N, double L_bits, double R_bps) {
    return (N + P - 1) * (L_bits / R_bps);        // pipeline: fill (N) + drain (P - 1)
}
}
```

</details>

**Q2.5** *(Compute)* — Instead of 4 packets of 1500 B, send the same 6000 B as ONE packet over the same 3 links
(10 Mb/s, ignore propagation). How long does it take, and what does this say about packet size?

<details>
<summary><b>Solution</b> (click to expand)</summary>

$3 \cdot \dfrac{48000}{10^7}$ = **14.4 ms**, versus 7.2 ms when split into 4 packets.
Smaller packets let the links pipeline. They are not free, though: every packet carries a header, so
tiny packets waste capacity on overhead. Real networks settle around 1500 B, a value you will meet again
in Part 5.

</details>

## 2.7 Round-trip time and the bandwidth-delay product

The **round-trip time (RTT)** is the time from sending a message until its reply comes back:
at least twice the one-way propagation delay, plus the transmission, queuing and processing delays.

The **bandwidth-delay product** $\text{BDP} = R \cdot \text{RTT}$ is the amount of data that
"fits in the pipe": the number of bits a sender must have sent, but not yet seen acknowledged,
to keep the link busy while it waits for the first reply.

```
sender ===[b][b][b][b][b][b][b][b][b][b]===> receiver     (bits in flight, one direction)
       <==========  acknowledgement returns  ============
       |<------------------ RTT ---------------------->|
```

If a sender limits itself to $W$ unacknowledged bytes per RTT, its throughput is at most
$W \cdot 8 / \text{RTT}$ bits/s, no matter how fast the link is.

### ✍️ Exercise 2.7 — BDP and window-limited throughput

* `bdp_bytes(R_bps, rtt_s)`: the BDP in **bytes**.
* `max_throughput_bps(window_bytes, rtt_s)`: the throughput cap of a window-limited sender.

In [28]:
namespace ex2_7 {
double bdp_bytes(double R_bps, double rtt_s)                 { return R_bps * rtt_s / 8.0; }
double max_throughput_bps(double window_bytes, double rtt_s) { return window_bytes * 8.0 / rtt_s; }
}

**Test for Exercise 2.7** — run the next cell.

In [29]:
{
    CHECK_NEAR(ex2_7::bdp_bytes(1e9, 0.080), 10000000.0);
    CHECK_NEAR(ex2_7::max_throughput_bps(64 * 1024, 0.080), 6553600.0);
    std::cout << "64 KiB window over 80 ms RTT caps throughput at "
              << ex2_7::max_throughput_bps(64 * 1024, 0.080) / 1e6 << " Mb/s\n";
}

✅ ex2_7::bdp_bytes(1e9, 0.080) ≈ 10000000.0
✅ ex2_7::max_throughput_bps(64 * 1024, 0.080) ≈ 6553600.0
64 KiB window over 80 ms RTT caps throughput at 6.5536 Mb/s


<details>
<summary><b>Solution</b> (click to expand)</summary>

A 1 Gb/s path with 80 ms RTT holds 10 MB in flight. A sender that allows only 64 KiB outstanding
reaches just 6.55 Mb/s, under 1% of the link. Protocols that wait for acknowledgements therefore need
large windows on "long fat" paths.

```cpp
namespace ex2_7 {
double bdp_bytes(double R_bps, double rtt_s)                 { return R_bps * rtt_s / 8.0; }
double max_throughput_bps(double window_bytes, double rtt_s) { return window_bytes * 8.0 / rtt_s; }
}
```

</details>

**Q2.6** *(Compute)* — A transfer between Europe and the US uses a 10 Gb/s path with RTT 100 ms. How many bytes must be in flight
to use the full rate? Would a single 1500-byte packet at a time come close?

<details>
<summary><b>Solution</b> (click to expand)</summary>

$\text{BDP} = 10^{10} \cdot 0.1 / 8$ = **125 MB**. One 1500-byte packet per RTT gives
$1500\cdot8/0.1$ = 120 kb/s, a 83,333× shortfall. Long, fast paths need
many packets outstanding at once.

</details>

### 🔁 Recap

- Hosts and packet switches are connected by links; data travels in packets = header + payload.
- Packet switching shares links statistically (store-and-forward, queues); circuit switching reserves capacity.
- Four per-hop delays: processing, queuing, transmission $L/R$, and propagation $d/s$. Bandwidth only shrinks $L/R$.
- Queuing explodes as the traffic intensity $La/R \to 1$; a full buffer means packet loss.
- BDP $= R\cdot\text{RTT}$ is the data needed in flight to fill a path; window/RTT caps throughput.

**You should now be able to explain…**

- [ ] the difference between bandwidth and latency, with a concrete example;
- [ ] why a 10× faster link barely helps a transcontinental ping;
- [ ] why store-and-forward makes small packets pipeline better;
- [ ] how much data must be outstanding to fill a 1 Gb/s, 80 ms path.

# Part 3 — Layering

## 3.1 Why layers exist

A **protocol** is an agreement on the *format* of messages, their *order*, and the *actions*
taken on sending or receiving them. Networking needs dozens of protocols, and **layering** keeps
them manageable. Each layer:

* offers a **service** to the layer above (e.g. "deliver these bytes to that host");
* uses only the service of the layer below, without knowing how that layer works;
* talks to its **peer**, the same layer on the other machine, through its own header.

```
host A                                   host B
layer n   <------ peer protocol ------>  layer n
   | uses the service of the layer below    ^
   v                                        |
layer n-1 <------ peer protocol ------>  layer n-1
```

Postal analogy: you write a letter (application), the post office routes envelopes between cities
(network), and trucks and planes move sacks (link and physical). The letter's author never sees a truck.

**Q3.1** *(Why?)* — Give one major benefit and one real cost of layering.

<details>
<summary><b>Solution</b> (click to expand)</summary>

**Benefit, modularity:** any layer can be replaced without touching the others. The same web browser
works over Wi-Fi, Ethernet, or 5G because only the lowest layers differ.

**Costs:** (1) *overhead*: every layer adds its own header bytes; (2) *information hiding*: a layer
cannot see what it might need. A web browser does not know the link is a lossy radio, and the radio does
not know which bytes are urgent. (3) Some functions get *duplicated* at several layers (error detection
exists at layers 2 and 4). *Misconception:* layers are a law of nature; in fact they are an engineering
convention that real systems sometimes bend for performance.

</details>

## 3.2 The OSI 7-layer model

The **OSI** (Open Systems Interconnection) reference model names seven layers. Engineers still
say "layer 2" or "an L7 load balancer" using this numbering. The **PDU** (protocol data unit)
is the name of a layer's chunk of data.

| # | Layer | One-line job | PDU | Example protocols |
|---|---|---|---|---|
| 7 | Application | the service a program wants | data (message) | HTTP (fetch web pages), DNS (look up names) |
| 6 | Presentation | how data is represented: encoding, compression, encryption | data | UTF-8, JPEG |
| 5 | Session | managing a long dialogue: open, checkpoint, resume | data | RPC session control |
| 4 | Transport | deliver data between **programs** on two hosts | segment (TCP) / datagram (UDP) | TCP (reliable byte stream), UDP (single messages) |
| 3 | Network | deliver packets between **hosts** across many networks | packet | IP (Internet Protocol) |
| 2 | Data link | deliver frames between **neighbours** on one link | frame | Ethernet (wired), Wi-Fi |
| 1 | Physical | turn bits into signals on a medium | bits | copper, fibre, radio signalling |

Mnemonic, bottom-up: **P**lease **D**o **N**ot **T**hrow **S**ausage **P**izza **A**way.

Each layer has its own address:
* a **MAC address** (L2) is a 48-bit identifier of a network interface, meaningful on the local link only;
* an **IP address** (L3) is a 32-bit identifier of a host's interface, meaningful across the Internet;
* a **port** (L4) is a 16-bit number identifying which program on a host the data is for.

**Q3.2** *(Concept)* — At which OSI layer does each live: (a) a MAC address, (b) an IP address, (c) a port number?
And what is the PDU called at layers 2, 3, and 4?

<details>
<summary><b>Solution</b> (click to expand)</summary>

(a) **L2** Data link, (b) **L3** Network, (c) **L4** Transport. PDUs: L2 **frame**, L3 **packet**,
L4 **segment** (TCP) or **datagram** (UDP).

The layers answer successive questions: *which neighbour on this wire?* (MAC), *which host on the
Internet?* (IP), *which program on that host?* (port). *Misconception:* people say "packet" for every
layer. In an interview, "frame" vs "packet" vs "segment" signals which header you are talking about.

</details>

## 3.3 The TCP/IP model and how it maps onto OSI

The Internet was not built from OSI; it follows the simpler **TCP/IP model** with four layers:

```
     OSI                      TCP/IP (Internet) model
7  Application   ┐
6  Presentation  ├───────►  Application   HTTP, DNS    (programs handle 5-7 themselves)
5  Session       ┘
4  Transport     ────────►  Transport     TCP, UDP
3  Network       ────────►  Internet      IP
2  Data link     ┐
1  Physical      ┘───────►  Link          Ethernet, Wi-Fi (framing + signalling)
```

Many textbooks use a 5-layer hybrid (keeping Physical separate). OSI is the *vocabulary*;
TCP/IP is what actually runs. An operating system implements Transport, Internet and part of Link;
the NIC hardware (Part 4) implements Physical and part of Link; applications implement the rest.

**Q3.3** *(Why?)* — OSI's Session and Presentation layers have no separate TCP/IP layer. Where did their jobs go?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Into the **application** and the libraries it links: character encoding, compression, serialization
formats (JSON, Protobuf), and encryption libraries all run inside the application process. Session-like
behaviour (logins, resuming a download) is part of the application protocol itself. *Misconception:*
"those layers do not exist". The functions exist; they simply are not a separate, standardised layer in the
OS network stack.

</details>

## 3.4 Encapsulation and decapsulation

Going **down** the stack, each layer treats everything it receives from above as an opaque
**payload** and prepends its own **header**. The link layer also appends a **trailer** (an error
check). This is **encapsulation**:

```
Application                               [       data        ]
Transport                       [TCP hdr ][       data        ]              <- segment
Network               [IP hdr  ][TCP hdr ][       data        ]              <- packet
Link        [Eth hdr ][IP hdr  ][TCP hdr ][       data        ][ trailer ]   <- frame
Physical    0110100101110001011010 ... bits as signals on the medium
```

Going **up** on the receiver, each layer checks and strips its own header, then hands the
payload to the layer above: **decapsulation**. Every header contains a field naming the protocol
of its payload (for example "this IP packet carries TCP"), so the receiver knows which upper-layer
code to call. Dispatching on that field is **demultiplexing**.

**Q3.4** *(Compute)* — A 100-byte application message goes out over TCP (20-byte header), IP (20-byte header), and
Ethernet (14-byte header + 4-byte trailer). How big is the frame, and what fraction of it is application data?

<details>
<summary><b>Solution</b> (click to expand)</summary>

100 + 20 + 20 + 14 + 4 = **158 bytes**; efficiency = 100/158 = **63.3%**.

Headers are a fixed cost per packet, so small messages are inefficient. With a 1460-byte message the
efficiency is 96.2%. This is one reason chatty protocols batch small writes.

</details>

**Q3.5** *(Why?)* — Why must each header carry a "type of my payload" field? Couldn't the receiver just guess from the bytes?

<details>
<summary><b>Solution</b> (click to expand)</summary>

The payload is just bytes, and the same bytes could be a valid start of many different protocols, so guessing
is ambiguous and fragile. An IP layer may receive TCP, UDP or control messages; Ethernet may carry IPv4,
IPv6 or other protocols. An explicit type field makes demultiplexing a simple table lookup and keeps the
layers independent, since the lower layer never parses the upper header.

</details>

### A toy protocol stack

To practise, use a toy format in which **every** layer's header is 3 bytes:

```
offset:  0          1   2
        +----------+--------------------+
        |  tag     |  payload length    |   length = number of bytes AFTER the header,
        | (1 byte) | (uint16, big-end.) |            in network byte order
        +----------+--------------------+
tags:  'T' transport     'N' network     'L' link
```

Your code needs the byte-order helpers from Part 1; they are provided in the next cell.

In [30]:
namespace lib3 {

// PROVIDED (you wrote this in §1.6)
uint16_t read_be16(const uint8_t* p) {
    return uint16_t((uint16_t(p[0]) << 8) | p[1]);
}
uint32_t read_be32(const uint8_t* p) {
    return (uint32_t(p[0]) << 24) | (uint32_t(p[1]) << 16) |
           (uint32_t(p[2]) << 8)  |  uint32_t(p[3]);
}
void write_be16(uint8_t* p, uint16_t v) {
    p[0] = uint8_t(v >> 8);      // most significant byte first
    p[1] = uint8_t(v & 0xFF);
}
void write_be32(uint8_t* p, uint32_t v) {
    p[0] = uint8_t(v >> 24);
    p[1] = uint8_t(v >> 16);     // the uint8_t cast keeps only the low 8 bits
    p[2] = uint8_t(v >> 8);
    p[3] = uint8_t(v);
}

} // namespace lib3

### ✍️ Exercise 3.4a — Add and strip one toy header

* `add_header(tag, payload)`: return `tag`, then the 2-byte big-endian length, then the payload.
* `strip_header(pdu, tag, payload_out)`: if `pdu` is at least 3 bytes long, starts with `tag`, **and** its
  length field equals the number of bytes that follow, copy those bytes into `payload_out` and return `true`;
  otherwise return `false`.

`std::vector<uint8_t>` is the standard growable byte buffer: `v.push_back(b)`, `v.size()`,
`v.insert(v.end(), other.begin(), other.end())` appends another vector, and `v.data()` gives a
`uint8_t*` to its bytes.

In [31]:
namespace ex3_4a {
std::vector<uint8_t> add_header(uint8_t tag, const std::vector<uint8_t>& payload) {
    std::vector<uint8_t> out(3);                         // room for the header
    out[0] = tag;
    lib3::write_be16(&out[1], uint16_t(payload.size()));
    out.insert(out.end(), payload.begin(), payload.end());
    return out;
}
bool strip_header(const std::vector<uint8_t>& pdu, uint8_t tag, std::vector<uint8_t>& payload_out) {
    if (pdu.size() < 3 || pdu[0] != tag) return false;
    uint16_t len = lib3::read_be16(&pdu[1]);
    if (len != pdu.size() - 3) return false;            // truncated or trailing garbage
    payload_out.assign(pdu.begin() + 3, pdu.end());
    return true;
}
}

**Test for Exercise 3.4a** — run the next cell.

In [32]:
{
    std::vector<uint8_t> payload = {'h', 'i'};
    auto t = ex3_4a::add_header('T', payload);
    CHECK_EQ(t, (std::vector<uint8_t>{'T', 0x00, 0x02, 'h', 'i'}));

    std::vector<uint8_t> back;
    CHECK(ex3_4a::strip_header(t, 'T', back));
    CHECK_EQ(back, payload);
    CHECK(!ex3_4a::strip_header(t, 'N', back));                                  // wrong tag
    CHECK(!ex3_4a::strip_header(std::vector<uint8_t>{'T', 0x00, 0x05, 'h'}, 'T', back)); // truncated
    CHECK(!ex3_4a::strip_header(std::vector<uint8_t>{'T', 0x00}, 'T', back));   // too short for a header
}

✅ t == (std::vector<uint8_t>{'T', 0x00, 0x02, 'h', 'i'})
✅ ex3_4a::strip_header(t, 'T', back)
✅ back == payload
✅ !ex3_4a::strip_header(t, 'N', back)
✅ !ex3_4a::strip_header(std::vector<uint8_t>{'T', 0x00, 0x05, 'h'}, 'T', back)
✅ !ex3_4a::strip_header(std::vector<uint8_t>{'T', 0x00}, 'T', back)


<details>
<summary><b>Solution</b> (click to expand)</summary>

Validating the length field against the real size is not pedantry: a receiver that trusts a length field
blindly reads past the end of its buffer. That is exactly how famous memory-disclosure bugs in network
code happen. Always check that a header fits **before** reading it, and that its length claims are consistent.

```cpp
namespace ex3_4a {
std::vector<uint8_t> add_header(uint8_t tag, const std::vector<uint8_t>& payload) {
    std::vector<uint8_t> out(3);                         // room for the header
    out[0] = tag;
    lib3::write_be16(&out[1], uint16_t(payload.size()));
    out.insert(out.end(), payload.begin(), payload.end());
    return out;
}
bool strip_header(const std::vector<uint8_t>& pdu, uint8_t tag, std::vector<uint8_t>& payload_out) {
    if (pdu.size() < 3 || pdu[0] != tag) return false;
    uint16_t len = lib3::read_be16(&pdu[1]);
    if (len != pdu.size() - 3) return false;            // truncated or trailing garbage
    payload_out.assign(pdu.begin() + 3, pdu.end());
    return true;
}
}
```

</details>

In [33]:
namespace lib3b {

// PROVIDED (you wrote this in §3.4a)
std::vector<uint8_t> add_header(uint8_t tag, const std::vector<uint8_t>& payload) {
    std::vector<uint8_t> out(3);                         // room for the header
    out[0] = tag;
    lib3::write_be16(&out[1], uint16_t(payload.size()));
    out.insert(out.end(), payload.begin(), payload.end());
    return out;
}
bool strip_header(const std::vector<uint8_t>& pdu, uint8_t tag, std::vector<uint8_t>& payload_out) {
    if (pdu.size() < 3 || pdu[0] != tag) return false;
    uint16_t len = lib3::read_be16(&pdu[1]);
    if (len != pdu.size() - 3) return false;            // truncated or trailing garbage
    payload_out.assign(pdu.begin() + 3, pdu.end());
    return true;
}

} // namespace lib3b

### ✍️ Exercise 3.4b — Encapsulate and decapsulate a message

* `encapsulate(msg)`: wrap the message's bytes with `'T'`, then `'N'`, then `'L'` (outermost).
* `decapsulate(frame)`: undo it, strictly in the order L, N, T, and return the message.

`decapsulate` returns `std::optional<std::string>` (from `<optional>`): an object that holds either a value
or nothing. Return `std::nullopt` for "nothing" (any layer failed) or the string itself on success. The caller
tests it with `if (r)` and reads it with `*r`. Use `lib3b::add_header` / `lib3b::strip_header` (PROVIDED above).

For `"hi"` the frame is 11 bytes:
```
0000  4c 00 08 4e 00 05 54 00  02 68 69                 |L..N..T..hi|
```

In [34]:
namespace ex3_4b {
std::vector<uint8_t> encapsulate(const std::string& msg) {
    std::vector<uint8_t> data(msg.begin(), msg.end());      // application data as bytes
    auto segment = lib3b::add_header('T', data);           // transport
    auto packet  = lib3b::add_header('N', segment);        // network
    return lib3b::add_header('L', packet);                 // link (outermost)
}
std::optional<std::string> decapsulate(const std::vector<uint8_t>& frame) {
    std::vector<uint8_t> packet, segment, data;
    if (!lib3b::strip_header(frame, 'L', packet))   return std::nullopt;
    if (!lib3b::strip_header(packet, 'N', segment)) return std::nullopt;
    if (!lib3b::strip_header(segment, 'T', data))   return std::nullopt;
    return std::string(data.begin(), data.end());
}
}

**Test for Exercise 3.4b** — run the next cell.

In [35]:
{
    auto f = ex3_4b::encapsulate("hi");
    hexdump(f);
    CHECK_EQ(f, (std::vector<uint8_t>{
        0x4c, 0x00, 0x08, 0x4e, 0x00, 0x05, 0x54, 0x00, 0x02, 0x68, 0x69,
    }));
    auto m = ex3_4b::decapsulate(f);
    CHECK(m.has_value());
    CHECK_EQ(m.value_or("<nothing>"), "hi");

    auto big = ex3_4b::encapsulate(std::string(300, 'x'));
    CHECK_EQ(big.size(), 309u);
    CHECK_EQ(ex3_4b::decapsulate(big).value_or("<nothing>").size(), 300u);

    auto bad = f;
    if (bad.size() > 3) bad[3] = 'X';                        // corrupt the network tag
    CHECK(!ex3_4b::decapsulate(bad).has_value());
}

0000  4c 00 08 4e 00 05 54 00  02 68 69                 |L..N..T..hi|
✅ f == (std::vector<uint8_t>{ 0x4c, 0x00, 0x08, 0x4e, 0x00, 0x05, 0x54, 0x00, 0x02, 0x68, 0x69, })
✅ m.has_value()
✅ m.value_or("<nothing>") == "hi"
✅ big.size() == 309u
✅ ex3_4b::decapsulate(big).value_or("<nothing>").size() == 300u
✅ !ex3_4b::decapsulate(bad).has_value()


<details>
<summary><b>Solution</b> (click to expand)</summary>

The byte layout makes the nesting visible: `L 00 08 | N 00 05 | T 00 02 | h i`. Each length counts
everything **inside** it, including inner headers. The order matters: decapsulation strictly reverses
encapsulation, and a failure at any layer discards the whole frame. That is what real stacks do with a
malformed header.

(The corruption test in the test cell guards with `if (bad.size() > 3)` so that the *unsolved* stub,
which returns an empty vector, cannot index out of bounds.)

```cpp
namespace ex3_4b {
std::vector<uint8_t> encapsulate(const std::string& msg) {
    std::vector<uint8_t> data(msg.begin(), msg.end());      // application data as bytes
    auto segment = lib3b::add_header('T', data);           // transport
    auto packet  = lib3b::add_header('N', segment);        // network
    return lib3b::add_header('L', packet);                 // link (outermost)
}
std::optional<std::string> decapsulate(const std::vector<uint8_t>& frame) {
    std::vector<uint8_t> packet, segment, data;
    if (!lib3b::strip_header(frame, 'L', packet))   return std::nullopt;
    if (!lib3b::strip_header(packet, 'N', segment)) return std::nullopt;
    if (!lib3b::strip_header(segment, 'T', data))   return std::nullopt;
    return std::string(data.begin(), data.end());
}
}
```

</details>

### 🔁 Recap

- A protocol defines message format, order, and actions; layers each offer a service upward and talk to their peer.
- OSI: Physical, Data link, Network, Transport, Session, Presentation, Application (PDUs: bits, frame, packet, segment/datagram, data).
- TCP/IP folds OSI 5–7 into Application and 1–2 into Link.
- Encapsulation prepends a header per layer (plus a link trailer); decapsulation strips them in reverse.
- A 'next protocol' field in each header drives demultiplexing.

**You should now be able to explain…**

- [ ] what lives at L2, L3 and L4, and each layer's PDU name;
- [ ] how the OSI and TCP/IP models map onto each other;
- [ ] the byte overhead of headers for a small message;
- [ ] why a parser must check a length field before trusting it.

# Part 4 — Physical layer and hardware

## 4.1 Transmission media

The **medium** is what physically carries the signal:

| medium | signal | typical use | notes |
|---|---|---|---|
| twisted-pair copper | voltage | Ethernet cables in homes and offices | ≤ 100 m per cable run; pairs are twisted to cancel interference |
| coaxial copper | voltage (radio frequencies) | cable TV / cable Internet | shielded; shared by a whole neighbourhood |
| optical fibre | light pulses | long haul, data centres, fibre-to-the-home | km to 100 km+ per span; immune to electrical noise |
| radio | electromagnetic waves | Wi-Fi, cellular | shared by everyone in range; interference; no cable |

Guided media (copper, fibre) confine the signal to a cable. Radio is **broadcast** by nature:
every receiver in range hears every transmission.

**Q4.1** *(Why?)* — Why do long-distance and data-centre links use fibre rather than copper, even though the signal
speed in both is about ⅔ of *c*?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Not for *propagation* speed, which is similar. Fibre wins on **attenuation and bandwidth**: light in glass
loses very little power per km and supports enormous rates, so spans of tens of km need no amplifier, while
copper at high rates degrades within metres to about 100 m. Fibre also ignores electromagnetic interference
and cannot be tapped by induction. *Misconception:* "fibre is faster because light is faster than
electricity". Both signals travel at roughly 2·10⁸ m/s.

</details>

## 4.2 Bits as signals: symbols and baud

The transmitter sends one **symbol** per time slot. A symbol is one distinguishable signal
state, such as a voltage level or a light intensity. The **symbol rate** is measured in **baud**
(symbols per second). With $M$ distinguishable levels a symbol carries $\log_2 M$ bits, so

$$\text{bit rate} = \text{baud} \times \log_2 M.$$

```
M = 2 levels (1 bit/symbol)        M = 4 levels (2 bits/symbol)
+1 ‾‾‾|   |‾‾‾                     +3 ‾‾‾|        11
      |   |                        +1    |‾‾‾|    10
-1    |___|                        -1        |    01
                                   -3        |___ 00
```

**Noise** (random disturbance added by the medium and electronics) blurs the received level.
When levels are closer together, a smaller disturbance turns one symbol into its neighbour, and
the result is a **bit error**.

**Q4.2** *(Compute)* — A link sends 125 Mbaud with 4 levels per symbol on each of 4 wire pairs in parallel.
What is the total bit rate? And why not simply use 1024 levels to go 5× faster?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Per pair: 125·10⁶ × log₂4 = 250 Mb/s; four pairs: **1000 Mb/s**. This is
roughly how gigabit Ethernet over copper works; the real scheme is more elaborate, and some levels
carry error-correction redundancy rather than data.

With 1024 levels (10 bits/symbol) at the same maximum voltage, adjacent levels are much closer than with
4 levels (341×), so ordinary noise causes constant symbol errors. Noise, not cleverness, limits bits per symbol.

</details>

## 4.3 Line coding and clock recovery

The receiver must also know **when** to sample each symbol. Its clock drifts relative to the
sender's, so it re-synchronises on signal **transitions** (changes of level). In plain
**NRZ** coding (non-return-to-zero: high = 1, low = 0) a long run of identical bits has no
transitions, and the receiver loses count of how many bits went by.

**Manchester coding**, used by 10 Mb/s Ethernet, guarantees a transition in the **middle of
every bit**. Each bit becomes two half-bit levels; IEEE 802.3 (the Ethernet standard) sends
`1` as low→high and `0` as high→low:

```
bits:          1       0       1       1       0
NRZ:        ‾‾‾‾‾‾‾|_______|‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾|_______
Manchester: ___|‾‾‾ ‾‾‾|___ ___|‾‾‾ ___|‾‾‾ ‾‾‾|___
halves:      0   1   1   0   0   1   0   1   1   0
```

The price is two signal halves per bit, i.e. double the symbol rate. Faster Ethernets use
cheaper codes (e.g. 64b/66b) that add only a few extra bits per block to guarantee transitions.

### ✍️ Exercise 4.3 — Manchester encoder and decoder

* `manchester_encode(b)`: return 16 half-bit levels (`1` = high, `0` = low) for the byte `b`, **most
  significant bit first**. (Real Ethernet transmits each byte least-significant bit first; MSB-first keeps this
  exercise readable.)
* `manchester_decode(levels)`: the inverse. Return `std::nullopt` if `levels.size() != 16` or any pair is
  `00` or `11`: such a pair has no mid-bit transition, so it is a coding violation.

For `0xA5` the expected output is `0110011010011001`.

In [36]:
namespace ex4_3 {
std::vector<uint8_t> manchester_encode(uint8_t b) {
    std::vector<uint8_t> out;
    for (int i = 7; i >= 0; --i) {             // MSB first
        bool bit = (b >> i) & 1;
        if (bit) { out.push_back(0); out.push_back(1); }   // 1: low -> high
        else     { out.push_back(1); out.push_back(0); }   // 0: high -> low
    }
    return out;
}
std::optional<uint8_t> manchester_decode(const std::vector<uint8_t>& levels) {
    if (levels.size() != 16) return std::nullopt;
    uint8_t b = 0;
    for (int i = 0; i < 8; ++i) {
        uint8_t first = levels[2 * i], second = levels[2 * i + 1];
        if (first == second) return std::nullopt;         // no transition: violation
        b = uint8_t((b << 1) | (second == 1 ? 1 : 0));    // low->high means 1
    }
    return b;
}
}

**Test for Exercise 4.3** — run the next cell.

In [37]:
{
    auto enc = ex4_3::manchester_encode(0xA5);
    CHECK_EQ(enc, (std::vector<uint8_t>{0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1}));
    CHECK_EQ(ex4_3::manchester_encode(0x00).size(), 16u);
    CHECK_EQ(int(ex4_3::manchester_decode(enc).value_or(0)), 0xA5);
    bool all_round_trip = true;
    for (int v = 0; v < 256; ++v) {
        auto d = ex4_3::manchester_decode(ex4_3::manchester_encode(uint8_t(v)));
        if (!d || *d != v) all_round_trip = false;
    }
    CHECK(all_round_trip);
    auto bad = enc;
    if (bad.size() == 16) bad[3] = bad[2];          // destroy one mid-bit transition
    CHECK(!ex4_3::manchester_decode(bad).has_value());
}

✅ enc == (std::vector<uint8_t>{0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1})
✅ ex4_3::manchester_encode(0x00).size() == 16u
✅ int(ex4_3::manchester_decode(enc).value_or(0)) == 0xA5
✅ all_round_trip
✅ !ex4_3::manchester_decode(bad).has_value()


<details>
<summary><b>Solution</b> (click to expand)</summary>

Every bit produces exactly one mid-bit transition, so even `0x00` or `0xFF` (long runs in NRZ) toggles 8 times
per byte. Rejecting `00`/`11` pairs is a free error check: a single corrupted half-bit usually produces an
impossible pair. Note the direct link to §1.3: the encoder is just "test bit *i*, most significant first".

```cpp
namespace ex4_3 {
std::vector<uint8_t> manchester_encode(uint8_t b) {
    std::vector<uint8_t> out;
    for (int i = 7; i >= 0; --i) {             // MSB first
        bool bit = (b >> i) & 1;
        if (bit) { out.push_back(0); out.push_back(1); }   // 1: low -> high
        else     { out.push_back(1); out.push_back(0); }   // 0: high -> low
    }
    return out;
}
std::optional<uint8_t> manchester_decode(const std::vector<uint8_t>& levels) {
    if (levels.size() != 16) return std::nullopt;
    uint8_t b = 0;
    for (int i = 0; i < 8; ++i) {
        uint8_t first = levels[2 * i], second = levels[2 * i + 1];
        if (first == second) return std::nullopt;         // no transition: violation
        b = uint8_t((b << 1) | (second == 1 ? 1 : 0));    // low->high means 1
    }
    return b;
}
}
```

</details>

## 4.4 The NIC: network interface controller

A **NIC** (network interface controller, or card) is the hardware that attaches a host to a
link. It has two halves: the **PHY** chip (physical layer: converts between bits and signals,
including line coding) and the **MAC** logic (link layer: frames, addresses, error checks). The NIC:

* **serialises**: turns the bytes of an outgoing frame into a timed sequence of symbols, and
  *deserialises* incoming signals back into bytes;
* **owns a MAC address**: a 48-bit identifier programmed at the factory, written as six hex
  bytes, e.g. `3c:22:fb:12:ab:9e`;
* **filters**: keeps frames addressed to its own MAC address (or to "everyone") and silently
  ignores the rest; *promiscuous mode*, used by packet sniffers, disables this filter;
* **checks** each frame's error-detection trailer and drops corrupted frames;
* **moves frames to and from host memory** without the CPU copying each byte, which is the next topic.

**Q4.3** *(Concept)* — You plug a laptop into a busy network and run a packet sniffer *without* promiscuous mode.
Which frames does the operating system get to see?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Only frames the NIC accepted: those addressed to **its own MAC address**, plus frames addressed to everyone
or to groups it has joined, plus the frames it sends itself. Other hosts' traffic is dropped **in hardware**
before the OS ever sees it. On a switched network (§4.7) most of it never reaches your port anyway.
*Misconception:* "the OS receives everything and discards most of it". The filter is in the NIC, which is
why it can keep up with line rate.

</details>

## 4.5 DMA and ring buffers

At 10 Gb/s a CPU cannot afford to copy every byte itself. With **DMA** (direct memory access)
the NIC reads and writes host RAM on its own. The **driver** (the OS code controlling the NIC)
and the NIC share a **ring buffer**: a fixed-size circular array of *descriptors*, each pointing
at a packet buffer in RAM.

On receive, the NIC DMA-writes a frame into the slot at **head** and advances head; it then
raises an **interrupt** (a hardware signal telling the CPU "work is ready"). The driver consumes
frames from **tail**. If the driver falls behind and the ring is full, the NIC must **drop**
new frames.

```
          tail (driver reads next)       head (NIC writes next)
             v                              v
    +-----+-----+-----+-----+-----+-----+-----+-----+
    |     |  p3 |  p4 |  p5 |  p6 |  p7 |     |     |    N = 8 slots
    +-----+-----+-----+-----+-----+-----+-----+-----+
    index advance with wrap-around:  i = (i + 1) & (N - 1)    (N a power of two)
```

### ✍️ Exercise 4.5 — A receive ring

Complete `RxRing`. Track fullness with `count`. `nic_receive` stores a packet id at `head`, or, if the ring
is full, increments `dropped` and returns `false`. `driver_poll` returns the oldest packet, or `std::nullopt`
when the ring is empty. Advance indices with the mask trick shown above.
(`std::array<int, N>` is a fixed-size array; `static constexpr` makes `N` a compile-time constant.)

In [38]:
namespace ex4_5 {
struct RxRing {
    static constexpr size_t N = 8;     // must be a power of two
    std::array<int, N> slot{};         // stands in for descriptors -> packet buffers
    size_t head = 0, tail = 0, count = 0;
    uint64_t dropped = 0;

    bool nic_receive(int pkt) {
        if (count == N) { ++dropped; return false; }   // ring full: the NIC drops
        slot[head] = pkt;
        head = (head + 1) & (N - 1);                    // same as % N, but one AND
        ++count;
        return true;
    }
    std::optional<int> driver_poll() {
        if (count == 0) return std::nullopt;
        int pkt = slot[tail];
        tail = (tail + 1) & (N - 1);
        --count;
        return pkt;
    }
};
}

**Test for Exercise 4.5** — run the next cell.

In [39]:
{
    ex4_5::RxRing ring;
    int accepted = 0;
    for (int p = 0; p < 10; ++p) accepted += ring.nic_receive(p);   // burst of 10 frames
    CHECK_EQ(accepted, 8);
    CHECK_EQ(ring.dropped, 2u);

    std::vector<int> first;
    for (int i = 0; i < 3; ++i) first.push_back(ring.driver_poll().value_or(-1));
    CHECK_EQ(first, (std::vector<int>{0, 1, 2}));

    int accepted2 = 0;
    for (int p = 10; p < 13; ++p) accepted2 += ring.nic_receive(p);  // wraps around the end
    CHECK_EQ(accepted2, 3);

    std::vector<int> rest;
    for (int i = 0; i < 20; ++i) {          // bounded loop: never spins forever
        auto p = ring.driver_poll();
        if (!p) break;
        rest.push_back(*p);
    }
    CHECK_EQ(rest, (std::vector<int>{3, 4, 5, 6, 7, 10, 11, 12}));
}

✅ accepted == 8
✅ ring.dropped == 2u
✅ first == (std::vector<int>{0, 1, 2})
✅ accepted2 == 3
✅ rest == (std::vector<int>{3, 4, 5, 6, 7, 10, 11, 12})


<details>
<summary><b>Solution</b> (click to expand)</summary>

The first burst fills all 8 slots, and frames 8 and 9 are dropped **by the NIC**: the OS never learns
their contents, only a "missed" counter increments (visible on Linux with `ip -s link`). After the driver
frees 3 slots, frames 10–12 wrap around to indices 0–2, and the order is still preserved: `3, 4, 5, 6, 7, 10, 11, 12`.
Using a power-of-two size turns `% N` into a single AND, the same mask idea as §1.3.

```cpp
namespace ex4_5 {
struct RxRing {
    static constexpr size_t N = 8;     // must be a power of two
    std::array<int, N> slot{};         // stands in for descriptors -> packet buffers
    size_t head = 0, tail = 0, count = 0;
    uint64_t dropped = 0;

    bool nic_receive(int pkt) {
        if (count == N) { ++dropped; return false; }   // ring full: the NIC drops
        slot[head] = pkt;
        head = (head + 1) & (N - 1);                    // same as % N, but one AND
        ++count;
        return true;
    }
    std::optional<int> driver_poll() {
        if (count == 0) return std::nullopt;
        int pkt = slot[tail];
        tail = (tail + 1) & (N - 1);
        --count;
        return pkt;
    }
};
}
```

</details>

**Q4.4** *(Why?)* — A server's NIC reports thousands of "rx missed" drops per second, yet the link is only 30% utilised.
What is the likely bottleneck?

<details>
<summary><b>Solution</b> (click to expand)</summary>

The **host**, not the wire: the driver/CPU is not emptying the receive ring fast enough (too few CPU cores
handling interrupts, a ring that is too small for bursts, or an application that is too slow). Average
utilisation hides **bursts**: a microsecond-scale burst can fill a small ring even at 30% average load, which
is the queuing lesson of §2.5 again. *Misconception:* "drops mean the network is congested". Here the queue
that overflowed is inside your own machine.

</details>

## 4.6 The modem: the edge of your home network

A **modem** (modulator–demodulator) converts between digital bits and a signal suited to a
medium that was *not* built for Ethernet. **Modulation** encodes bits by varying a carrier wave's
amplitude, frequency, or phase; **demodulation** recovers them.

* **DSL modem**: over the telephone company's copper pair.
* **Cable modem**: over the cable-TV coaxial network, following the DOCSIS standard.
* **ONT** (optical network terminal): for fibre-to-the-home, converts between light on the
  provider's fibre and Ethernet in your home.

The modem is the **edge device** between your home network and the **ISP**'s (Internet service
provider's) access network. It works at layers 1–2: it converts signals and frames, but it makes
no routing decisions.

**Q4.5** *(Why?)* — Why not simply run an Ethernet cable from your house to the ISP instead of using a modem?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Twisted-pair Ethernet is specified for **≤ 100 m**, while the ISP's equipment is often kilometres away.
ISPs also reuse wiring that already exists (phone copper, TV coax, shared fibre), which uses different
signalling and is shared among many homes. The modem adapts your standard Ethernet to that medium and to the
ISP's access protocol. *Misconception:* "the modem is the router". Often they share one box, but modulation
(L1/L2) and routing (L3) are different jobs.

</details>

## 4.7 Hub vs switch vs router vs access point

* **Hub** (L1, obsolete): repeats every incoming bit out of *all* other ports. All hosts share
  the bandwidth, and simultaneous senders **collide** (their signals overlap and are garbled).
* **Switch** (L2): reads each frame's destination MAC address and forwards the frame only out of
  the port where that address lives.
* **Router** (L3): connects *different* networks. It reads the destination IP address, picks the
  next hop, and forwards the packet inside a **new** L2 frame for the next link.
* **Access point (AP)** (L2): bridges frames between Wi-Fi radio clients and wired Ethernet.

A **LAN** (local area network) is a set of hosts that reach each other directly through
switches and APs, with no router in between. Routers connect LANs to each other and to the Internet.

```
  host A            switch            router            host B
 [ L7 app ]                                            [ L7 app ]
 [ L4     ]                                            [ L4     ]
 [ L3     ]                          [ L3     ]        [ L3     ]
 [ L2     ]        [ L2     ]        [ L2     ]        [ L2     ]
 [ L1     ]~~~~~~~~[ L1     ]~~~~~~~~[ L1     ]~~~~~~~~[ L1     ]
```
Each device decapsulates only as high as its job requires.

**Q4.6** *(Predict the output)* — A frame for host D arrives on port 1 of a 4-port device; D is on port 3.
Where does it go if the device is (a) a hub, (b) a switch that already knows D's location?

<details>
<summary><b>Solution</b> (click to expand)</summary>

(a) Hub: out of ports **2, 3, and 4**, i.e. everything except the port it came in on. The hub does not
understand addresses; every host receives the frame, and all but D drop it in their NIC filter.
(b) Switch: out of **port 3 only**. The other hosts never see it, which also improves privacy and lets
different port pairs transfer data simultaneously.

</details>

**Q4.7** *(Concept)* — Of hub, switch, router and AP, which must parse the **IP** header of a packet it forwards? Which one
rewrites the Ethernet source and destination addresses?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Only the **router** parses the IP header (L3), and it is also the one that builds a **new** Ethernet header for the
next link, with its own MAC as the source and the next hop's MAC as the destination. Switches and APs forward
frames with the MAC addresses unchanged. *Misconception:* "the destination MAC of my packet is the web
server's MAC". It is the MAC of the next device on your link, usually your router. Part 6 shows how that
MAC is found.

</details>

## 4.8 The home "router" box

The box your ISP gave you is several devices in one case:

```
              ┌───────────────── home "router" box ───────────────────┐
 laptop ~~~~~~│~~ Wi-Fi AP ─┐                                          │
              │             ├─ switch ── router + NAT ── WAN port ──────│──┐
 desktop =====│== LAN ports ┘               │                          │  │ Ethernet
              │                   DHCP server, DNS forwarder           │  │
              └────────────────────────────────────────────────────────┘  │
                                  modem / ONT (sometimes built in) ◄──────┘
                                       │  DSL / coax / fibre
                                      ISP ─── Internet
```

* **AP + switch**: connect all home devices into one LAN.
* **Router**: forwards packets between the home LAN and the ISP.
* **DHCP server**: automatically hands each new device an IP address and network settings.
* **NAT** (network address translation): lets every home device share the single public IP
  address the ISP gives you.
* **DNS forwarder**: relays name lookups (e.g. `www.example.com` → IP address) to the ISP's DNS
  servers and caches the answers.

**Q4.8** *(Concept)* — Your Wi-Fi laptop prints to a printer plugged into a LAN port of the same home box. Which
components of the box carry those frames? Is the router function involved?

<details>
<summary><b>Solution</b> (click to expand)</summary>

The **AP** (radio → Ethernet) and the **switch** (to the printer's port). Both devices are on the **same LAN**,
so frames are bridged at L2 and the router, NAT, and modem are not involved at all. The same LAN spans wired
and wireless ports. *Misconception:* "everything goes through the router because the box is called a
router". Only traffic leaving the LAN is routed.

</details>

## 4.9 Detecting bit errors: the parity bit

Noise flips bits. The simplest detector is **even parity**: the sender appends one bit chosen so
that the total number of 1-bits (data + parity) is **even**. The receiver recounts, and an odd
total means an error was detected.

```
data 1011 0010  -> four 1s  -> parity bit 0   total 4 (even)  ✓
data 1011 0011  -> five 1s  -> parity bit 1   total 6 (even)  ✓
```

Counting the 1-bits of a value is called **popcount**. The receiver cannot tell *which* bit
flipped, only that an odd number of them did.

### ✍️ Exercise 4.9 — Even parity over a buffer

* `popcount8(b)`: the number of 1 bits in a byte (loop over bits, §1.3).
* `even_parity_bit(data)`: the parity bit (0 or 1) for all bits of all bytes in `data`.
* `parity_ok(data, p)`: `true` if data plus parity bit `p` contain an even number of 1s.

For `{0x41, 0x42, 0x43}` there are 7 one-bits.

In [40]:
namespace ex4_9 {
int popcount8(uint8_t b) {
    int n = 0;
    for (int i = 0; i < 8; ++i) n += (b >> i) & 1;
    return n;
}
uint8_t even_parity_bit(const std::vector<uint8_t>& data) {
    int ones = 0;
    for (uint8_t b : data) ones += popcount8(b);
    return uint8_t(ones % 2);          // 1 if odd, so that the total becomes even
}
bool parity_ok(const std::vector<uint8_t>& data, uint8_t p) {
    return even_parity_bit(data) == p;
}
}

**Test for Exercise 4.9** — run the next cell.

In [41]:
{
    CHECK_EQ(ex4_9::popcount8(0xB2), 4);
    CHECK_EQ(ex4_9::popcount8(0xFF), 8);
    std::vector<uint8_t> data = {0x41, 0x42, 0x43};
    uint8_t p = ex4_9::even_parity_bit(data);
    CHECK_EQ(int(p), 1);
    CHECK(ex4_9::parity_ok(data, p));

    auto one_flip = data;  one_flip[1] ^= 0x08;                   // flip one bit
    CHECK(!ex4_9::parity_ok(one_flip, p));                       // detected

    auto two_flips = data; two_flips[0] ^= 0x01; two_flips[2] ^= 0x40;   // flip two bits
    std::cout << "two-bit error detected? " << !ex4_9::parity_ok(two_flips, p) << "\n";
    CHECK(ex4_9::parity_ok(two_flips, p));                       // NOT detected: parity is fooled
}

✅ ex4_9::popcount8(0xB2) == 4
✅ ex4_9::popcount8(0xFF) == 8
✅ int(p) == 1
✅ ex4_9::parity_ok(data, p)
✅ !ex4_9::parity_ok(one_flip, p)
two-bit error detected? 0
✅ ex4_9::parity_ok(two_flips, p)


<details>
<summary><b>Solution</b> (click to expand)</summary>

XOR-ing all bits together gives the same answer (`popcount % 2` is the XOR of all bits). The last check passes
precisely because parity is *fooled*: two flips change the number of 1s by 0 or ±2, and the parity of that number
is unchanged. The next question asks what this means in practice.

```cpp
namespace ex4_9 {
int popcount8(uint8_t b) {
    int n = 0;
    for (int i = 0; i < 8; ++i) n += (b >> i) & 1;
    return n;
}
uint8_t even_parity_bit(const std::vector<uint8_t>& data) {
    int ones = 0;
    for (uint8_t b : data) ones += popcount8(b);
    return uint8_t(ones % 2);          // 1 if odd, so that the total becomes even
}
bool parity_ok(const std::vector<uint8_t>& data, uint8_t p) {
    return even_parity_bit(data) == p;
}
}
```

</details>

**Q4.9** *(Why?)* — Why does parity miss every 2-bit error, and why is that a serious weakness on real links?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Parity only captures the count of 1s **mod 2**. Any even number of flips (2, 4, …) leaves it unchanged, so
parity detects only odd-sized errors. Real noise tends to arrive in **bursts** (a spike of interference
corrupts several *adjacent* bits), so multi-bit errors are common and would slip through ~50% of the time.
This motivates stronger codes that spread each bit's influence over many check bits: the 16-bit **Internet
checksum** (Part 6) and the 32-bit **CRC** that Ethernet appends to every frame (Part 5).

</details>

### 🔁 Recap

- Media: copper (≤100 m for Ethernet), coax, fibre (long, clean), radio (shared, broadcast).
- bit rate = baud × log₂(levels); noise limits the number of levels; line codes such as Manchester guarantee transitions for clock recovery.
- The NIC serialises bits, owns a MAC address, filters and error-checks frames, and moves them via DMA through ring buffers (full ring = drop).
- Modem/ONT = signal conversion at the edge to the ISP. Hub L1, switch L2, AP L2, router L3.
- The home box = AP + switch + router + NAT + DHCP + DNS forwarder (± modem). Parity detects only odd numbers of bit errors.

**You should now be able to explain…**

- [ ] the difference between baud and bit rate;
- [ ] what a NIC does between the cable and the CPU, including DMA rings and drops;
- [ ] which device handles which layer, and which one rewrites MAC addresses;
- [ ] what each function inside a home "router" does;
- [ ] why parity is not enough for real links.

# Part 5 — Link layer: Ethernet

**Ethernet** (IEEE 802.3) is the dominant wired link-layer protocol; Wi-Fi reuses its addressing
and frame ideas. Its job is to deliver a **frame** from one NIC to another on the same LAN.

## 5.1 The Ethernet frame

```
+----------+-----+---------+---------+-----------+-------------------+-------+
| preamble | SFD | dst MAC | src MAC | EtherType |      payload      |  FCS  |
| 7 bytes  |  1  |    6    |    6    |     2     |   46 … 1500 bytes |   4   |
+----------+-----+---------+---------+-----------+-------------------+-------+
 consumed by the NIC     offset 0   6         12          14
 (clock sync + start)    |<------- 14-byte header ------>|
```

* **Preamble + SFD** (start frame delimiter): alternating 1010… bits that let the receiver lock
  its clock, then `10101011` meaning "frame starts now". The NIC strips them; software never sees them.
* **EtherType** names the payload's protocol: `0x0800` IPv4, `0x0806` ARP (the protocol that finds
  a MAC address from an IP address), `0x86DD` IPv6 (the newer version of IP, with 128-bit addresses).
* **MTU** (maximum transmission unit) = the largest payload = **1500 bytes**.
* The payload is at least **46 bytes**; shorter payloads are **padded** with zeros, so the minimum frame
  (dst … FCS) is 64 bytes.
* **FCS** (frame check sequence): a 4-byte error-detection code over the frame (§5.5).

**Q5.1** *(Compute)* — (a) What is the largest frame from dst MAC to FCS? (b) A frame carries a 20-byte payload: how many
padding bytes are added? (c) Counting the preamble+SFD and the mandatory 12-byte idle gap between frames, what fraction of
the wire time carries payload for full-size frames?

<details>
<summary><b>Solution</b> (click to expand)</summary>

(a) 6 + 6 + 2 + 1500 + 4 = **1518 bytes**.
(b) 46 − 20 = **26 bytes** of padding (the receiver must use a length field inside the payload, such as IP's
total length, to know where real data ends).
(c) Per frame on the wire: 8 + 1518 + 12 = 1538 byte-times, so efficiency = 1500 / 1538 = **97.53%**.
A "1 Gb/s" Ethernet therefore delivers at most ≈ 975 Mb/s of Ethernet payload.

</details>

### ✍️ Exercise 5.1 — Frame size and padding

* `padding_bytes(payload_len)`: zero-padding bytes needed (payload ≥ 46 means none).
* `frame_bytes(payload_len)`: size from dst MAC to FCS inclusive, padding included; return 0 if
  `payload_len > 1500` (it does not fit in one frame).

In [42]:
namespace ex5_1 {
int padding_bytes(int payload_len) {
    return payload_len >= 46 ? 0 : 46 - payload_len;
}
int frame_bytes(int payload_len) {
    if (payload_len > 1500) return 0;                          // exceeds the MTU
    return 14 + payload_len + padding_bytes(payload_len) + 4;  // header + data + pad + FCS
}
}

**Test for Exercise 5.1** — run the next cell.

In [43]:
{
    CHECK_EQ(ex5_1::padding_bytes(20), 26);
    CHECK_EQ(ex5_1::padding_bytes(46), 0);
    CHECK_EQ(ex5_1::padding_bytes(0), 46);
    CHECK_EQ(ex5_1::frame_bytes(20), 64);      // the minimum frame
    CHECK_EQ(ex5_1::frame_bytes(1500), 1518);  // the maximum frame
    CHECK_EQ(ex5_1::frame_bytes(1501), 0);     // too big
}

✅ ex5_1::padding_bytes(20) == 26
✅ ex5_1::padding_bytes(46) == 0
✅ ex5_1::padding_bytes(0) == 46
✅ ex5_1::frame_bytes(20) == 64
✅ ex5_1::frame_bytes(1500) == 1518
✅ ex5_1::frame_bytes(1501) == 0


<details>
<summary><b>Solution</b> (click to expand)</summary>

Why a minimum at all? In the original shared-cable Ethernet a sender had to still be transmitting when news of a
collision came back from the far end of the cable, so every frame had to last at least one round trip of the
cable, i.e. 64 bytes. Switched full-duplex Ethernet no longer needs this, but the rule remains for compatibility.

```cpp
namespace ex5_1 {
int padding_bytes(int payload_len) {
    return payload_len >= 46 ? 0 : 46 - payload_len;
}
int frame_bytes(int payload_len) {
    if (payload_len > 1500) return 0;                          // exceeds the MTU
    return 14 + payload_len + padding_bytes(payload_len) + 4;  // header + data + pad + FCS
}
}
```

</details>

## 5.2 MAC addresses

A **MAC address** is 48 bits (6 bytes), written as six two-digit hex bytes separated by colons:
`3c:22:fb:12:ab:9e`. The first three bytes are the **OUI** (organizationally unique identifier),
a block assigned to the manufacturer; the manufacturer numbers the last three bytes itself.
Software handles addresses as 6 raw bytes and converts them to text only for display.

In [44]:
namespace lib5 {

// PROVIDED (you wrote this in §1.2)
std::string to_hex8(uint8_t b) {
    std::ostringstream os;
    // unsigned(b): print the number, not the character; setw(2)+setfill('0'): keep leading 0
    os << std::hex << std::setw(2) << std::setfill('0') << unsigned(b);
    return os.str();
}

// PROVIDED (you wrote this in §1.6)
uint16_t read_be16(const uint8_t* p) {
    return uint16_t((uint16_t(p[0]) << 8) | p[1]);
}
uint32_t read_be32(const uint8_t* p) {
    return (uint32_t(p[0]) << 24) | (uint32_t(p[1]) << 16) |
           (uint32_t(p[2]) << 8)  |  uint32_t(p[3]);
}
void write_be16(uint8_t* p, uint16_t v) {
    p[0] = uint8_t(v >> 8);      // most significant byte first
    p[1] = uint8_t(v & 0xFF);
}
void write_be32(uint8_t* p, uint32_t v) {
    p[0] = uint8_t(v >> 24);
    p[1] = uint8_t(v >> 16);     // the uint8_t cast keeps only the low 8 bits
    p[2] = uint8_t(v >> 8);
    p[3] = uint8_t(v);
}

} // namespace lib5

### ✍️ Exercise 5.2 — Format a MAC address

Write `format_mac(p)` for the 6 bytes at `p`, producing lowercase `aa:bb:cc:dd:ee:ff`.
Use `lib5::to_hex8`.

In [45]:
namespace ex5_2 {
std::string format_mac(const uint8_t* p) {
    std::string s;
    for (int i = 0; i < 6; ++i) {
        if (i) s += ':';          // separator between bytes, not after the last
        s += lib5::to_hex8(p[i]);
    }
    return s;
}
}

**Test for Exercise 5.2** — run the next cell.

In [46]:
{
    const uint8_t m1[6] = {0x3c, 0x22, 0xfb, 0x12, 0xab, 0x9e};
    const uint8_t m2[6] = {0x00, 0x00, 0x0a, 0x00, 0x00, 0x01};
    CHECK_EQ(ex5_2::format_mac(m1), "3c:22:fb:12:ab:9e");
    CHECK_EQ(ex5_2::format_mac(m2), "00:00:0a:00:00:01");   // leading zeros kept
}

✅ ex5_2::format_mac(m1) == "3c:22:fb:12:ab:9e"
✅ ex5_2::format_mac(m2) == "00:00:0a:00:00:01"


<details>
<summary><b>Solution</b> (click to expand)</summary>

The only traps are the separator (none after the last byte) and leading zeros (`0a`, not `a`); `to_hex8`
already handles the latter.

```cpp
namespace ex5_2 {
std::string format_mac(const uint8_t* p) {
    std::string s;
    for (int i = 0; i < 6; ++i) {
        if (i) s += ':';          // separator between bytes, not after the last
        s += lib5::to_hex8(p[i]);
    }
    return s;
}
}
```

</details>

## 5.3 Unicast, multicast, broadcast

Two bits of the **first** address byte have special meanings:

```
first byte:   b7 b6 b5 b4 b3 b2 [b1] [b0]
                                  |    └─ I/G bit: 0 = Individual (unicast), 1 = Group (multicast)
                                  └────── U/L bit: 0 = Universal (factory OUI), 1 = Locally administered
```

* **Unicast**: I/G = 0; the frame is for one NIC.
* **Multicast**: I/G = 1; the frame is for every NIC that joined that group.
* **Broadcast**: `ff:ff:ff:ff:ff:ff`, the special group "everyone on the LAN" (its I/G bit is 1 too).
* U/L = 1 marks addresses set by software, e.g. the randomised MACs phones use for privacy.

Why bit 0? Ethernet transmits each byte least-significant bit first, so the I/G bit is the
**very first address bit on the wire**, and hardware can start deciding before the rest arrives.

### ✍️ Exercise 5.3 — Classify a destination address

Implement the three predicates on the 6 bytes at `p`.

In [47]:
namespace ex5_3 {
bool is_broadcast(const uint8_t* p) {
    for (int i = 0; i < 6; ++i)
        if (p[i] != 0xff) return false;
    return true;
}
bool is_multicast(const uint8_t* p)            { return p[0] & 0x01; }   // I/G bit
bool is_locally_administered(const uint8_t* p) { return p[0] & 0x02; }   // U/L bit
}

**Test for Exercise 5.3** — run the next cell.

In [48]:
{
    const uint8_t bcast[6] = {0xff, 0xff, 0xff, 0xff, 0xff, 0xff};
    const uint8_t mdns[6]  = {0x01, 0x00, 0x5e, 0x00, 0x00, 0xfb};   // IPv4 multicast group
    const uint8_t uni[6]   = {0x3c, 0x22, 0xfb, 0x12, 0xab, 0x9e};
    const uint8_t rnd[6]   = {0x02, 0x11, 0x22, 0x33, 0x44, 0x55};   // randomised (local) MAC
    const uint8_t odd[6]   = {0x03, 0x00, 0x00, 0x00, 0x00, 0x01};

    CHECK(ex5_3::is_broadcast(bcast));
    CHECK(!ex5_3::is_broadcast(mdns));
    CHECK(ex5_3::is_multicast(bcast));          // broadcast is a group address
    CHECK(ex5_3::is_multicast(mdns));
    CHECK(!ex5_3::is_multicast(uni));
    CHECK(ex5_3::is_multicast(odd));            // 0x03 = ...0011: I/G = 1
    CHECK(ex5_3::is_locally_administered(rnd));
    CHECK(!ex5_3::is_locally_administered(uni));
}

✅ ex5_3::is_broadcast(bcast)
✅ !ex5_3::is_broadcast(mdns)
✅ ex5_3::is_multicast(bcast)
✅ ex5_3::is_multicast(mdns)
✅ !ex5_3::is_multicast(uni)
✅ ex5_3::is_multicast(odd)
✅ ex5_3::is_locally_administered(rnd)
✅ !ex5_3::is_locally_administered(uni)


<details>
<summary><b>Solution</b> (click to expand)</summary>

`p[0] & 0x01` is non-zero exactly when bit 0 is set; converting to `bool` yields `true`/`false`.
A common mistake is testing the *last* byte or the *high* bit of the first byte, because people
expect "first on the wire" to be the most significant bit, which it is not for Ethernet.

```cpp
namespace ex5_3 {
bool is_broadcast(const uint8_t* p) {
    for (int i = 0; i < 6; ++i)
        if (p[i] != 0xff) return false;
    return true;
}
bool is_multicast(const uint8_t* p)            { return p[0] & 0x01; }   // I/G bit
bool is_locally_administered(const uint8_t* p) { return p[0] & 0x02; }   // U/L bit
}
```

</details>

**Q5.2** *(Predict the output)* — Classify each as unicast / multicast / broadcast, and universal / local:
`33:33:00:00:00:01`, `3a:f1:9c:00:12:34`, `01:80:c2:00:00:00`.

<details>
<summary><b>Solution</b> (click to expand)</summary>

* `33:33:…`: 0x33 = `0011 0011`, so I/G = 1 → **multicast** (the IPv6 multicast prefix); U/L = 1.
* `3a:f1:…`: 0x3a = `0011 1010`, so I/G = 0 → **unicast**; U/L = 1 → **locally administered**, likely a
  phone's randomised MAC.
* `01:80:c2:00:00:00`: I/G = 1 → **multicast** (a group used by switches to talk among themselves); U/L = 0.

Only the lowest two bits of the *first* byte matter: convert that byte to binary and read bits 0 and 1.

</details>

## 5.4 Parsing an Ethernet header

Parsing means reading fields at fixed **offsets** from a byte buffer: dst at 0–5, src at 6–11,
EtherType at 12–13 (big-endian). A robust parser first checks that the buffer is long enough.

Here is a real frame, captured as software sees it (no preamble or FCS): a broadcast carrying an ARP
message, padded to the 46-byte minimum payload.

In [49]:
namespace frames5 {
const std::vector<uint8_t> arp_frame = {
    0xff, 0xff, 0xff, 0xff, 0xff, 0xff, 0x3c, 0x22, 0xfb, 0x12, 0xab, 0x9e,
    0x08, 0x06, 0x00, 0x01, 0x08, 0x00, 0x06, 0x04, 0x00, 0x01, 0x3c, 0x22,
    0xfb, 0x12, 0xab, 0x9e, 0xc0, 0xa8, 0x01, 0x0a, 0x00, 0x00, 0x00, 0x00,
    0x00, 0x00, 0xc0, 0xa8, 0x01, 0x01, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00,
    0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00,
};
}

In [50]:
hexdump(frames5::arp_frame);

0000  ff ff ff ff ff ff 3c 22  fb 12 ab 9e 08 06 00 01  |......<"........|
0010  08 00 06 04 00 01 3c 22  fb 12 ab 9e c0 a8 01 0a  |......<"........|
0020  00 00 00 00 00 00 c0 a8  01 01 00 00 00 00 00 00  |................|
0030  00 00 00 00 00 00 00 00  00 00 00 00              |............|


In [51]:
namespace lib5b {

// PROVIDED (you wrote this in §5.2)
std::string format_mac(const uint8_t* p) {
    std::string s;
    for (int i = 0; i < 6; ++i) {
        if (i) s += ':';          // separator between bytes, not after the last
        s += lib5::to_hex8(p[i]);
    }
    return s;
}

// PROVIDED (you wrote this in §5.3)
bool is_broadcast(const uint8_t* p) {
    for (int i = 0; i < 6; ++i)
        if (p[i] != 0xff) return false;
    return true;
}
bool is_multicast(const uint8_t* p)            { return p[0] & 0x01; }   // I/G bit
bool is_locally_administered(const uint8_t* p) { return p[0] & 0x02; }   // U/L bit

} // namespace lib5b

### ✍️ Exercise 5.4 — parse_eth

Fill in `parse_eth(p, n)`. Return `std::nullopt` if `n < 14`. Copy the addresses with `std::memcpy`
(`h.dst.data()` gives a writable pointer into a `std::array`) and read the EtherType with `lib5::read_be16`.

In [52]:
namespace ex5_4 {
struct EthHeader {
    std::array<uint8_t, 6> dst{};
    std::array<uint8_t, 6> src{};
    uint16_t ethertype = 0;
};
std::optional<EthHeader> parse_eth(const uint8_t* p, size_t n) {
    if (n < 14) return std::nullopt;            // too short to hold the header
    EthHeader h;
    std::memcpy(h.dst.data(), p + 0, 6);
    std::memcpy(h.src.data(), p + 6, 6);
    h.ethertype = lib5::read_be16(p + 12);      // network byte order
    return h;
}
}

**Test for Exercise 5.4** — run the next cell.

In [53]:
{
    const auto& f = frames5::arp_frame;
    auto h = ex5_4::parse_eth(f.data(), f.size());
    CHECK(h.has_value());
    auto hh = h.value_or(ex5_4::EthHeader{});        // safe even if parsing failed
    CHECK_EQ(lib5b::format_mac(hh.dst.data()), "ff:ff:ff:ff:ff:ff");
    CHECK_EQ(lib5b::format_mac(hh.src.data()), "3c:22:fb:12:ab:9e");
    CHECK_EQ(hh.ethertype, 0x0806);
    CHECK(lib5b::is_broadcast(hh.dst.data()));
    CHECK(!ex5_4::parse_eth(f.data(), 13).has_value());   // truncated
    std::cout << "payload bytes after the header: " << f.size() - 14 << "\n";
}

✅ h.has_value()
✅ lib5b::format_mac(hh.dst.data()) == "ff:ff:ff:ff:ff:ff"
✅ lib5b::format_mac(hh.src.data()) == "3c:22:fb:12:ab:9e"
✅ hh.ethertype == 0x0806
✅ lib5b::is_broadcast(hh.dst.data())
✅ !ex5_4::parse_eth(f.data(), 13).has_value()
payload bytes after the header: 46


<details>
<summary><b>Solution</b> (click to expand)</summary>

The EtherType bytes are `08 06`. Read big-endian that is 0x0806 = ARP; a naive little-endian load would give
0x0608, which is not a known protocol. Returning `std::optional` forces the caller to handle "not a valid
header" explicitly instead of reading garbage from a short buffer.

```cpp
namespace ex5_4 {
struct EthHeader {
    std::array<uint8_t, 6> dst{};
    std::array<uint8_t, 6> src{};
    uint16_t ethertype = 0;
};
std::optional<EthHeader> parse_eth(const uint8_t* p, size_t n) {
    if (n < 14) return std::nullopt;            // too short to hold the header
    EthHeader h;
    std::memcpy(h.dst.data(), p + 0, 6);
    std::memcpy(h.src.data(), p + 6, 6);
    h.ethertype = lib5::read_be16(p + 12);      // network byte order
    return h;
}
}
```

</details>

**Q5.3** *(Why?)* — Your parser reads EtherType `0x8100` from a frame captured on an office network. What happened,
and where is the real EtherType?

<details>
<summary><b>Solution</b> (click to expand)</summary>

`0x8100` marks an **802.1Q VLAN tag**: 4 extra bytes inserted after the source MAC (2 bytes `0x8100` plus
2 bytes holding a 12-bit VLAN ID and a priority). The real EtherType follows at offset **16**. VLANs (virtual
LANs) let one physical switch carry several isolated LANs. *Lesson:* never hard-code "payload starts at 14"
without checking the EtherType.

</details>

## 5.5 The FCS: a CRC in the trailer (concept only)

The **FCS** is a **CRC-32** (cyclic redundancy check) computed by the sender over dst MAC …
payload and appended as the 4-byte trailer. The receiving NIC recomputes it; on mismatch the frame
is **silently dropped**. Ethernet never retransmits and never notifies anyone.

Idea: treat the frame's bits as the coefficients of a huge polynomial over GF(2) (arithmetic
modulo 2, where addition is XOR). Divide it by a fixed degree-32 *generator* polynomial and send
the 32-bit **remainder**; the receiver checks that the remainder matches.

Unlike parity, every data bit influences many check bits. CRC-32 detects **all** burst errors
of length ≤ 32 bits and all 1-, 2- and 3-bit errors in any Ethernet-sized frame; other error
patterns slip through with probability ≈ 2⁻³². NICs compute it in hardware at line rate.

**Q5.4** *(Why?)* — A switch receives a frame whose FCS does not match. What does it do, and who eventually recovers the lost data?

<details>
<summary><b>Solution</b> (click to expand)</summary>

It **drops** the frame and increments an error counter: no retransmission and no message to the sender.
Recovery, if any, is left to the endpoints' **higher layers**: a reliable transport protocol notices the missing
data and resends it; an application using unreliable delivery simply loses it. *Misconception:* "Ethernet is
reliable because it has a CRC". Error **detection** is not error **recovery**.

</details>

## 5.6 How a switch learns

A switch keeps a **MAC table** (MAC address → port). For each frame arriving on port *in*:

1. **Learn**: record `table[src] = in`. The source address shows where that host lives, and a
   later frame from a new port overwrites the entry, so hosts can move.
2. **Forward**:
   * destination is multicast or broadcast, or not in the table → **flood**: send out of every
     port except *in*;
   * destination known on port *in* itself → **filter**: drop it (the destination already heard it on that segment);
   * otherwise → **forward** out of that one port.

Real switches also **age out** entries not refreshed within a few minutes. No configuration is needed;
this is why you can plug devices into a switch and it just works.

```
     A(port 1)     B(port 2)     C(port 3)     D, E(port 4, via a hub)
         \             |             |             /
          +--------------- switch -----------------+
```

In [54]:
namespace lib5c {

// PROVIDED (you wrote this in §5.3)
bool is_broadcast(const uint8_t* p) {
    for (int i = 0; i < 6; ++i)
        if (p[i] != 0xff) return false;
    return true;
}
bool is_multicast(const uint8_t* p)            { return p[0] & 0x01; }   // I/G bit
bool is_locally_administered(const uint8_t* p) { return p[0] & 0x02; }   // U/L bit

// GIVEN helper (new here, not an exercise)
// "aa:bb:cc:dd:ee:ff" -> 6 bytes (assumes well-formed input); std::stoi(text, nullptr, 16) parses hex
std::array<uint8_t, 6> parse_mac(const std::string& s) {
    std::array<uint8_t, 6> m{};
    for (int i = 0; i < 6; ++i) m[i] = uint8_t(std::stoi(s.substr(3 * i, 2), nullptr, 16));
    return m;
}

} // namespace lib5c

### ✍️ Exercise 5.6 — A learning switch

Implement `handle(in_port, src, dst)`: learn, then return the output ports in **ascending** order
(an empty vector means "filtered"). Ports are numbered `1 … n_ports`. `std::map<K, V>` is an ordered
key → value table: `table[k] = v` inserts or overwrites; `table.find(k) == table.end()` means "absent".
For the multicast test use `lib5c::is_multicast(lib5c::parse_mac(dst).data())`.

In [55]:
namespace ex5_6 {
struct LearningSwitch {
    int n_ports;
    std::map<std::string, int> table;     // MAC -> port

    std::vector<int> handle(int in_port, const std::string& src, const std::string& dst) {
        table[src] = in_port;                                         // 1. learn (or move)
        bool group = lib5c::is_multicast(lib5c::parse_mac(dst).data());
        auto it = table.find(dst);
        std::vector<int> out;
        if (group || it == table.end()) {                             // 2a. flood
            for (int p = 1; p <= n_ports; ++p)
                if (p != in_port) out.push_back(p);
        } else if (it->second != in_port) {                           // 2c. forward
            out.push_back(it->second);
        }                                                             // 2b. else: filter
        return out;
    }
};
}

**Test for Exercise 5.6** — run the next cell.

In [56]:
{
    struct Frame { int in_port; std::string src, dst, label; };
    std::vector<Frame> script = {
        {1, "02:00:00:00:00:0a", "02:00:00:00:00:0b", "A->B, B unknown yet"},
        {2, "02:00:00:00:00:0b", "02:00:00:00:00:0a", "B->A, A was learned from frame 1"},
        {1, "02:00:00:00:00:0a", "02:00:00:00:00:0b", "A->B, B was learned from frame 2"},
        {3, "02:00:00:00:00:0c", "ff:ff:ff:ff:ff:ff", "C->broadcast"},
        {4, "02:00:00:00:00:0d", "02:00:00:00:00:0c", "D->C"},
        {2, "02:00:00:00:00:0b", "02:00:00:00:00:0d", "B->D"},
        {3, "02:00:00:00:00:0a", "02:00:00:00:00:0d", "A->D, A moved to port 3!"},
        {2, "02:00:00:00:00:0b", "02:00:00:00:00:0a", "B->A, table was updated by frame 7"},
        {3, "02:00:00:00:00:0c", "01:00:5e:00:00:fb", "C->multicast"},
        {4, "02:00:00:00:00:0e", "02:00:00:00:00:0d", "E->D, E and D share port 4 (via a hub)"},
    };
    std::vector<std::vector<int>> expected = {{2, 3, 4}, {1}, {2}, {1, 2, 4}, {3}, {4}, {4}, {3}, {1, 2, 4}, {}};

    ex5_6::LearningSwitch sw{4, {}};
    for (size_t i = 0; i < script.size(); ++i) {
        auto out = sw.handle(script[i].in_port, script[i].src, script[i].dst);
        std::cout << "frame " << i + 1 << " (" << script[i].label << ") in port "
                  << script[i].in_port << " -> "
                  << (out.empty() ? "FILTER" : out.size() > 1 ? "FLOOD" : "FORWARD");
        for (int p : out) std::cout << " " << p;
        std::cout << "\n";
        CHECK_EQ(out, expected[i]);
    }
}

frame 1 (A->B, B unknown yet) in port 1 -> FLOOD 2 3 4
✅ out == expected[i]
frame 2 (B->A, A was learned from frame 1) in port 2 -> FORWARD 1
✅ out == expected[i]
frame 3 (A->B, B was learned from frame 2) in port 1 -> FORWARD 2
✅ out == expected[i]
frame 4 (C->broadcast) in port 3 -> FLOOD 1 2 4
✅ out == expected[i]
frame 5 (D->C) in port 4 -> FORWARD 3
✅ out == expected[i]
frame 6 (B->D) in port 2 -> FORWARD 4
✅ out == expected[i]
frame 7 (A->D, A moved to port 3!) in port 3 -> FORWARD 4
✅ out == expected[i]
frame 8 (B->A, table was updated by frame 7) in port 2 -> FORWARD 3
✅ out == expected[i]
frame 9 (C->multicast) in port 3 -> FLOOD 1 2 4
✅ out == expected[i]
frame 10 (E->D, E and D share port 4 (via a hub)) in port 4 -> FILTER
✅ out == expected[i]


<details>
<summary><b>Solution</b> (click to expand)</summary>

Walk through the interesting frames: frame 1 floods because B is unknown; frame 2 teaches the switch where B is,
so frame 3 is forwarded to port 2 only; frame 7 comes from A on a **new** port and silently updates the table,
so frame 8 reaches A at its new location; frame 9 is multicast and floods; frame 10's destination D lives on the
same port it arrived on, so it is **filtered**. D already received it through the hub.

```cpp
namespace ex5_6 {
struct LearningSwitch {
    int n_ports;
    std::map<std::string, int> table;     // MAC -> port

    std::vector<int> handle(int in_port, const std::string& src, const std::string& dst) {
        table[src] = in_port;                                         // 1. learn (or move)
        bool group = lib5c::is_multicast(lib5c::parse_mac(dst).data());
        auto it = table.find(dst);
        std::vector<int> out;
        if (group || it == table.end()) {                             // 2a. flood
            for (int p = 1; p <= n_ports; ++p)
                if (p != in_port) out.push_back(p);
        } else if (it->second != in_port) {                           // 2c. forward
            out.push_back(it->second);
        }                                                             // 2b. else: filter
        return out;
    }
};
}
```

</details>

**Q5.5** *(Why?)* — Why does a switch learn only from the **source** address, never from the destination?

<details>
<summary><b>Solution</b> (click to expand)</summary>

The source address of a frame arriving on port *p* **proves** that host is reachable via *p*: it just transmitted
from there. A destination address says nothing about location; the host may not even exist. Learning from
sources and flooding unknown destinations is enough, because the destination's reply (almost every protocol
replies) teaches the switch its location. *Misconception:* "switches must be configured with MAC tables". They
build them automatically from observed traffic.

</details>

### 🔁 Recap

- Ethernet frame: preamble/SFD (hardware only), dst(6), src(6), EtherType(2), payload 46–1500 (MTU 1500), FCS(4).
- EtherType demultiplexes the payload: 0x0800 IPv4, 0x0806 ARP, 0x86DD IPv6 (0x8100 = VLAN tag).
- Bit 0 of the first MAC byte = I/G (multicast); bit 1 = U/L; ff:ff:ff:ff:ff:ff = broadcast.
- The FCS is a CRC-32: detection only; bad frames are silently dropped.
- Switches learn from source addresses; flood unknown/group destinations; filter same-port destinations.

**You should now be able to explain…**

- [ ] the byte offsets of every Ethernet header field;
- [ ] why the I/G bit is bit 0 of the first byte;
- [ ] why a 20-byte payload produces a 64-byte frame;
- [ ] exactly what a switch does with each incoming frame, including after a host moves.

# Part 6 — Network layer: IPv4

Ethernet delivers frames within one LAN. **IP** (Internet Protocol) delivers **packets** between
hosts anywhere, across many networks connected by routers. This part covers IPv4, version 4,
which still carries most Internet traffic.

## 6.1 IPv4 addresses: dotted decimal ↔ `uint32_t`

An IPv4 address is 32 bits, written as four decimal **octets** (bytes) separated by dots:

```
192   .  168   .    1   .   10
0xC0     0xA8      0x01     0x0A     ->  as one number: 0xC0A8010A = 3232235786
```

In this notebook an address is a `uint32_t` **value** whose most significant byte is the
first octet. On the wire the same address is 4 big-endian bytes, so `read_be32` converts one
into the other. The POSIX function `inet_pton(AF_INET, text, &addr)` also parses text; it stores
the address in a `struct in_addr` whose field `s_addr` is in **network byte order**.

### ✍️ Exercise 6.1 — Parse and format dotted-decimal

* `parse_ipv4(s)`: return the address, or `std::nullopt` unless `s` is exactly four octets of 1–3 digits, each
  ≤ 255, separated by single dots (no spaces, no signs). Loop over the characters, keeping the current octet
  value, the number of digits in it, and the number of dots seen.
* `format_ipv4(ip)`: the inverse (`std::to_string` converts a number to text).

In [57]:
namespace ex6_1 {
std::optional<uint32_t> parse_ipv4(const std::string& s) {
    uint32_t result = 0, octet = 0;
    int dots = 0, digits = 0;
    for (char c : s) {
        if (c >= '0' && c <= '9') {
            octet = octet * 10 + uint32_t(c - '0');
            if (++digits > 3 || octet > 255) return std::nullopt;
        } else if (c == '.') {
            if (digits == 0 || dots == 3) return std::nullopt;   // empty octet / too many dots
            result = (result << 8) | octet;                     // shift in the finished octet
            ++dots; octet = 0; digits = 0;
        } else {
            return std::nullopt;                                // any other character
        }
    }
    if (dots != 3 || digits == 0) return std::nullopt;
    return (result << 8) | octet;
}
std::string format_ipv4(uint32_t ip) {
    return std::to_string(ip >> 24) + "." + std::to_string((ip >> 16) & 0xFF) + "." +
           std::to_string((ip >> 8) & 0xFF) + "." + std::to_string(ip & 0xFF);
}
}

**Test for Exercise 6.1** — run the next cell.

In [58]:
{
    CHECK_EQ(ex6_1::parse_ipv4("192.168.1.10").value_or(0), 3232235786u);
    CHECK_EQ(ex6_1::parse_ipv4("10.0.0.1").value_or(0), 167772161u);
    CHECK_EQ(ex6_1::parse_ipv4("255.255.255.255").value_or(0), 0xFFFFFFFFu);
    CHECK(ex6_1::parse_ipv4("0.0.0.0").has_value());
    for (const char* bad : {"256.1.1.1", "1.2.3", "1.2.3.4.5", "a.b.c.d", "1..2.3", "", "1.2.3.", "1234.1.1.1"})
        CHECK(!ex6_1::parse_ipv4(bad).has_value());
    CHECK_EQ(ex6_1::format_ipv4(3232235786u), "192.168.1.10");
    CHECK_EQ(ex6_1::format_ipv4(0), "0.0.0.0");

    in_addr a{};                                   // cross-check against the C library
    inet_pton(AF_INET, "172.20.3.4", &a);
    CHECK_EQ(ex6_1::parse_ipv4("172.20.3.4").value_or(0), ntohl(a.s_addr));
}

✅ ex6_1::parse_ipv4("192.168.1.10").value_or(0) == 3232235786u
✅ ex6_1::parse_ipv4("10.0.0.1").value_or(0) == 167772161u
✅ ex6_1::parse_ipv4("255.255.255.255").value_or(0) == 0xFFFFFFFFu
✅ ex6_1::parse_ipv4("0.0.0.0").has_value()
✅ !ex6_1::parse_ipv4(bad).has_value()
✅ !ex6_1::parse_ipv4(bad).has_value()
✅ !ex6_1::parse_ipv4(bad).has_value()
✅ !ex6_1::parse_ipv4(bad).has_value()
✅ !ex6_1::parse_ipv4(bad).has_value()
✅ !ex6_1::parse_ipv4(bad).has_value()
✅ !ex6_1::parse_ipv4(bad).has_value()
✅ !ex6_1::parse_ipv4(bad).has_value()
✅ ex6_1::format_ipv4(3232235786u) == "192.168.1.10"
✅ ex6_1::format_ipv4(0) == "0.0.0.0"
✅ ex6_1::parse_ipv4("172.20.3.4").value_or(0) == ntohl(a.s_addr)


<details>
<summary><b>Solution</b> (click to expand)</summary>

The parser builds the number the same way `read_be32` does, shifting each finished octet in from the right. The
validation cases are exactly the ones real parsers get wrong; `"1.2.3.4.5"` and `"1..2.3"` have caused security bugs
in allow-lists. Comparing with `inet_pton` needs `ntohl`, because `s_addr` holds the bytes in network order.

```cpp
namespace ex6_1 {
std::optional<uint32_t> parse_ipv4(const std::string& s) {
    uint32_t result = 0, octet = 0;
    int dots = 0, digits = 0;
    for (char c : s) {
        if (c >= '0' && c <= '9') {
            octet = octet * 10 + uint32_t(c - '0');
            if (++digits > 3 || octet > 255) return std::nullopt;
        } else if (c == '.') {
            if (digits == 0 || dots == 3) return std::nullopt;   // empty octet / too many dots
            result = (result << 8) | octet;                     // shift in the finished octet
            ++dots; octet = 0; digits = 0;
        } else {
            return std::nullopt;                                // any other character
        }
    }
    if (dots != 3 || digits == 0) return std::nullopt;
    return (result << 8) | octet;
}
std::string format_ipv4(uint32_t ip) {
    return std::to_string(ip >> 24) + "." + std::to_string((ip >> 16) & 0xFF) + "." +
           std::to_string((ip >> 8) & 0xFF) + "." + std::to_string(ip & 0xFF);
}
}
```

</details>

## 6.2 Subnets, masks and CIDR

A **subnet** is a block of consecutive addresses that share their top $n$ bits, the
**prefix**. The notation `10.1.2.3/20` (**CIDR**, classless inter-domain routing) means "the
first 20 bits identify the network; the remaining 12 identify the host".
The **subnet mask** has the top $n$ bits set: `/20` → `255.255.240.0`.

```
address    10.1.2.3       00001010.00000001.0000|0010.00000011
mask /20   255.255.240.0  11111111.11111111.1111|0000.00000000
network    ip & mask      00001010.00000001.0000|0000.00000000  = 10.1.0.0
broadcast  net | ~mask    00001010.00000001.0000|1111.11111111  = 10.1.15.255
```

The all-zeros host part names the **network** and the all-ones host part is the subnet's
**broadcast** address, so a /n subnet has $2^{32-n} - 2$ usable host addresses. (Exceptions: a /31
has 2 usable addresses for point-to-point links, and a /32 is one single host.)

**Q6.1** *(Compute)* — For `172.16.45.200/22`: what are the mask, network address, broadcast address and number of usable hosts?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Mask **255.255.252.0**, network **172.16.44.0**, broadcast **172.16.47.255**, usable hosts
$2^{10} - 2$ = **1022**.

Shortcut: /22 leaves 2 host bits in the third octet, so the network's third octet is a multiple of 4:
45 → 44, and the block spans 44–47. The common mistake is to "round" 45 to 40 or 32 by eye.

</details>

In [59]:
namespace lib6 {

// PROVIDED (you wrote this in §1.6)
uint16_t read_be16(const uint8_t* p) {
    return uint16_t((uint16_t(p[0]) << 8) | p[1]);
}
uint32_t read_be32(const uint8_t* p) {
    return (uint32_t(p[0]) << 24) | (uint32_t(p[1]) << 16) |
           (uint32_t(p[2]) << 8)  |  uint32_t(p[3]);
}
void write_be16(uint8_t* p, uint16_t v) {
    p[0] = uint8_t(v >> 8);      // most significant byte first
    p[1] = uint8_t(v & 0xFF);
}
void write_be32(uint8_t* p, uint32_t v) {
    p[0] = uint8_t(v >> 24);
    p[1] = uint8_t(v >> 16);     // the uint8_t cast keeps only the low 8 bits
    p[2] = uint8_t(v >> 8);
    p[3] = uint8_t(v);
}

// PROVIDED (you wrote this in §6.1)
std::optional<uint32_t> parse_ipv4(const std::string& s) {
    uint32_t result = 0, octet = 0;
    int dots = 0, digits = 0;
    for (char c : s) {
        if (c >= '0' && c <= '9') {
            octet = octet * 10 + uint32_t(c - '0');
            if (++digits > 3 || octet > 255) return std::nullopt;
        } else if (c == '.') {
            if (digits == 0 || dots == 3) return std::nullopt;   // empty octet / too many dots
            result = (result << 8) | octet;                     // shift in the finished octet
            ++dots; octet = 0; digits = 0;
        } else {
            return std::nullopt;                                // any other character
        }
    }
    if (dots != 3 || digits == 0) return std::nullopt;
    return (result << 8) | octet;
}
std::string format_ipv4(uint32_t ip) {
    return std::to_string(ip >> 24) + "." + std::to_string((ip >> 16) & 0xFF) + "." +
           std::to_string((ip >> 8) & 0xFF) + "." + std::to_string(ip & 0xFF);
}

} // namespace lib6

### ✍️ Exercise 6.2 — Masks, network, broadcast, host count

Implement the four functions. `prefix_to_mask(0)` must be `0`; remember the `<< 32` trap from §1.1.

In [60]:
namespace ex6_2 {
uint32_t prefix_to_mask(int n) {
    if (n <= 0) return 0;                       // shifting by 32 would be undefined behaviour
    return 0xFFFFFFFFu << (32 - n);             // n ones followed by 32-n zeros
}
uint32_t network_addr(uint32_t ip, int n)   { return ip & prefix_to_mask(n); }
uint32_t broadcast_addr(uint32_t ip, int n) { return network_addr(ip, n) | ~prefix_to_mask(n); }
uint64_t host_count(int n) {
    if (n == 32) return 1;
    if (n == 31) return 2;
    return (uint64_t(1) << (32 - n)) - 2;       // 64-bit: 1 << 32 is fine for /0
}
}

**Test for Exercise 6.2** — run the next cell.

In [61]:
{
    auto ip = [](const char* s) { return lib6::parse_ipv4(s).value_or(0); };
    CHECK_EQ(lib6::format_ipv4(ex6_2::prefix_to_mask(20)), "255.255.240.0");
    CHECK_EQ(ex6_2::prefix_to_mask(0), 0u);
    CHECK_EQ(ex6_2::prefix_to_mask(32), 0xFFFFFFFFu);
    CHECK_EQ(lib6::format_ipv4(ex6_2::network_addr(ip("10.1.2.3"), 20)), "10.1.0.0");
    CHECK_EQ(lib6::format_ipv4(ex6_2::broadcast_addr(ip("10.1.2.3"), 20)), "10.1.15.255");
    CHECK_EQ(lib6::format_ipv4(ex6_2::network_addr(ip("172.16.45.200"), 22)), "172.16.44.0");
    CHECK_EQ(lib6::format_ipv4(ex6_2::broadcast_addr(ip("192.168.1.130"), 25)), "192.168.1.255");
    CHECK_EQ(ex6_2::host_count(24), 254u);
    CHECK_EQ(ex6_2::host_count(20), 4094u);
    CHECK_EQ(ex6_2::host_count(31), 2u);
    CHECK_EQ(ex6_2::host_count(0), 4294967294ull);
}

✅ lib6::format_ipv4(ex6_2::prefix_to_mask(20)) == "255.255.240.0"
✅ ex6_2::prefix_to_mask(0) == 0u
✅ ex6_2::prefix_to_mask(32) == 0xFFFFFFFFu
✅ lib6::format_ipv4(ex6_2::network_addr(ip("10.1.2.3"), 20)) == "10.1.0.0"
✅ lib6::format_ipv4(ex6_2::broadcast_addr(ip("10.1.2.3"), 20)) == "10.1.15.255"
✅ lib6::format_ipv4(ex6_2::network_addr(ip("172.16.45.200"), 22)) == "172.16.44.0"
✅ lib6::format_ipv4(ex6_2::broadcast_addr(ip("192.168.1.130"), 25)) == "192.168.1.255"
✅ ex6_2::host_count(24) == 254u
✅ ex6_2::host_count(20) == 4094u
✅ ex6_2::host_count(31) == 2u
✅ ex6_2::host_count(0) == 4294967294ull


<details>
<summary><b>Solution</b> (click to expand)</summary>

`~prefix_to_mask(n)` is the *host mask* (the low 32−n bits set), so OR-ing it onto the network address sets
every host bit. `host_count` uses 64-bit arithmetic because a /0 has 2³² addresses, which does not fit in 32 bits.

```cpp
namespace ex6_2 {
uint32_t prefix_to_mask(int n) {
    if (n <= 0) return 0;                       // shifting by 32 would be undefined behaviour
    return 0xFFFFFFFFu << (32 - n);             // n ones followed by 32-n zeros
}
uint32_t network_addr(uint32_t ip, int n)   { return ip & prefix_to_mask(n); }
uint32_t broadcast_addr(uint32_t ip, int n) { return network_addr(ip, n) | ~prefix_to_mask(n); }
uint64_t host_count(int n) {
    if (n == 32) return 1;
    if (n == 31) return 2;
    return (uint64_t(1) << (32 - n)) - 2;       // 64-bit: 1 << 32 is fine for /0
}
}
```

</details>

## 6.3 Deliver directly, or send to the default gateway

Every host is configured with an address, a prefix length, and a **default gateway**: the IP
address of a router on its own subnet. To send a packet to `dst`, a host decides:

```
if network(dst, n) == network(my_ip, n):   deliver directly: dst is on my LAN
else:                                      hand the packet to the default gateway
```

Either way the packet travels inside an Ethernet frame addressed to the MAC of the chosen
**next hop**, which is `dst` itself or the gateway. The IP header always keeps the *final* destination.

In [62]:
namespace lib6b {

// PROVIDED (you wrote this in §6.2)
uint32_t prefix_to_mask(int n) {
    if (n <= 0) return 0;                       // shifting by 32 would be undefined behaviour
    return 0xFFFFFFFFu << (32 - n);             // n ones followed by 32-n zeros
}
uint32_t network_addr(uint32_t ip, int n)   { return ip & prefix_to_mask(n); }
uint32_t broadcast_addr(uint32_t ip, int n) { return network_addr(ip, n) | ~prefix_to_mask(n); }
uint64_t host_count(int n) {
    if (n == 32) return 1;
    if (n == 31) return 2;
    return (uint64_t(1) << (32 - n)) - 2;       // 64-bit: 1 << 32 is fine for /0
}

} // namespace lib6b

### ✍️ Exercise 6.3 — Same subnet? Next hop?

* `in_same_subnet(a, b, n)`
* `next_hop(my_ip, n, dst, gateway)`: returns the IP address whose MAC the frame must be sent to.

In [63]:
namespace ex6_3 {
bool in_same_subnet(uint32_t a, uint32_t b, int n) {
    return lib6b::network_addr(a, n) == lib6b::network_addr(b, n);
}
uint32_t next_hop(uint32_t my_ip, int n, uint32_t dst, uint32_t gateway) {
    return in_same_subnet(my_ip, dst, n) ? dst : gateway;
}
}

**Test for Exercise 6.3** — run the next cell.

In [64]:
{
    auto ip = [](const char* s) { return lib6::parse_ipv4(s).value_or(0); };
    uint32_t me = ip("192.168.1.10"), gw = ip("192.168.1.1");
    CHECK(ex6_3::in_same_subnet(me, ip("192.168.1.200"), 24));
    CHECK(!ex6_3::in_same_subnet(me, ip("192.168.2.5"), 24));
    CHECK(ex6_3::in_same_subnet(me, ip("192.168.2.5"), 16));          // the mask decides!
    CHECK_EQ(lib6::format_ipv4(ex6_3::next_hop(me, 24, ip("192.168.1.77"), gw)), "192.168.1.77");
    CHECK_EQ(lib6::format_ipv4(ex6_3::next_hop(me, 24, ip("203.0.113.9"), gw)), "192.168.1.1");
}

✅ ex6_3::in_same_subnet(me, ip("192.168.1.200"), 24)
✅ !ex6_3::in_same_subnet(me, ip("192.168.2.5"), 24)
✅ ex6_3::in_same_subnet(me, ip("192.168.2.5"), 16)
✅ lib6::format_ipv4(ex6_3::next_hop(me, 24, ip("192.168.1.77"), gw)) == "192.168.1.77"
✅ lib6::format_ipv4(ex6_3::next_hop(me, 24, ip("203.0.113.9"), gw)) == "192.168.1.1"


<details>
<summary><b>Solution</b> (click to expand)</summary>

The whole decision is one comparison of network addresses. Note the third check: the *same* pair of
addresses is local under /16 but remote under /24. A mistyped prefix length therefore makes a host send
local traffic to the router, or ARP for remote hosts that never answer.

```cpp
namespace ex6_3 {
bool in_same_subnet(uint32_t a, uint32_t b, int n) {
    return lib6b::network_addr(a, n) == lib6b::network_addr(b, n);
}
uint32_t next_hop(uint32_t my_ip, int n, uint32_t dst, uint32_t gateway) {
    return in_same_subnet(my_ip, dst, n) ? dst : gateway;
}
}
```

</details>

**Q6.2** *(Concept)* — Host 192.168.1.10/24 sends a packet to 93.184.216.34. In the Ethernet frame that leaves the laptop,
whose MAC address is the destination, and whose IP address is the IP destination?

<details>
<summary><b>Solution</b> (click to expand)</summary>

The **Ethernet** destination is the **gateway's MAC** (the next hop on this link); the **IP** destination is
**93.184.216.34** (the final destination). L2 addresses change at every hop, while L3 addresses stay end-to-end.
(NAT in Part 7 is the notable exception that rewrites IP addresses.) This is the single most important
idea connecting Parts 5 and 6.

</details>

## 6.4 Routing tables and longest-prefix match

A router (and every host) has a **routing table** of entries *prefix/len → next hop*. The
entry `0.0.0.0/0` matches every address: it is the **default route**. A destination can match
several entries, and the rule is **longest-prefix match** (LPM): use the most specific one.

```
prefix/len        next hop          dst 10.1.2.200 matches?
0.0.0.0/0         isp               yes (len 0)
10.0.0.0/8        core              yes (len 8)
10.1.0.0/16       campus            yes (len 16)
10.1.2.0/24       lab               yes (len 24)
10.1.2.128/25     lab-b             yes (len 25)   <- longest: wins
```

Routers fill these tables using **routing protocols** (e.g. OSPF inside one organisation and BGP
between ISPs); their internals are beyond this notebook.

### ✍️ Exercise 6.4 — Longest-prefix match

`lpm(table, dst)` returns the next hop of the longest matching entry, or `std::nullopt` if nothing
matches. The result must not depend on the order of the table.

In [65]:
namespace ex6_4 {
struct Route { uint32_t prefix; int len; std::string next_hop; };
std::optional<std::string> lpm(const std::vector<Route>& table, uint32_t dst) {
    int best_len = -1;
    std::string best;
    for (const auto& r : table) {
        bool match = lib6b::network_addr(dst, r.len) == r.prefix;   // dst inside prefix/len?
        if (match && r.len > best_len) { best_len = r.len; best = r.next_hop; }
    }
    if (best_len < 0) return std::nullopt;
    return best;
}
}

**Test for Exercise 6.4** — run the next cell.

In [66]:
{
    auto ip = [](const char* s) { return lib6::parse_ipv4(s).value_or(0); };
    std::vector<ex6_4::Route> table = {
        {ip("0.0.0.0"), 0, "isp"},
        {ip("10.0.0.0"), 8, "core"},
        {ip("10.1.0.0"), 16, "campus"},
        {ip("10.1.2.0"), 24, "lab"},
        {ip("10.1.2.128"), 25, "lab-b"},
        {ip("192.168.1.0"), 24, "direct"},
    };
    std::vector<std::string> dsts = {"10.1.2.200", "10.1.2.5", "10.1.3.4", "10.9.9.9", "8.8.8.8", "192.168.1.77"};
    std::vector<std::string> want = {"lab-b", "lab", "campus", "core", "isp", "direct"};
    for (size_t i = 0; i < dsts.size(); ++i)
        CHECK_EQ(ex6_4::lpm(table, ip(dsts[i].c_str())).value_or("<none>"), want[i]);

    std::reverse(table.begin(), table.end());                 // order must not matter
    CHECK_EQ(ex6_4::lpm(table, ip("10.1.2.200")).value_or("<none>"), "lab-b");

    std::vector<ex6_4::Route> no_default = {{ip("10.0.0.0"), 8, "core"}};
    CHECK(!ex6_4::lpm(no_default, ip("8.8.8.8")).has_value());   // no route: packet dropped
}

✅ ex6_4::lpm(table, ip(dsts[i].c_str())).value_or("<none>") == want[i]
✅ ex6_4::lpm(table, ip(dsts[i].c_str())).value_or("<none>") == want[i]
✅ ex6_4::lpm(table, ip(dsts[i].c_str())).value_or("<none>") == want[i]
✅ ex6_4::lpm(table, ip(dsts[i].c_str())).value_or("<none>") == want[i]
✅ ex6_4::lpm(table, ip(dsts[i].c_str())).value_or("<none>") == want[i]
✅ ex6_4::lpm(table, ip(dsts[i].c_str())).value_or("<none>") == want[i]
✅ ex6_4::lpm(table, ip("10.1.2.200")).value_or("<none>") == "lab-b"
✅ !ex6_4::lpm(no_default, ip("8.8.8.8")).has_value()


<details>
<summary><b>Solution</b> (click to expand)</summary>

A linear scan is the clearest correct algorithm. Real routers hold about a million Internet prefixes and must decide
in nanoseconds, so they use tries (prefix trees) or special memory (TCAM) that compares all entries in parallel.
The semantics, longest match wins, are identical. A packet that matches no entry is dropped, and the router may send
the source an error message saying the destination is unreachable.

```cpp
namespace ex6_4 {
struct Route { uint32_t prefix; int len; std::string next_hop; };
std::optional<std::string> lpm(const std::vector<Route>& table, uint32_t dst) {
    int best_len = -1;
    std::string best;
    for (const auto& r : table) {
        bool match = lib6b::network_addr(dst, r.len) == r.prefix;   // dst inside prefix/len?
        if (match && r.len > best_len) { best_len = r.len; best = r.next_hop; }
    }
    if (best_len < 0) return std::nullopt;
    return best;
}
}
```

</details>

**Q6.3** *(Predict the output)* — With the table above, where do packets to `10.1.2.127` and `10.1.2.128` go?

<details>
<summary><b>Solution</b> (click to expand)</summary>

`10.1.2.127` → **lab**, and `10.1.2.128` → **lab-b**. The /25 covers only
.128–.255, so .127 falls back to the /24. Boundary addresses are where hand-computed LPM goes wrong: write the
last octet in binary (`0111 1111` vs `1000 0000`) and compare the 25th bit.

</details>

## 6.5 ARP: from an IP next hop to a MAC address

§6.3 gives the next hop's **IP** address, but the NIC can only send frames to a **MAC**
address. **ARP** (Address Resolution Protocol) bridges L3 → L2 on a LAN:

```
192.168.1.10                                       192.168.1.1 (gateway)
     |-- ARP request, dst MAC ff:ff:ff:ff:ff:ff -->|  "Who has 192.168.1.1? Tell 192.168.1.10"
     |   (broadcast: every host on the LAN hears)  |   every other host ignores it
     |<-- ARP reply, dst MAC = requester's MAC ----|  "192.168.1.1 is at aa:aa:aa:00:00:01"
```

The answer is stored in an **ARP cache** (IP → MAC) for a limited time (tens of seconds to
minutes) so that not every packet needs a lookup. ARP messages ride directly in Ethernet
frames with EtherType `0x0806`. The frame you parsed in §5.4 **was** an ARP request:

In [67]:
hexdump(frames5::arp_frame);

0000  ff ff ff ff ff ff 3c 22  fb 12 ab 9e 08 06 00 01  |......<"........|
0010  08 00 06 04 00 01 3c 22  fb 12 ab 9e c0 a8 01 0a  |......<"........|
0020  00 00 00 00 00 00 c0 a8  01 01 00 00 00 00 00 00  |................|
0030  00 00 00 00 00 00 00 00  00 00 00 00              |............|


```
ARP payload (starts at frame offset 14)
offset  size  field                    value in the frame above
0       2     hardware type            0x0001 = Ethernet
2       2     protocol type            0x0800 = IPv4
4       1     hardware address length  6
5       1     protocol address length  4
6       2     operation                1 = request, 2 = reply
8       6     sender MAC               3c:22:fb:12:ab:9e
14      4     sender IP                192.168.1.10
18      6     target MAC               00:00:00:00:00:00  (unknown: that is the question)
24      4     target IP                192.168.1.1
```

In [68]:
namespace lib6c {

// PROVIDED (you wrote this in §5.2)
std::string format_mac(const uint8_t* p) {
    std::string s;
    for (int i = 0; i < 6; ++i) {
        if (i) s += ':';          // separator between bytes, not after the last
        s += lib5::to_hex8(p[i]);
    }
    return s;
}

} // namespace lib6c

### ✍️ Exercise 6.5a — Decode the ARP message

Fill in `parse_arp(p, n)`, where `p` points at the ARP payload (frame offset 14). Return `std::nullopt` if
`n < 28` or if the hardware/protocol type and lengths are not Ethernet/IPv4 (1, 0x0800, 6, 4).

In [69]:
namespace ex6_5a {
struct Arp {
    uint16_t op = 0;
    std::string sender_mac, target_mac;
    uint32_t sender_ip = 0, target_ip = 0;
};
std::optional<Arp> parse_arp(const uint8_t* p, size_t n) {
    if (n < 28) return std::nullopt;
    if (lib6::read_be16(p) != 1 || lib6::read_be16(p + 2) != 0x0800 || p[4] != 6 || p[5] != 4)
        return std::nullopt;                        // not Ethernet/IPv4 ARP
    Arp a;
    a.op         = lib6::read_be16(p + 6);
    a.sender_mac = lib6c::format_mac(p + 8);
    a.sender_ip  = lib6::read_be32(p + 14);
    a.target_mac = lib6c::format_mac(p + 18);
    a.target_ip  = lib6::read_be32(p + 24);
    return a;
}
}

**Test for Exercise 6.5a** — run the next cell.

In [70]:
{
    const auto& f = frames5::arp_frame;
    auto a = ex6_5a::parse_arp(f.data() + 14, f.size() - 14).value_or(ex6_5a::Arp{});
    CHECK_EQ(a.op, 1);
    CHECK_EQ(a.sender_mac, "3c:22:fb:12:ab:9e");
    CHECK_EQ(lib6::format_ipv4(a.sender_ip), "192.168.1.10");
    CHECK_EQ(a.target_mac, "00:00:00:00:00:00");
    CHECK_EQ(lib6::format_ipv4(a.target_ip), "192.168.1.1");
    CHECK(!ex6_5a::parse_arp(f.data() + 14, 20).has_value());
    std::cout << "Who has " << lib6::format_ipv4(a.target_ip) << "? Tell "
              << lib6::format_ipv4(a.sender_ip) << "\n";
}

✅ a.op == 1
✅ a.sender_mac == "3c:22:fb:12:ab:9e"
✅ lib6::format_ipv4(a.sender_ip) == "192.168.1.10"
✅ a.target_mac == "00:00:00:00:00:00"
✅ lib6::format_ipv4(a.target_ip) == "192.168.1.1"
✅ !ex6_5a::parse_arp(f.data() + 14, 20).has_value()
Who has 192.168.1.1? Tell 192.168.1.10


<details>
<summary><b>Solution</b> (click to expand)</summary>

Every field sits at a fixed offset, so parsing is the same pattern as for Ethernet: check the length, then read at
offsets with the big-endian helpers. Checking the type and length fields up front rejects ARP for other
hardware or protocols instead of misreading it.

```cpp
namespace ex6_5a {
struct Arp {
    uint16_t op = 0;
    std::string sender_mac, target_mac;
    uint32_t sender_ip = 0, target_ip = 0;
};
std::optional<Arp> parse_arp(const uint8_t* p, size_t n) {
    if (n < 28) return std::nullopt;
    if (lib6::read_be16(p) != 1 || lib6::read_be16(p + 2) != 0x0800 || p[4] != 6 || p[5] != 4)
        return std::nullopt;                        // not Ethernet/IPv4 ARP
    Arp a;
    a.op         = lib6::read_be16(p + 6);
    a.sender_mac = lib6c::format_mac(p + 8);
    a.sender_ip  = lib6::read_be32(p + 14);
    a.target_mac = lib6c::format_mac(p + 18);
    a.target_ip  = lib6::read_be32(p + 24);
    return a;
}
}
```

</details>

### ✍️ Exercise 6.5b — Simulate an ARP cache

Implement `resolve(ip, now)`:
1. If the cache holds `ip` and `now < expires`: return the cached MAC (**hit**).
2. Otherwise (absent or expired): erase any stale entry, increment `requests_sent`, and print
   `ARP request: who has <ip>?`. If `lan` contains `ip`, print `ARP reply: <ip> is at <mac>`, cache it with
   `expires = now + ttl`, and return the MAC. If not, return `std::nullopt` (nobody answered).

`lan` stands in for the other hosts on the LAN: the map of who would answer. `m.count(k)` returns 1 if a
key is present, and `m.erase(k)` removes it.

In [71]:
namespace ex6_5b {
struct ArpCache {
    struct Entry { std::string mac; double expires; };
    std::map<uint32_t, Entry> cache;
    std::map<uint32_t, std::string> lan;   // who answers ARP on this LAN (simulation input)
    double ttl = 60.0;
    int requests_sent = 0;

    std::optional<std::string> resolve(uint32_t ip, double now) {
        auto it = cache.find(ip);
        if (it != cache.end() && now < it->second.expires) return it->second.mac;   // hit
        cache.erase(ip);                                         // expired (or absent)
        ++requests_sent;
        std::cout << "  ARP request: who has " << lib6::format_ipv4(ip) << "?\n";   // broadcast
        if (!lan.count(ip)) return std::nullopt;                 // nobody answered
        const std::string& mac = lan.at(ip);
        std::cout << "  ARP reply: " << lib6::format_ipv4(ip) << " is at " << mac << "\n";  // unicast
        cache[ip] = Entry{mac, now + ttl};
        return mac;
    }
};
}

**Test for Exercise 6.5b** — run the next cell.

In [72]:
{
    auto ip = [](const char* s) { return lib6::parse_ipv4(s).value_or(0); };
    ex6_5b::ArpCache host;
    host.lan[ip("192.168.1.1")] = "aa:aa:aa:00:00:01";
    host.lan[ip("192.168.1.50")] = "aa:aa:aa:00:00:50";
    struct Step { double t; const char* ip; const char* want_mac; int want_requests; };
    std::vector<Step> steps = {
        {0.0, "192.168.1.1", "aa:aa:aa:00:00:01", 1},
        {10.0, "192.168.1.1", "aa:aa:aa:00:00:01", 1},
        {30.0, "192.168.1.50", "aa:aa:aa:00:00:50", 2},
        {70.0, "192.168.1.1", "aa:aa:aa:00:00:01", 3},
        {71.0, "192.168.1.99", "<no reply>", 4},
        {72.0, "192.168.1.99", "<no reply>", 5},
    };
    for (const auto& s : steps) {
        std::cout << "t=" << s.t << "s resolve " << s.ip << "\n";
        auto mac = host.resolve(ip(s.ip), s.t);
        CHECK_EQ(mac.value_or("<no reply>"), s.want_mac);
        CHECK_EQ(host.requests_sent, s.want_requests);
    }
}

t=0s resolve 192.168.1.1
  ARP request: who has 192.168.1.1?
  ARP reply: 192.168.1.1 is at aa:aa:aa:00:00:01
✅ mac.value_or("<no reply>") == s.want_mac
✅ host.requests_sent == s.want_requests
t=10s resolve 192.168.1.1
✅ mac.value_or("<no reply>") == s.want_mac
✅ host.requests_sent == s.want_requests
t=30s resolve 192.168.1.50
  ARP request: who has 192.168.1.50?
  ARP reply: 192.168.1.50 is at aa:aa:aa:00:00:50
✅ mac.value_or("<no reply>") == s.want_mac
✅ host.requests_sent == s.want_requests
t=70s resolve 192.168.1.1
  ARP request: who has 192.168.1.1?
  ARP reply: 192.168.1.1 is at aa:aa:aa:00:00:01
✅ mac.value_or("<no reply>") == s.want_mac
✅ host.requests_sent == s.want_requests
t=71s resolve 192.168.1.99
  ARP request: who has 192.168.1.99?
✅ mac.value_or("<no reply>") == s.want_mac
✅ host.requests_sent == s.want_requests
t=72s resolve 192.168.1.99
  ARP request: who has 192.168.1.99?
✅ mac.value_or("<no reply>") == s.want_mac
✅ host.requests_sent == s.want_requests


<details>
<summary><b>Solution</b> (click to expand)</summary>

The trace shows the cache at work: the second lookup of the gateway (t=10) costs nothing; at t=70 the entry is
60 s old and has expired, so it is re-requested. Entries expire because the IP → MAC binding can change (a new
device takes over the address, or a NIC is replaced). The unknown host triggers a request **every** time, which is
why real stacks rate-limit ARP and briefly remember failures.

```cpp
namespace ex6_5b {
struct ArpCache {
    struct Entry { std::string mac; double expires; };
    std::map<uint32_t, Entry> cache;
    std::map<uint32_t, std::string> lan;   // who answers ARP on this LAN (simulation input)
    double ttl = 60.0;
    int requests_sent = 0;

    std::optional<std::string> resolve(uint32_t ip, double now) {
        auto it = cache.find(ip);
        if (it != cache.end() && now < it->second.expires) return it->second.mac;   // hit
        cache.erase(ip);                                         // expired (or absent)
        ++requests_sent;
        std::cout << "  ARP request: who has " << lib6::format_ipv4(ip) << "?\n";   // broadcast
        if (!lan.count(ip)) return std::nullopt;                 // nobody answered
        const std::string& mac = lan.at(ip);
        std::cout << "  ARP reply: " << lib6::format_ipv4(ip) << " is at " << mac << "\n";  // unicast
        cache[ip] = Entry{mac, now + ttl};
        return mac;
    }
};
}
```

</details>

**Q6.4** *(Concept)* — Your laptop (192.168.1.10/24) opens a web page on 93.184.216.34. Does it send an ARP request for
93.184.216.34? Explain.

<details>
<summary><b>Solution</b> (click to expand)</summary>

**No.** ARP only works within the LAN: a broadcast never crosses a router, and the web server is not on this
link. By §6.3 the next hop is the **gateway**, so the laptop ARPs for **192.168.1.1**, and only if that entry is
not already cached. The router then does its own ARP (or equivalent) on the next link. *Misconception:* "ARP
finds the MAC of the destination host". It finds the MAC of the **next hop**.

</details>

**Q6.5** *(Why?)* — Why is the ARP request broadcast but the reply unicast? What does that imply about trusting ARP replies?

<details>
<summary><b>Solution</b> (click to expand)</summary>

The requester does not yet know **which** host owns the IP, so it must ask everyone; the replier knows exactly who
asked, because the request carries the sender's MAC and IP, so it answers only that host. ARP has **no
authentication**: any host can send a forged reply ("192.168.1.1 is at *my* MAC") and hosts will usually cache it.
This is *ARP spoofing*, a classic way to intercept LAN traffic.

</details>

## 6.6 The IPv4 header

```
byte  0          1          2          3
   +----------+----------+----------+----------+
 0 |ver | IHL | DSCP/ECN |     total length    |   ver = 4; IHL = header length in 32-bit words (5..15)
   +----------+----------+----------+----------+
 4 |   identification    |flg| fragment offset |   flags (3 bits): DF = don't fragment, MF = more fragments
   +----------+----------+----------+----------+
 8 |   TTL    | protocol |   header checksum   |   protocol: 1 = ICMP, 6 = TCP, 17 = UDP
   +----------+----------+----------+----------+
12 |               source address              |
   +----------+----------+----------+----------+
16 |            destination address            |
   +----------+----------+----------+----------+
20 |    options (only if IHL > 5) ...          |
```

* **total length** = header + payload in bytes (so an IP packet is at most 65 535 bytes).
* **TTL** (time to live): decremented by each router (§6.8).
* **protocol** names the payload: this is IP's demultiplexing field.
* **DSCP/ECN** mark priority and congestion; identification, flags and fragment offset serve
  fragmentation (§6.9).

In [73]:
namespace frames6 {
// UDP-carrying packet: 20-byte header + 32 payload bytes
const std::vector<uint8_t> ip_pkt = {
    0x45, 0x00, 0x00, 0x34, 0x1c, 0x46, 0x40, 0x00, 0x40, 0x11, 0x20, 0xb9,
    0xc0, 0xa8, 0x01, 0x0a, 0xcb, 0x00, 0x71, 0x07, 0x00, 0x01, 0x02, 0x03,
    0x04, 0x05, 0x06, 0x07, 0x08, 0x09, 0x0a, 0x0b, 0x0c, 0x0d, 0x0e, 0x0f,
    0x10, 0x11, 0x12, 0x13, 0x14, 0x15, 0x16, 0x17, 0x18, 0x19, 0x1a, 0x1b,
    0x1c, 0x1d, 0x1e, 0x1f,
};
// TCP-carrying packet with 4 bytes of options (IHL = 6) + 20 payload bytes
const std::vector<uint8_t> ip_opts = {
    0x46, 0x00, 0x00, 0x2c, 0xab, 0xcd, 0x00, 0x00, 0x01, 0x06, 0xf6, 0xfb,
    0x0a, 0x00, 0x00, 0x01, 0x0a, 0x00, 0x00, 0x02, 0x01, 0x01, 0x01, 0x00,
    0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00,
    0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00,
};
}

In [74]:
hexdump(frames6::ip_pkt);

0000  45 00 00 34 1c 46 40 00  40 11 20 b9 c0 a8 01 0a  |E..4.F@.@. .....|
0010  cb 00 71 07 00 01 02 03  04 05 06 07 08 09 0a 0b  |..q.............|
0020  0c 0d 0e 0f 10 11 12 13  14 15 16 17 18 19 1a 1b  |................|
0030  1c 1d 1e 1f                                       |....|


**Q6.6** *(Compute)* — A header begins `46 00 02 00`. What are the version, header length in bytes, total length, and payload length?

<details>
<summary><b>Solution</b> (click to expand)</summary>

0x46 → version = **4**, IHL = 6 → header = 6 × 4 = **24 bytes** (it has 4 bytes of options).
Total length = 0x0200 = **512** bytes → payload = 512 − 24 = **488 bytes**.

*Misconception:* IHL counts **32-bit words**, not bytes. Reading IHL = 6 as 6 bytes is the classic parser bug.

</details>

In [75]:
namespace lib6d {

// PROVIDED (you wrote this in §1.4)
uint8_t get_field(uint8_t b, int shift, int width) {
    uint8_t mask = uint8_t((1u << width) - 1);   // width ones, e.g. width 3 -> 0b111
    return uint8_t((b >> shift) & mask);
}
uint8_t pack_nibbles(uint8_t hi, uint8_t lo) {
    return uint8_t((hi << 4) | (lo & 0x0F));
}

} // namespace lib6d

### ✍️ Exercise 6.6 — Parse an IPv4 header

Fill in `parse_ipv4_header(p, n)`. Return `std::nullopt` unless: `n ≥ 20`, version is 4, IHL ≥ 5,
the header fits (`IHL*4 ≤ n`), and `IHL*4 ≤ total_len ≤ n`. The flags are the top 3 bits of byte 6;
the fragment offset is the low 13 bits of bytes 6–7.

In [76]:
namespace ex6_6 {
struct Ipv4Header {
    uint8_t version = 0, ihl = 0, ttl = 0, protocol = 0, flags = 0;
    uint16_t total_len = 0, ident = 0, frag_offset = 0, checksum = 0;
    uint32_t src = 0, dst = 0;
    size_t header_len() const { return size_t(ihl) * 4; }
};
std::optional<Ipv4Header> parse_ipv4_header(const uint8_t* p, size_t n) {
    if (n < 20) return std::nullopt;
    Ipv4Header h;
    h.version     = lib6d::get_field(p[0], 4, 4);     // high nibble
    h.ihl         = lib6d::get_field(p[0], 0, 4);     // low nibble, in 32-bit words
    h.total_len   = lib6::read_be16(p + 2);
    h.ident       = lib6::read_be16(p + 4);
    h.flags       = lib6d::get_field(p[6], 5, 3);     // top 3 bits of byte 6
    h.frag_offset = lib6::read_be16(p + 6) & 0x1FFF;  // low 13 bits
    h.ttl         = p[8];
    h.protocol    = p[9];
    h.checksum    = lib6::read_be16(p + 10);
    h.src         = lib6::read_be32(p + 12);
    h.dst         = lib6::read_be32(p + 16);
    if (h.version != 4 || h.ihl < 5) return std::nullopt;
    if (h.header_len() > n || h.total_len < h.header_len() || h.total_len > n) return std::nullopt;
    return h;
}
}

**Test for Exercise 6.6** — run the next cell.

In [77]:
{
    const auto& pk = frames6::ip_pkt;
    auto h = ex6_6::parse_ipv4_header(pk.data(), pk.size()).value_or(ex6_6::Ipv4Header{});
    CHECK_EQ(h.version, 4);
    CHECK_EQ(h.header_len(), 20u);
    CHECK_EQ(h.total_len, 52);
    CHECK_EQ(h.ident, 0x1c46);
    CHECK_EQ(h.flags, 0x2);               // DF set
    CHECK_EQ(h.frag_offset, 0);
    CHECK_EQ(h.ttl, 64);
    CHECK_EQ(h.protocol, 17);             // UDP
    CHECK_EQ(h.checksum, 0x20b9);
    CHECK_EQ(lib6::format_ipv4(h.src), "192.168.1.10");
    CHECK_EQ(lib6::format_ipv4(h.dst), "203.0.113.7");

    const auto& po = frames6::ip_opts;
    auto h2 = ex6_6::parse_ipv4_header(po.data(), po.size()).value_or(ex6_6::Ipv4Header{});
    CHECK_EQ(h2.header_len(), 24u);        // IHL = 6
    CHECK_EQ(h2.protocol, 6);              // TCP
    CHECK_EQ(h2.flags, 0);

    CHECK(!ex6_6::parse_ipv4_header(pk.data(), 19).has_value());   // truncated
    std::vector<uint8_t> v6 = pk;  v6[0] = 0x65;                   // version 6
    CHECK(!ex6_6::parse_ipv4_header(v6.data(), v6.size()).has_value());
    std::vector<uint8_t> small = pk;  small[0] = 0x44;             // IHL = 4: impossible
    CHECK(!ex6_6::parse_ipv4_header(small.data(), small.size()).has_value());
}

✅ h.version == 4
✅ h.header_len() == 20u
✅ h.total_len == 52
✅ h.ident == 0x1c46
✅ h.flags == 0x2
✅ h.frag_offset == 0
✅ h.ttl == 64
✅ h.protocol == 17
✅ h.checksum == 0x20b9
✅ lib6::format_ipv4(h.src) == "192.168.1.10"
✅ lib6::format_ipv4(h.dst) == "203.0.113.7"
✅ h2.header_len() == 24u
✅ h2.protocol == 6
✅ h2.flags == 0
✅ !ex6_6::parse_ipv4_header(pk.data(), 19).has_value()
✅ !ex6_6::parse_ipv4_header(v6.data(), v6.size()).has_value()
✅ !ex6_6::parse_ipv4_header(small.data(), small.size()).has_value()


<details>
<summary><b>Solution</b> (click to expand)</summary>

Two details deserve attention. First, field extraction reuses §1.4: `get_field(p[0], 4, 4)` is the version. Second,
the validation order: a real stack checks that the header **fits** before trusting IHL, and that `total_len` is
consistent, because an attacker controls every byte. Trailing bytes beyond `total_len` (for example Ethernet padding)
are allowed, which is why the check is `total_len ≤ n`, not `==`.

```cpp
namespace ex6_6 {
struct Ipv4Header {
    uint8_t version = 0, ihl = 0, ttl = 0, protocol = 0, flags = 0;
    uint16_t total_len = 0, ident = 0, frag_offset = 0, checksum = 0;
    uint32_t src = 0, dst = 0;
    size_t header_len() const { return size_t(ihl) * 4; }
};
std::optional<Ipv4Header> parse_ipv4_header(const uint8_t* p, size_t n) {
    if (n < 20) return std::nullopt;
    Ipv4Header h;
    h.version     = lib6d::get_field(p[0], 4, 4);     // high nibble
    h.ihl         = lib6d::get_field(p[0], 0, 4);     // low nibble, in 32-bit words
    h.total_len   = lib6::read_be16(p + 2);
    h.ident       = lib6::read_be16(p + 4);
    h.flags       = lib6d::get_field(p[6], 5, 3);     // top 3 bits of byte 6
    h.frag_offset = lib6::read_be16(p + 6) & 0x1FFF;  // low 13 bits
    h.ttl         = p[8];
    h.protocol    = p[9];
    h.checksum    = lib6::read_be16(p + 10);
    h.src         = lib6::read_be32(p + 12);
    h.dst         = lib6::read_be32(p + 16);
    if (h.version != 4 || h.ihl < 5) return std::nullopt;
    if (h.header_len() > n || h.total_len < h.header_len() || h.total_len > n) return std::nullopt;
    return h;
}
}
```

</details>

## 6.7 The Internet checksum

The header checksum protects the IPv4 header (not the payload) against corruption in routers'
memory or on links. It is the **one's complement** of the **one's-complement sum** of all 16-bit
big-endian words:

1. Set the checksum field to 0 and add all 16-bit words in a wide (32-bit) accumulator.
2. **Fold** the carries: while the sum exceeds 16 bits, `sum = (sum & 0xFFFF) + (sum >> 16)`.
   This *end-around carry* is what makes it one's-complement arithmetic. Example:
   `0xFFFF + 0x0002 = 0x10001` → fold → `0x0002`.
3. Complement: `checksum = ~sum` (16 bits). With an odd number of bytes, pad a zero byte at the end.

**Verifying:** sum all words *including* the checksum and fold. A correct header gives
`0xFFFF`, whose complement is `0`. So a receiver simply checks `internet_checksum(header) == 0`.

### ✍️ Exercise 6.7 — Compute and verify the Internet checksum

* `internet_checksum(p, n)`: steps 1–3 over `n` bytes (handle odd `n`).
* `ipv4_header_ok(p, n)`: `true` if the IPv4 header at `p` (length from IHL) checksums to 0.
  Return `false` if the header does not fit in `n` bytes.

In [78]:
namespace ex6_7 {
uint16_t internet_checksum(const uint8_t* p, size_t n) {
    uint32_t sum = 0;
    for (size_t i = 0; i + 1 < n; i += 2)
        sum += (uint32_t(p[i]) << 8) | p[i + 1];      // big-endian 16-bit words
    if (n % 2) sum += uint32_t(p[n - 1]) << 8;       // odd length: pad with a zero byte
    while (sum >> 16) sum = (sum & 0xFFFF) + (sum >> 16);   // end-around carry
    return uint16_t(~sum);
}
bool ipv4_header_ok(const uint8_t* p, size_t n) {
    if (n < 20) return false;
    size_t hl = size_t(p[0] & 0x0F) * 4;
    if (hl < 20 || hl > n) return false;
    return internet_checksum(p, hl) == 0;             // includes the stored checksum
}
}

**Test for Exercise 6.7** — run the next cell.

In [79]:
{
    const auto& pk = frames6::ip_pkt;
    std::vector<uint8_t> h(pk.begin(), pk.begin() + 20);
    h[10] = 0; h[11] = 0;                                        // sender's view: field zeroed
    CHECK_EQ(ex6_7::internet_checksum(h.data(), h.size()), 0x20b9);
    CHECK(ex6_7::ipv4_header_ok(pk.data(), pk.size()));
    CHECK(ex6_7::ipv4_header_ok(frames6::ip_opts.data(), frames6::ip_opts.size()));

    const uint8_t odd[] = {0x01, 0x02, 0x03};
    CHECK_EQ(ex6_7::internet_checksum(odd, 3), 0xfbfd);

    std::vector<uint8_t> bad = pk;
    bad[8] = uint8_t(bad[8] - 1);                               // a router decremented TTL...
    CHECK(!ex6_7::ipv4_header_ok(bad.data(), bad.size()));       // ...but forgot the checksum
}

✅ ex6_7::internet_checksum(h.data(), h.size()) == 0x20b9
✅ ex6_7::ipv4_header_ok(pk.data(), pk.size())
✅ ex6_7::ipv4_header_ok(frames6::ip_opts.data(), frames6::ip_opts.size())
✅ ex6_7::internet_checksum(odd, 3) == 0xfbfd
✅ !ex6_7::ipv4_header_ok(bad.data(), bad.size())


<details>
<summary><b>Solution</b> (click to expand)</summary>

The 32-bit accumulator cannot overflow for any IP-sized input, and folding in a loop handles the case where the
first fold itself produces a carry. Verifying by checksumming the *whole* header, stored checksum included, and
expecting 0 is simpler and less error-prone than zeroing the field and comparing.

```cpp
namespace ex6_7 {
uint16_t internet_checksum(const uint8_t* p, size_t n) {
    uint32_t sum = 0;
    for (size_t i = 0; i + 1 < n; i += 2)
        sum += (uint32_t(p[i]) << 8) | p[i + 1];      // big-endian 16-bit words
    if (n % 2) sum += uint32_t(p[n - 1]) << 8;       // odd length: pad with a zero byte
    while (sum >> 16) sum = (sum & 0xFFFF) + (sum >> 16);   // end-around carry
    return uint16_t(~sum);
}
bool ipv4_header_ok(const uint8_t* p, size_t n) {
    if (n < 20) return false;
    size_t hl = size_t(p[0] & 0x0F) * 4;
    if (hl < 20 || hl > n) return false;
    return internet_checksum(p, hl) == 0;             // includes the stored checksum
}
}
```

</details>

**Q6.7** *(Why?)* — Every router must recompute the IPv4 header checksum for every packet it forwards. Why? And give one kind of
corruption the Internet checksum cannot detect.

<details>
<summary><b>Solution</b> (click to expand)</summary>

Each router **decrements TTL**, which changes the header, so the checksum must change too. Routers update it
incrementally, without re-summing everything. Because addition is **commutative**, swapping two 16-bit words
(e.g. corrupted memory that exchanges the source and destination addresses) leaves the sum unchanged and goes
undetected. So do errors that cancel, such as +1 in one word and −1 in another. That is why links add a CRC as well.
IPv6 dropped the header checksum entirely and relies on the link-layer and transport-layer checks.

</details>

## 6.8 TTL, ICMP, and traceroute

If a routing mistake creates a **loop**, a packet could circle forever. So every router decrements
the **TTL** field, and a packet whose TTL reaches 0 is **discarded**. The router then sends the
source an **ICMP** *Time Exceeded* message.

**ICMP** (Internet Control Message Protocol, IP protocol 1) carries network error and diagnostic
messages inside IP packets. **Ping** sends ICMP *Echo Request* and times the *Echo Reply*.

**Traceroute** exploits TTL: it sends probes with TTL = 1, 2, 3, …. The probe with TTL = *k* dies
at the *k*-th router, whose Time Exceeded reply reveals that router's address. Once TTL is large
enough, the probe reaches the destination, which replies normally.

```
TTL=1:  me --> R1 (TTL 1->0: drop)  ==> R1 replies "Time Exceeded"
TTL=2:  me --> R1 --> R2 (drop)     ==> R2 replies
TTL=5:  me --> R1 --> R2 --> R3 --> R4 --> server  ==> server replies
```

### ✍️ Exercise 6.8 — Simulate traceroute

`traceroute(routers, dest, max_ttl)`: for each TTL from 1 to `max_ttl`, determine who answers the probe and
append `"<ttl> <name>"`. Stop after the destination answers.

In [80]:
namespace ex6_8 {
std::vector<std::string> traceroute(const std::vector<std::string>& routers,
                                    const std::string& dest, int max_ttl) {
    std::vector<std::string> out;
    for (int ttl = 1; ttl <= max_ttl; ++ttl) {
        if (ttl <= int(routers.size())) {
            // the TTL hits 0 at router number ttl, which sends Time Exceeded
            out.push_back(std::to_string(ttl) + " " + routers[ttl - 1]);
        } else {
            out.push_back(std::to_string(ttl) + " " + dest);   // reached the destination
            break;
        }
    }
    return out;
}
}

**Test for Exercise 6.8** — run the next cell.

In [81]:
{
    std::vector<std::string> path = {"home-gw", "isp-edge", "isp-core", "ix-peer"};
    auto t = ex6_8::traceroute(path, "server", 30);
    for (const auto& line : t) std::cout << line << "\n";
    CHECK_EQ(t, (std::vector<std::string>{"1 home-gw", "2 isp-edge", "3 isp-core", "4 ix-peer", "5 server"}));
    CHECK_EQ(ex6_8::traceroute(path, "server", 3), (std::vector<std::string>{"1 home-gw", "2 isp-edge", "3 isp-core"}));
}

1 home-gw
2 isp-edge
3 isp-core
4 ix-peer
5 server
✅ t == (std::vector<std::string>{"1 home-gw", "2 isp-edge", "3 isp-core", "4 ix-peer", "5 server"})
✅ ex6_8::traceroute(path, "server", 3) == (std::vector<std::string>{"1 home-gw", "2 isp-edge", "3 isp-core"})


<details>
<summary><b>Solution</b> (click to expand)</summary>

The router at position *k* is exactly the one where a TTL-*k* probe expires. Real traceroute sends 3 probes per TTL
and prints their round-trip times; routers that do not send ICMP show up as `* * *`. Because every probe is an
independent packet, successive probes may take different paths.

```cpp
namespace ex6_8 {
std::vector<std::string> traceroute(const std::vector<std::string>& routers,
                                    const std::string& dest, int max_ttl) {
    std::vector<std::string> out;
    for (int ttl = 1; ttl <= max_ttl; ++ttl) {
        if (ttl <= int(routers.size())) {
            // the TTL hits 0 at router number ttl, which sends Time Exceeded
            out.push_back(std::to_string(ttl) + " " + routers[ttl - 1]);
        } else {
            out.push_back(std::to_string(ttl) + " " + dest);   // reached the destination
            break;
        }
    }
    return out;
}
}
```

</details>

**Q6.8** *(Compute)* — A packet arrives at your server with TTL = 57. The sender's OS starts packets at TTL 64. How many routers did it cross?
If a routing loop between two routers forms, what happens to a packet with TTL = 64 that enters the loop?

<details>
<summary><b>Solution</b> (click to expand)</summary>

64 − 57 = **7 routers** (each decrements once). In the loop it bounces between the two routers, losing 1 per hop,
until TTL hits 0 after at most 64 hops, and a Time Exceeded message goes back to the source. Without TTL, loops
would fill links with immortal packets. *Misconception:* "TTL is a time in seconds". Originally it was meant as
seconds, but in practice it is a **hop count**.

</details>

## 6.9 MTU and fragmentation (briefly)

If a packet is larger than the **MTU** of the next link (1500 for Ethernet), an IPv4 router may
**fragment** it: it splits the payload into pieces that each get a copy of the header, with the same
*identification*, a *fragment offset* (in units of 8 bytes), and the **MF** (more fragments) flag
on all but the last piece. Only the destination **reassembles** them, and losing any fragment loses the whole packet.

If the **DF** (don't fragment) flag is set, the router drops the packet instead and returns an
ICMP *Fragmentation Needed* message. Modern hosts set DF and use those messages to discover the
path's smallest MTU (**path MTU discovery**), then send packets that fit.

**Q6.9** *(Compute)* — A 4000-byte IPv4 packet (20-byte header + 3980 bytes of data, DF clear) must cross a link with MTU 1500.
List each fragment's data length, fragment-offset field value, and MF flag.

<details>
<summary><b>Solution</b> (click to expand)</summary>

Each fragment carries at most 1500 − 20 = 1480 data bytes (a multiple of 8, as offsets require):

| fragment | data bytes | offset field (×8 bytes) | MF |
|---|---|---|---|
| 1 | 1480 | 0 | 1 |
| 2 | 1480 | 185 | 1 |
| 3 | 1020 | 370 | 0 |

The offsets are 0, 1480, 2960 bytes, divided by 8. *Misconception:* forgetting that each fragment needs its own 20-byte header, and thus using 1500 data bytes per fragment.

</details>

> **IPv6 in one table** (for comparison only)
>
> | | IPv4 | IPv6 |
> |---|---|---|
> | address size | 32 bits | 128 bits, written in hex, e.g. `2001:db8::1` |
> | header | 20–60 bytes, with checksum | fixed 40 bytes, **no** checksum |
> | fragmentation | by routers or source | by the source only |
> | L3 → L2 resolution | ARP (broadcast) | Neighbor Discovery (multicast ICMPv6) |
> | address shortage | yes → NAT (Part 7) | no: every device can have a public address |

### 🔁 Recap

- IPv4 address = 32 bits; prefix/n splits it into network and host parts; network = ip & mask, broadcast = network | ~mask.
- Same subnet → deliver directly; otherwise → default gateway. The frame goes to the next hop's MAC; the IP destination is final.
- Routing = longest-prefix match over prefix/len entries; 0.0.0.0/0 is the default route.
- ARP maps a next-hop IP to a MAC: broadcast request, unicast reply, cached with a timeout, unauthenticated.
- IPv4 header: IHL in words, total length, TTL (hop count, drives traceroute via ICMP), protocol, one's-complement checksum.

**You should now be able to explain…**

- [ ] how to compute network, broadcast and host count for any CIDR block;
- [ ] which MAC and which IP a packet to a remote server carries on the first link;
- [ ] why longest-prefix match picks the most specific route;
- [ ] every field of the IPv4 header and its offset;
- [ ] how traceroute discovers routers, and why routers must update the checksum.

# Part 7 — Home and ISP infrastructure services

Part 4 listed the functions inside a home "router" box. Now that you know IP, here is how
each one works.

## 7.1 Private and public addresses

IPv4 has only $2^{32} \approx 4.3$ billion addresses, far fewer than devices. **RFC 1918**
therefore reserves three **private** ranges that anyone may use inside their own network and
that Internet routers **never** forward:

| range | CIDR | typical use |
|---|---|---|
| 10.0.0.0 – 10.255.255.255 | 10.0.0.0/8 | companies, clouds |
| 172.16.0.0 – 172.31.255.255 | 172.16.0.0/12 | companies, container networks |
| 192.168.0.0 – 192.168.255.255 | 192.168.0.0/16 | home networks |

Other special ranges: `127.0.0.0/8` **loopback** (packets never leave the host; `127.0.0.1` is
"this machine"), `169.254.0.0/16` **link-local** (self-assigned when no DHCP server answers), and
`100.64.0.0/10` **shared** space used by ISPs that NAT their customers. A **public** address is
globally unique and routable on the Internet.

In [82]:
namespace lib7 {

// PROVIDED (you wrote this in §1.6)
uint16_t read_be16(const uint8_t* p) {
    return uint16_t((uint16_t(p[0]) << 8) | p[1]);
}
uint32_t read_be32(const uint8_t* p) {
    return (uint32_t(p[0]) << 24) | (uint32_t(p[1]) << 16) |
           (uint32_t(p[2]) << 8)  |  uint32_t(p[3]);
}
void write_be16(uint8_t* p, uint16_t v) {
    p[0] = uint8_t(v >> 8);      // most significant byte first
    p[1] = uint8_t(v & 0xFF);
}
void write_be32(uint8_t* p, uint32_t v) {
    p[0] = uint8_t(v >> 24);
    p[1] = uint8_t(v >> 16);     // the uint8_t cast keeps only the low 8 bits
    p[2] = uint8_t(v >> 8);
    p[3] = uint8_t(v);
}

// PROVIDED (you wrote this in §6.1)
std::optional<uint32_t> parse_ipv4(const std::string& s) {
    uint32_t result = 0, octet = 0;
    int dots = 0, digits = 0;
    for (char c : s) {
        if (c >= '0' && c <= '9') {
            octet = octet * 10 + uint32_t(c - '0');
            if (++digits > 3 || octet > 255) return std::nullopt;
        } else if (c == '.') {
            if (digits == 0 || dots == 3) return std::nullopt;   // empty octet / too many dots
            result = (result << 8) | octet;                     // shift in the finished octet
            ++dots; octet = 0; digits = 0;
        } else {
            return std::nullopt;                                // any other character
        }
    }
    if (dots != 3 || digits == 0) return std::nullopt;
    return (result << 8) | octet;
}
std::string format_ipv4(uint32_t ip) {
    return std::to_string(ip >> 24) + "." + std::to_string((ip >> 16) & 0xFF) + "." +
           std::to_string((ip >> 8) & 0xFF) + "." + std::to_string(ip & 0xFF);
}

// PROVIDED (you wrote this in §6.2)
uint32_t prefix_to_mask(int n) {
    if (n <= 0) return 0;                       // shifting by 32 would be undefined behaviour
    return 0xFFFFFFFFu << (32 - n);             // n ones followed by 32-n zeros
}
uint32_t network_addr(uint32_t ip, int n)   { return ip & prefix_to_mask(n); }
uint32_t broadcast_addr(uint32_t ip, int n) { return network_addr(ip, n) | ~prefix_to_mask(n); }
uint64_t host_count(int n) {
    if (n == 32) return 1;
    if (n == 31) return 2;
    return (uint64_t(1) << (32 - n)) - 2;       // 64-bit: 1 << 32 is fine for /0
}

} // namespace lib7

### ✍️ Exercise 7.1 — Classify an address

Return `"private"`, `"loopback"`, `"link-local"`, `"shared"` or `"public"`. A helper
`in_block(ip, "a.b.c.d", n)` built from `lib7::network_addr` and `lib7::parse_ipv4` keeps it short.

In [83]:
namespace ex7_1 {
bool in_block(uint32_t ip, const char* net, int n) {
    return lib7::network_addr(ip, n) == lib7::parse_ipv4(net).value_or(0);
}
std::string classify(uint32_t ip) {
    if (in_block(ip, "10.0.0.0", 8) || in_block(ip, "172.16.0.0", 12) || in_block(ip, "192.168.0.0", 16))
        return "private";
    if (in_block(ip, "127.0.0.0", 8))    return "loopback";
    if (in_block(ip, "169.254.0.0", 16)) return "link-local";
    if (in_block(ip, "100.64.0.0", 10))  return "shared";
    return "public";
}
}

**Test for Exercise 7.1** — run the next cell.

In [84]:
{
    std::vector<std::string> ips  = {"10.20.30.40", "172.16.0.1", "172.31.255.254", "172.32.0.1", "192.168.1.10", "127.0.0.1", "169.254.12.34", "100.72.1.1", "8.8.8.8", "192.169.0.1"};
    std::vector<std::string> want = {"private", "private", "private", "public", "private", "loopback", "link-local", "shared", "public", "public"};
    for (size_t i = 0; i < ips.size(); ++i) {
        std::string got = ex7_1::classify(lib7::parse_ipv4(ips[i]).value_or(0));
        std::cout << ips[i] << " -> " << got << "\n";
        CHECK_EQ(got, want[i]);
    }
}

10.20.30.40 -> private
✅ got == want[i]
172.16.0.1 -> private
✅ got == want[i]
172.31.255.254 -> private
✅ got == want[i]
172.32.0.1 -> public
✅ got == want[i]
192.168.1.10 -> private
✅ got == want[i]
127.0.0.1 -> loopback
✅ got == want[i]
169.254.12.34 -> link-local
✅ got == want[i]
100.72.1.1 -> shared
✅ got == want[i]
8.8.8.8 -> public
✅ got == want[i]
192.169.0.1 -> public
✅ got == want[i]


<details>
<summary><b>Solution</b> (click to expand)</summary>

Reusing `network_addr` makes each range test one line. The tricky cases are the boundaries of the /12:
`172.31.255.254` is private while `172.32.0.1` is public, and `192.169.0.1` is public despite "looking like" a
home address.

```cpp
namespace ex7_1 {
bool in_block(uint32_t ip, const char* net, int n) {
    return lib7::network_addr(ip, n) == lib7::parse_ipv4(net).value_or(0);
}
std::string classify(uint32_t ip) {
    if (in_block(ip, "10.0.0.0", 8) || in_block(ip, "172.16.0.0", 12) || in_block(ip, "192.168.0.0", 16))
        return "private";
    if (in_block(ip, "127.0.0.0", 8))    return "loopback";
    if (in_block(ip, "169.254.0.0", 16)) return "link-local";
    if (in_block(ip, "100.64.0.0", 10))  return "shared";
    return "public";
}
}
```

</details>

**Q7.1** *(Concept)* — A colleague says "172.32.10.5 is a private address, like 172.16.x.x". Right or wrong?

<details>
<summary><b>Solution</b> (click to expand)</summary>

**Wrong.** `172.16.0.0/12` fixes the top 12 bits: 172 = `1010 1100`, then the high nibble of the second octet must
be `0001`, so the second octet runs 16 (`0001 0000`) … 31 (`0001 1111`). 32 is `0010 0000` → outside. Only
172.16–172.31 are private. This is a favourite interview trap.

</details>

## 7.2 NAT and PAT

Private addresses cannot appear on the Internet, so the home router performs **NAT** (network
address translation), or more precisely **PAT** (port address translation). It rewrites every
outgoing packet's source IP and port to its **one public IP** and a port it picks, and remembers
the mapping:

```
inside (private)                   NAT table                     outside (public)
192.168.1.10:5555 ──► | 192.168.1.10:5555 <-> 203.0.113.5:50000 | ──► 203.0.113.5:50000 -> server:80
192.168.1.11:5555 ──► | 192.168.1.11:5555 <-> 203.0.113.5:50001 | ──► 203.0.113.5:50001 -> server:80
replies to 203.0.113.5:50001 ──► look up port 50001 ──► rewrite destination to 192.168.1.11:5555
```

A reply whose destination port has **no** mapping is **dropped**. The router must also fix the
IP header checksum and the transport-layer checksum, because the transport checksum covers the
addresses too. Here, a *port* is the 16-bit number from Part 3 that identifies a program on a host.

### ✍️ Exercise 7.2 — Simulate a NAT

* `outbound(p)`: look up `(p.src_ip, p.src_port)`; if absent, allocate `next_port++` and record the mapping
  in both maps. Return the packet with the source rewritten to `(public_ip, public port)`.
* `inbound(p)`: if `p.dst_port` is mapped, return the packet with its destination rewritten to the private
  endpoint; otherwise return `std::nullopt` (drop).

`std::pair<A, B>` groups two values (`.first`, `.second`); pairs compare lexicographically, so they work as
`std::map` keys.

In [85]:
namespace ex7_2 {
struct Packet { uint32_t src_ip; uint16_t src_port; uint32_t dst_ip; uint16_t dst_port; };
struct Nat {
    uint32_t public_ip = 0;
    uint16_t next_port = 50000;
    std::map<std::pair<uint32_t, uint16_t>, uint16_t> out_map;   // private endpoint -> public port
    std::map<uint16_t, std::pair<uint32_t, uint16_t>> in_map;    // public port -> private endpoint

    Packet outbound(Packet p) {
        auto key = std::make_pair(p.src_ip, p.src_port);
        auto it = out_map.find(key);
        uint16_t pub;
        if (it == out_map.end()) {              // first packet of this flow: new mapping
            pub = next_port++;
            out_map[key] = pub;
            in_map[pub] = key;
        } else {
            pub = it->second;                   // existing flow: reuse
        }
        p.src_ip = public_ip;
        p.src_port = pub;
        return p;
    }
    std::optional<Packet> inbound(Packet p) {
        auto it = in_map.find(p.dst_port);
        if (it == in_map.end()) return std::nullopt;   // unsolicited: drop
        p.dst_ip = it->second.first;
        p.dst_port = it->second.second;
        return p;
    }
};
}

**Test for Exercise 7.2** — run the next cell.

In [86]:
{
    auto ip = [](const char* s) { return lib7::parse_ipv4(s).value_or(0); };
    auto show = [](const ex7_2::Packet& p) {
        return lib7::format_ipv4(p.src_ip) + ":" + std::to_string(p.src_port) + " -> " +
               lib7::format_ipv4(p.dst_ip) + ":" + std::to_string(p.dst_port);
    };
    ex7_2::Nat nat;
    nat.public_ip = ip("203.0.113.5");
    uint32_t laptop = ip("192.168.1.10"), phone = ip("192.168.1.11"), server = ip("93.184.216.34");

    auto o1 = nat.outbound({laptop, 5555, server, 80});
    auto o2 = nat.outbound({phone, 5555, server, 80});        // same private port, other host
    auto o3 = nat.outbound({laptop, 5555, ip("1.1.1.1"), 53}); // same flow source, reused mapping
    std::cout << "out: " << show(o1) << "\nout: " << show(o2) << "\nout: " << show(o3) << "\n";
    CHECK_EQ(lib7::format_ipv4(o1.src_ip), "203.0.113.5");
    CHECK_EQ(o1.src_port, 50000);
    CHECK_EQ(o2.src_port, 50001);
    CHECK_EQ(o3.src_port, 50000);
    CHECK_EQ(o1.dst_port, 80);                                 // destination untouched

    auto i1 = nat.inbound({server, 80, nat.public_ip, 50001});
    CHECK(i1.has_value());
    if (i1) std::cout << "in:  " << show(*i1) << "\n";
    CHECK_EQ(i1 ? lib7::format_ipv4(i1->dst_ip) : std::string("<dropped>"), "192.168.1.11");
    CHECK_EQ(i1 ? i1->dst_port : 0, 5555);
    CHECK(!nat.inbound({ip("198.51.100.66"), 4444, nat.public_ip, 50007}).has_value());  // unsolicited
}

out: 203.0.113.5:50000 -> 93.184.216.34:80
out: 203.0.113.5:50001 -> 93.184.216.34:80
out: 203.0.113.5:50000 -> 1.1.1.1:53
✅ lib7::format_ipv4(o1.src_ip) == "203.0.113.5"
✅ o1.src_port == 50000
✅ o2.src_port == 50001
✅ o3.src_port == 50000
✅ o1.dst_port == 80
✅ i1.has_value()
in:  93.184.216.34:80 -> 192.168.1.11:5555
✅ i1 ? lib7::format_ipv4(i1->dst_ip) : std::string("<dropped>") == "192.168.1.11"
✅ i1 ? i1->dst_port : 0 == 5555
✅ !nat.inbound({ip("198.51.100.66"), 4444, nat.public_ip, 50007}).has_value()


<details>
<summary><b>Solution</b> (click to expand)</summary>

Two maps give O(log n) lookups in both directions. Our NAT keys only on the private endpoint (an
*endpoint-independent* mapping, which is why `o3` reuses port 50000). Real NATs also track the protocol, expire idle
mappings after a timeout, and often allow a reply only from the remote endpoint that was originally contacted.

```cpp
namespace ex7_2 {
struct Packet { uint32_t src_ip; uint16_t src_port; uint32_t dst_ip; uint16_t dst_port; };
struct Nat {
    uint32_t public_ip = 0;
    uint16_t next_port = 50000;
    std::map<std::pair<uint32_t, uint16_t>, uint16_t> out_map;   // private endpoint -> public port
    std::map<uint16_t, std::pair<uint32_t, uint16_t>> in_map;    // public port -> private endpoint

    Packet outbound(Packet p) {
        auto key = std::make_pair(p.src_ip, p.src_port);
        auto it = out_map.find(key);
        uint16_t pub;
        if (it == out_map.end()) {              // first packet of this flow: new mapping
            pub = next_port++;
            out_map[key] = pub;
            in_map[pub] = key;
        } else {
            pub = it->second;                   // existing flow: reuse
        }
        p.src_ip = public_ip;
        p.src_port = pub;
        return p;
    }
    std::optional<Packet> inbound(Packet p) {
        auto it = in_map.find(p.dst_port);
        if (it == in_map.end()) return std::nullopt;   // unsolicited: drop
        p.dst_ip = it->second.first;
        p.dst_port = it->second.second;
        return p;
    }
};
}
```

</details>

**Q7.2** *(Why?)* — Why can't you run a web server on your laptop at home and have a friend connect to your public IP, unless
you configure "port forwarding"? And why do two home devices both using source port 5555 not collide?

<details>
<summary><b>Solution</b> (click to expand)</summary>

An inbound connection attempt arrives at the router's public IP with **no mapping** yet (mappings are only created
by *outbound* packets), so the NAT drops it. **Port forwarding** is a manually configured, permanent mapping
(public port 80 → 192.168.1.10:80). As a side effect, NAT behaves like a simple inbound firewall.
The two devices do not collide because the NAT gives each flow its **own public port**, so the public side sees
two different endpoints. *Misconception:* "NAT is a security feature". It blocks unsolicited inbound traffic only
incidentally, and it breaks the end-to-end model that IPv6 restores.

</details>

## 7.3 DHCP: getting an address automatically

A new device has no IP address, no mask, no gateway. **DHCP** (Dynamic Host Configuration
Protocol) provides all of them. It runs over **UDP** (a transport protocol that sends single
messages to a port with no connection setup), from client port 68 to server port 67, in four steps
(**DORA**):

```
client (no IP yet)                                              DHCP server (home box)
  |-- Discover  src 0.0.0.0 -> dst 255.255.255.255 (broadcast) -->|  "any DHCP server?"
  |<-- Offer    "you may use 192.168.1.23 /24, gateway .1, DNS .1, lease 24 h"
  |-- Request   (broadcast) "I take 192.168.1.23 from server X" ->|  other servers withdraw
  |<-- Ack      "confirmed; lease starts now" --------------------|
```

The client identifies itself by its **MAC** address. The address is **leased** for a limited time,
and the client renews at about half the lease. `255.255.255.255` is the *limited broadcast*
address: every host on the local link, never forwarded by routers.

### ✍️ Exercise 7.3 — A DHCP server's address pool

The server owns the inclusive range `[pool_start, pool_end]` and `leases` maps MAC → IP.
* `offer(mac)`: the MAC's existing lease if it has one; otherwise the **lowest** pool address not leased to
  anyone; `std::nullopt` if the pool is exhausted.
* `request(mac, ip)`: **ACK** (`true`) if `ip` is in the pool and not leased to a *different* MAC: record the
  lease, replacing any old lease of this MAC. Otherwise **NAK** (`false`).
* `release(mac)`: forget the MAC's lease.

In [87]:
namespace ex7_3 {
struct DhcpServer {
    uint32_t pool_start = 0, pool_end = 0;
    std::map<std::string, uint32_t> leases;   // MAC -> IP

    // helper: which MAC holds ip ("" if free)
    std::string owner(uint32_t ip) const {
        for (const auto& kv : leases)
            if (kv.second == ip) return kv.first;
        return "";
    }
    std::optional<uint32_t> offer(const std::string& mac) {
        auto it = leases.find(mac);
        if (it != leases.end()) return it->second;              // returning client: same address
        for (uint32_t ip = pool_start; ip <= pool_end; ++ip)
            if (owner(ip).empty()) return ip;                    // lowest free address
        return std::nullopt;                                     // pool exhausted
    }
    bool request(const std::string& mac, uint32_t ip) {
        if (ip < pool_start || ip > pool_end) return false;
        std::string o = owner(ip);
        if (!o.empty() && o != mac) return false;                // taken by someone else: NAK
        leases[mac] = ip;                                        // ACK (replaces an old lease)
        return true;
    }
    void release(const std::string& mac) { leases.erase(mac); }
};
}

**Test for Exercise 7.3** — run the next cell.

In [88]:
{
    auto ip = [](const char* s) { return lib7::parse_ipv4(s).value_or(0); };
    ex7_3::DhcpServer srv;
    srv.pool_start = ip("192.168.1.100");
    srv.pool_end   = ip("192.168.1.102");
    struct Ev { const char* op; const char* mac; const char* arg; const char* want; };
    std::vector<Ev> events = {
        {"offer", "A", "", "192.168.1.100"},
        {"request", "A", "192.168.1.100", "ACK"},
        {"offer", "B", "", "192.168.1.101"},
        {"request", "B", "192.168.1.101", "ACK"},
        {"offer", "A", "", "192.168.1.100"},
        {"offer", "C", "", "192.168.1.102"},
        {"request", "C", "192.168.1.102", "ACK"},
        {"offer", "D", "", "<none>"},
        {"request", "D", "192.168.1.101", "NAK"},
        {"release", "B", "", "-"},
        {"offer", "D", "", "192.168.1.101"},
        {"request", "D", "192.168.1.101", "ACK"},
    };
    for (const auto& e : events) {
        std::string got;
        std::string op = e.op;
        if (op == "offer") {
            auto r = srv.offer(e.mac);
            got = r ? lib7::format_ipv4(*r) : "<none>";
        } else if (op == "request") {
            got = srv.request(e.mac, ip(e.arg)) ? "ACK" : "NAK";
        } else {
            srv.release(e.mac);
            got = "-";
        }
        std::cout << op << "(" << e.mac << (e.arg[0] ? std::string(", ") + e.arg : "") << ") -> " << got << "\n";
        CHECK_EQ(got, e.want);
    }
}

offer(A) -> 192.168.1.100
✅ got == e.want
request(A, 192.168.1.100) -> ACK
✅ got == e.want
offer(B) -> 192.168.1.101
✅ got == e.want
request(B, 192.168.1.101) -> ACK
✅ got == e.want
offer(A) -> 192.168.1.100
✅ got == e.want
offer(C) -> 192.168.1.102
✅ got == e.want
request(C, 192.168.1.102) -> ACK
✅ got == e.want
offer(D) -> <none>
✅ got == e.want
request(D, 192.168.1.101) -> NAK
✅ got == e.want
release(B) -> -
✅ got == e.want
offer(D) -> 192.168.1.101
✅ got == e.want
request(D, 192.168.1.101) -> ACK
✅ got == e.want


<details>
<summary><b>Solution</b> (click to expand)</summary>

Keying leases by MAC is what makes a returning device get the **same** address (A's second offer). The NAK for D
shows why the *Request* step exists: an offer is not a reservation, and the server re-checks ownership at
request time. Once B releases its lease, D obtains `.101`. Real servers also expire leases whose owners stop
renewing.

```cpp
namespace ex7_3 {
struct DhcpServer {
    uint32_t pool_start = 0, pool_end = 0;
    std::map<std::string, uint32_t> leases;   // MAC -> IP

    // helper: which MAC holds ip ("" if free)
    std::string owner(uint32_t ip) const {
        for (const auto& kv : leases)
            if (kv.second == ip) return kv.first;
        return "";
    }
    std::optional<uint32_t> offer(const std::string& mac) {
        auto it = leases.find(mac);
        if (it != leases.end()) return it->second;              // returning client: same address
        for (uint32_t ip = pool_start; ip <= pool_end; ++ip)
            if (owner(ip).empty()) return ip;                    // lowest free address
        return std::nullopt;                                     // pool exhausted
    }
    bool request(const std::string& mac, uint32_t ip) {
        if (ip < pool_start || ip > pool_end) return false;
        std::string o = owner(ip);
        if (!o.empty() && o != mac) return false;                // taken by someone else: NAK
        leases[mac] = ip;                                        // ACK (replaces an old lease)
        return true;
    }
    void release(const std::string& mac) { leases.erase(mac); }
};
}
```

</details>

**Q7.3** *(Why?)* — Your laptop reports the address 169.254.37.12 and cannot reach anything. What went wrong? And why is the
DHCP Request sent as a broadcast even though the client already knows the server's address?

<details>
<summary><b>Solution</b> (click to expand)</summary>

169.254.0.0/16 is **link-local**: the OS self-assigns such an address when **no DHCP server answered** (the
server is down, the Wi-Fi association failed, or the cable is bad). Link-local addresses work only on the local
link and have no gateway, so there is no Internet access.

The Request is broadcast because (1) the client still has no confirmed IP address to use as a source, and
(2) **several** servers may have made offers: the broadcast tells all of them which offer was accepted, so the
others can release the addresses they tentatively offered.

</details>

## 7.4 DNS: names to addresses

Humans use names; IP uses addresses. **DNS** (Domain Name System) is a distributed database that
maps names to **records**: `A` (IPv4 address), `AAAA` (IPv6), `CNAME` (alias for another name),
`MX` (mail server), `NS` (which servers are authoritative for a zone). Names form a tree read right to left:

```
stub resolver (in your OS)
   └─ asks ─► recursive resolver (ISP / public, often via the home box's DNS forwarder)
                 ├─► root server:  "who handles .com?"        -> the .com servers
                 ├─► .com server:  "who handles example.com?" -> example.com's servers
                 └─► authoritative server for example.com: "www.example.com A?" -> 93.184.216.34
```

Every answer carries a **TTL** (a *time in seconds*, unrelated to IP's hop count), and resolvers
**cache** answers for that long, so most lookups never reach the root. Queries normally use UDP
port 53; large answers fall back to TCP. On the wire a name is a sequence of **labels**, each
prefixed by its length, ending in a zero byte: `www.example.com` → `03 www 07 example 03 com 00`.

### ✍️ Exercise 7.4a — Encode a DNS name

`encode_dns_name(name)` produces the label format. Accept an optional trailing dot. Return an **empty** vector if
any label is empty (e.g. `"a..b"`) or longer than 63 bytes (the length byte has 6 usable bits).
For `"www.example.com"`:
```
0000  03 77 77 77 07 65 78 61  6d 70 6c 65 03 63 6f 6d  |.www.example.com|
0010  00                                                |.|
```

In [89]:
namespace ex7_4a {
std::vector<uint8_t> encode_dns_name(const std::string& name) {
    std::string n = name;
    if (!n.empty() && n.back() == '.') n.pop_back();      // "example.com." == "example.com"
    std::vector<uint8_t> out;
    size_t start = 0;
    while (true) {
        size_t dot = n.find('.', start);                   // std::string::npos if none
        std::string label = n.substr(start, dot == std::string::npos ? std::string::npos : dot - start);
        if (label.empty() || label.size() > 63) return {};
        out.push_back(uint8_t(label.size()));              // length prefix
        out.insert(out.end(), label.begin(), label.end());
        if (dot == std::string::npos) break;
        start = dot + 1;
    }
    out.push_back(0);                                      // the root label ends the name
    return out;
}
}

**Test for Exercise 7.4a** — run the next cell.

In [90]:
{
    auto e = ex7_4a::encode_dns_name("www.example.com");
    hexdump(e);
    CHECK_EQ(e, (std::vector<uint8_t>{0x03, 0x77, 0x77, 0x77, 0x07, 0x65, 0x78, 0x61, 0x6d, 0x70, 0x6c, 0x65, 0x03, 0x63, 0x6f, 0x6d, 0x00}));
    CHECK_EQ(ex7_4a::encode_dns_name("www.example.com."), e);                 // trailing dot
    CHECK_EQ(ex7_4a::encode_dns_name("localhost").size(), 11u);
    CHECK(ex7_4a::encode_dns_name("a..b").empty());
    CHECK(ex7_4a::encode_dns_name(std::string(64, 'x') + ".com").empty());
}

0000  03 77 77 77 07 65 78 61  6d 70 6c 65 03 63 6f 6d  |.www.example.com|
0010  00                                                |.|
✅ e == (std::vector<uint8_t>{0x03, 0x77, 0x77, 0x77, 0x07, 0x65, 0x78, 0x61, 0x6d, 0x70, 0x6c, 0x65, 0x03, 0x63, 0x6f, 0x6d, 0x00})
✅ ex7_4a::encode_dns_name("www.example.com.") == e
✅ ex7_4a::encode_dns_name("localhost").size() == 11u
✅ ex7_4a::encode_dns_name("a..b").empty()
✅ ex7_4a::encode_dns_name(std::string(64, 'x') + ".com").empty()


<details>
<summary><b>Solution</b> (click to expand)</summary>

`std::string::find` returns `npos` when there is no further dot, which marks the last label. Length-prefixed
labels (rather than dots) let a parser skip a label without scanning for a delimiter, the same idea as the
length-prefixed framing you will build in Part 9.

```cpp
namespace ex7_4a {
std::vector<uint8_t> encode_dns_name(const std::string& name) {
    std::string n = name;
    if (!n.empty() && n.back() == '.') n.pop_back();      // "example.com." == "example.com"
    std::vector<uint8_t> out;
    size_t start = 0;
    while (true) {
        size_t dot = n.find('.', start);                   // std::string::npos if none
        std::string label = n.substr(start, dot == std::string::npos ? std::string::npos : dot - start);
        if (label.empty() || label.size() > 63) return {};
        out.push_back(uint8_t(label.size()));              // length prefix
        out.insert(out.end(), label.begin(), label.end());
        if (dot == std::string::npos) break;
        start = dot + 1;
    }
    out.push_back(0);                                      // the root label ends the name
    return out;
}
}
```

</details>

A DNS **query message** is a 12-byte header followed by the question:

```
offset 0  ID (2)         random; echoed in the answer to match queries with replies
       2  flags (2)      0x0100 = standard query with RD ("recursion desired") set
       4  QDCOUNT (2)    number of questions = 1
       6  ANCOUNT, NSCOUNT, ARCOUNT (2 each) = 0 in a query
12        QNAME          encoded name
          QTYPE (2)      1 = A, 28 = AAAA
          QCLASS (2)     1 = IN (Internet)
```

In [91]:
namespace lib7b {

// PROVIDED (you wrote this in §7.4a)
std::vector<uint8_t> encode_dns_name(const std::string& name) {
    std::string n = name;
    if (!n.empty() && n.back() == '.') n.pop_back();      // "example.com." == "example.com"
    std::vector<uint8_t> out;
    size_t start = 0;
    while (true) {
        size_t dot = n.find('.', start);                   // std::string::npos if none
        std::string label = n.substr(start, dot == std::string::npos ? std::string::npos : dot - start);
        if (label.empty() || label.size() > 63) return {};
        out.push_back(uint8_t(label.size()));              // length prefix
        out.insert(out.end(), label.begin(), label.end());
        if (dot == std::string::npos) break;
        start = dot + 1;
    }
    out.push_back(0);                                      // the root label ends the name
    return out;
}

} // namespace lib7b

### ✍️ Exercise 7.4b — Build a complete DNS query

`build_dns_query(id, name, qtype)` returns the full message. Use `lib7::write_be16` and `lib7b::encode_dns_name`.
For ID 0x1a2b, `"www.example.com"`, type A, the correct 33 bytes are:
```
0000  1a 2b 01 00 00 01 00 00  00 00 00 00 03 77 77 77  |.+...........www|
0010  07 65 78 61 6d 70 6c 65  03 63 6f 6d 00 00 01 00  |.example.com....|
0020  01                                                |.|
```

In [92]:
namespace ex7_4b {
std::vector<uint8_t> build_dns_query(uint16_t id, const std::string& name, uint16_t qtype) {
    std::vector<uint8_t> m(12, 0);                  // header, counts start at zero
    lib7::write_be16(&m[0], id);
    lib7::write_be16(&m[2], 0x0100);                // RD = 1
    lib7::write_be16(&m[4], 1);                     // one question
    auto qname = lib7b::encode_dns_name(name);
    m.insert(m.end(), qname.begin(), qname.end());
    uint8_t tail[4];
    lib7::write_be16(tail, qtype);
    lib7::write_be16(tail + 2, 1);                  // class IN
    m.insert(m.end(), tail, tail + 4);
    return m;
}
}

**Test for Exercise 7.4b** — run the next cell.

In [93]:
{
    auto m = ex7_4b::build_dns_query(0x1a2b, "www.example.com", 1);
    hexdump(m);
    CHECK_EQ(m, (std::vector<uint8_t>{
        0x1a, 0x2b, 0x01, 0x00, 0x00, 0x01, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00,
        0x03, 0x77, 0x77, 0x77, 0x07, 0x65, 0x78, 0x61, 0x6d, 0x70, 0x6c, 0x65,
        0x03, 0x63, 0x6f, 0x6d, 0x00, 0x00, 0x01, 0x00, 0x01,
    }));
    CHECK_EQ(ex7_4b::build_dns_query(7, "www.example.com", 28).size(), 33u);   // AAAA: same size
}

0000  1a 2b 01 00 00 01 00 00  00 00 00 00 03 77 77 77  |.+...........www|
0010  07 65 78 61 6d 70 6c 65  03 63 6f 6d 00 00 01 00  |.example.com....|
0020  01                                                |.|
✅ m == (std::vector<uint8_t>{ 0x1a, 0x2b, 0x01, 0x00, 0x00, 0x01, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x03, 0x77, 0x77, 0x77, 0x07, 0x65, 0x78, 0x61, 0x6d, 0x70, 0x6c, 0x65, 0x03, 0x63, 0x6f, 0x6d, 0x00, 0x00, 0x01, 0x00, 0x01, })
✅ ex7_4b::build_dns_query(7, "www.example.com", 28).size() == 33u


<details>
<summary><b>Solution</b> (click to expand)</summary>

These are the exact bytes a resolver library puts in the payload of a UDP datagram to port 53; tools such as `dig`
send nothing more. The answer echoes the ID and the question, then appends resource records whose names are often
*compressed* (a 2-byte pointer back to an earlier name).

```cpp
namespace ex7_4b {
std::vector<uint8_t> build_dns_query(uint16_t id, const std::string& name, uint16_t qtype) {
    std::vector<uint8_t> m(12, 0);                  // header, counts start at zero
    lib7::write_be16(&m[0], id);
    lib7::write_be16(&m[2], 0x0100);                // RD = 1
    lib7::write_be16(&m[4], 1);                     // one question
    auto qname = lib7b::encode_dns_name(name);
    m.insert(m.end(), qname.begin(), qname.end());
    uint8_t tail[4];
    lib7::write_be16(tail, qtype);
    lib7::write_be16(tail + 2, 1);                  // class IN
    m.insert(m.end(), tail, tail + 4);
    return m;
}
}
```

</details>

**Q7.4** *(Why?)* — Resolvers cache answers for their TTL. What is gained, and what goes wrong when an operator moves a
website to a new IP address but the old record had TTL = 86400?

<details>
<summary><b>Solution</b> (click to expand)</summary>

**Gain:** most lookups are answered from a nearby cache in about a millisecond instead of a multi-server
walk from the root, and the load on root and TLD servers stays manageable.
**Cost, staleness:** caches may keep returning the **old** IP for up to 86 400 s (24 h) after the change. Operators
therefore lower the TTL (e.g. to 60 s) a day **before** a migration. *Misconception:* "DNS changes propagate".
Nothing is pushed; old cached answers simply expire.

</details>

## 7.5 Putting it together

You now know every service a home network needs. The next question is a classic interview
prompt; answer it on paper before expanding the solution.

**Q7.5** *(Concept)* — A laptop joins a home Wi-Fi network and opens `http://example.com`. List, **in order**, every protocol
exchange and every device involved until the first byte of the page arrives. (Plain HTTP on port 80, so no encryption
step.)

<details>
<summary><b>Solution</b> (click to expand)</summary>

1. **Wi-Fi association** (L1/L2): the laptop's NIC associates with the **AP** in the home box.
2. **DHCP DORA** (UDP 68→67, broadcast frames to `ff:ff:ff:ff:ff:ff`): the AP bridges them to the box's **DHCP
   server**, and the laptop learns, e.g., `192.168.1.23/24`, gateway `192.168.1.1`, DNS server `192.168.1.1`.
3. **ARP** for `192.168.1.1` (broadcast request, unicast reply): the laptop needs the gateway's MAC because
   both the DNS server and every remote host are reached through it.
4. **DNS** query `example.com A?` (UDP port 53) to the box's **DNS forwarder**. On a cache miss the forwarder
   sends it on: **router** → **NAT** rewrites the source to the public IP → **modem** modulates it onto
   DSL/coax/fibre → **ISP routers** (longest-prefix match at each hop, TTL−1, new L2 header per link) → the
   **recursive resolver**, which asks root → `.com` → authoritative servers if its cache is cold. The answer
   returns along the reverse path; NAT translates it back, and the forwarder caches it.
5. The laptop compares the server IP with its own subnet: it is not local, so the next hop is the gateway (ARP
   cache hit this time).
6. **TCP connection setup** to `server:80`: a short exchange of three segments (Part 8), each packet crossing
   laptop NIC → AP → router (+NAT, new public port) → modem → ISP routers → **web server**, and back.
7. **HTTP** `GET /` request over that connection; the server's response travels back through the ISP → modem →
   router/NAT (destination rewritten to the laptop) → switch/AP → laptop NIC (DMA) → OS stack → browser.

Common omissions: the ARP step, the fact that DNS runs *before* any TCP connection exists, and NAT rewriting
*both* directions.

</details>

### 🔁 Recap

- RFC 1918 private ranges: 10/8, 172.16/12, 192.168/16; also 127/8 loopback, 169.254/16 link-local, 100.64/10 shared.
- NAT/PAT maps (private IP, port) ↔ (public IP, public port); mappings are created by outbound traffic; unmapped inbound traffic is dropped.
- DHCP DORA hands out IP, mask, gateway, DNS and a lease; broadcast because the client has no address yet.
- DNS resolves names through a hierarchy (root → TLD → authoritative) via a recursive resolver; answers are cached for their TTL.
- Opening a web page = DHCP → ARP → DNS → TCP → HTTP, crossing AP, router/NAT, modem, ISP routers.

**You should now be able to explain…**

- [ ] whether any given address is private, and why 172.32.0.1 is not;
- [ ] how NAT lets many devices share one public IP, and why inbound connections fail;
- [ ] each DHCP message and why the Request is broadcast;
- [ ] the wire format of a DNS query, and what a DNS TTL means;
- [ ] the full sequence of events when a laptop joins Wi-Fi and loads a page.

# Part 8 — Transport layer: UDP and TCP

IP delivers packets from host to host, best-effort: packets may be lost, duplicated, reordered,
or corrupted. The transport layer delivers data between **programs**, and TCP adds reliability on top.

## 8.1 Ports, multiplexing, and the 5-tuple

A **port** (16 bits) identifies an endpoint inside a host. Servers **listen** on well-known
ports (0–1023, e.g. 53 DNS, 80 HTTP); clients use an **ephemeral** port that the OS picks from
a high range (Linux: 32768–60999). A **socket** is the OS object a program uses as an endpoint.

* **Multiplexing**: many programs share one IP address; the port field separates their data.
* **Demultiplexing**, UDP: the destination (IP, port) alone selects the socket.
* **Demultiplexing**, TCP: a *connection* is identified by the **5-tuple**
  (protocol, source IP, source port, destination IP, destination port). A segment that matches
  no connection but targets a **listening** socket's port starts a new connection.

```
server 10.0.0.5:80  ── listening socket (accepts new connections)
                    ├── conn: 198.51.100.7:40000  <->  10.0.0.5:80
                    ├── conn: 198.51.100.7:40001  <->  10.0.0.5:80
                    └── conn: 203.0.113.9:40000   <->  10.0.0.5:80
```
A listening socket bound to `0.0.0.0` (`INADDR_ANY`) accepts on every local address.

In [94]:
namespace lib8 {

// PROVIDED (you wrote this in §1.6)
uint16_t read_be16(const uint8_t* p) {
    return uint16_t((uint16_t(p[0]) << 8) | p[1]);
}
uint32_t read_be32(const uint8_t* p) {
    return (uint32_t(p[0]) << 24) | (uint32_t(p[1]) << 16) |
           (uint32_t(p[2]) << 8)  |  uint32_t(p[3]);
}
void write_be16(uint8_t* p, uint16_t v) {
    p[0] = uint8_t(v >> 8);      // most significant byte first
    p[1] = uint8_t(v & 0xFF);
}
void write_be32(uint8_t* p, uint32_t v) {
    p[0] = uint8_t(v >> 24);
    p[1] = uint8_t(v >> 16);     // the uint8_t cast keeps only the low 8 bits
    p[2] = uint8_t(v >> 8);
    p[3] = uint8_t(v);
}

// PROVIDED (you wrote this in §6.1)
std::optional<uint32_t> parse_ipv4(const std::string& s) {
    uint32_t result = 0, octet = 0;
    int dots = 0, digits = 0;
    for (char c : s) {
        if (c >= '0' && c <= '9') {
            octet = octet * 10 + uint32_t(c - '0');
            if (++digits > 3 || octet > 255) return std::nullopt;
        } else if (c == '.') {
            if (digits == 0 || dots == 3) return std::nullopt;   // empty octet / too many dots
            result = (result << 8) | octet;                     // shift in the finished octet
            ++dots; octet = 0; digits = 0;
        } else {
            return std::nullopt;                                // any other character
        }
    }
    if (dots != 3 || digits == 0) return std::nullopt;
    return (result << 8) | octet;
}
std::string format_ipv4(uint32_t ip) {
    return std::to_string(ip >> 24) + "." + std::to_string((ip >> 16) & 0xFF) + "." +
           std::to_string((ip >> 8) & 0xFF) + "." + std::to_string(ip & 0xFF);
}

} // namespace lib8

### ✍️ Exercise 8.1 — Demultiplex TCP segments to sockets

`demux_tcp(socks, seg)` returns the id of the socket that receives the segment:
1. a **connected** socket whose (local IP, local port, remote IP, remote port) equals the segment's
   (dst IP, dst port, src IP, src port);
2. otherwise a **listening** socket on the destination port whose local IP is the destination IP or `0`;
3. otherwise `-1` (the kernel would answer with a reset).

In [95]:
namespace ex8_1 {
struct Sock { int id; bool listening; uint32_t local_ip; uint16_t local_port; uint32_t remote_ip; uint16_t remote_port; };
struct Seg  { uint32_t src_ip; uint16_t src_port; uint32_t dst_ip; uint16_t dst_port; };
int demux_tcp(const std::vector<Sock>& socks, const Seg& s) {
    for (const auto& k : socks)                              // 1. exact 5-tuple match
        if (!k.listening && k.local_ip == s.dst_ip && k.local_port == s.dst_port &&
            k.remote_ip == s.src_ip && k.remote_port == s.src_port)
            return k.id;
    for (const auto& k : socks)                              // 2. a listener on that port
        if (k.listening && k.local_port == s.dst_port && (k.local_ip == s.dst_ip || k.local_ip == 0))
            return k.id;
    return -1;                                               // 3. nobody: reset
}
}

**Test for Exercise 8.1** — run the next cell.

In [96]:
{
    auto ip = [](const char* s) { return lib8::parse_ipv4(s).value_or(0); };
    std::vector<ex8_1::Sock> socks = {
        {1, true, ip("0.0.0.0"), 80, ip("0.0.0.0"), 0},
        {2, false, ip("10.0.0.5"), 80, ip("198.51.100.7"), 40000},
        {3, false, ip("10.0.0.5"), 80, ip("198.51.100.7"), 40001},
        {4, false, ip("10.0.0.5"), 80, ip("203.0.113.9"), 40000},
        {5, true, ip("10.0.0.5"), 22, ip("0.0.0.0"), 0},
    };
    std::vector<ex8_1::Seg> segs = {
        {ip("198.51.100.7"), 40001, ip("10.0.0.5"), 80},   // existing connection
        {ip("203.0.113.9"), 40000, ip("10.0.0.5"), 80},   // same client port as socket 2, other client IP
        {ip("192.0.2.1"), 5000, ip("10.0.0.5"), 80},   // new client: goes to the listening socket
        {ip("192.0.2.1"), 5001, ip("10.0.0.5"), 22},   // new SSH client
        {ip("192.0.2.1"), 5002, ip("10.0.0.5"), 8080},   // nobody listening
    };
    std::vector<int> want = {3, 4, 1, 5, -1};
    for (size_t i = 0; i < segs.size(); ++i)
        CHECK_EQ(ex8_1::demux_tcp(socks, segs[i]), want[i]);
}

✅ ex8_1::demux_tcp(socks, segs[i]) == want[i]
✅ ex8_1::demux_tcp(socks, segs[i]) == want[i]
✅ ex8_1::demux_tcp(socks, segs[i]) == want[i]
✅ ex8_1::demux_tcp(socks, segs[i]) == want[i]
✅ ex8_1::demux_tcp(socks, segs[i]) == want[i]


<details>
<summary><b>Solution</b> (click to expand)</summary>

The exact-match pass must come first: segments of established connections also target port 80, and without that
precedence every segment would land on the listener. Sockets 2 and 4 share client port 40000 but differ in client
IP, so the full 5-tuple is needed. This is how one server port handles thousands of simultaneous clients.

```cpp
namespace ex8_1 {
struct Sock { int id; bool listening; uint32_t local_ip; uint16_t local_port; uint32_t remote_ip; uint16_t remote_port; };
struct Seg  { uint32_t src_ip; uint16_t src_port; uint32_t dst_ip; uint16_t dst_port; };
int demux_tcp(const std::vector<Sock>& socks, const Seg& s) {
    for (const auto& k : socks)                              // 1. exact 5-tuple match
        if (!k.listening && k.local_ip == s.dst_ip && k.local_port == s.dst_port &&
            k.remote_ip == s.src_ip && k.remote_port == s.src_port)
            return k.id;
    for (const auto& k : socks)                              // 2. a listener on that port
        if (k.listening && k.local_port == s.dst_port && (k.local_ip == s.dst_ip || k.local_ip == 0))
            return k.id;
    return -1;                                               // 3. nobody: reset
}
}
```

</details>

**Q8.1** *(Compute)* — A client host with one IP address opens connections to a single server `203.0.113.10:443`.
Using Linux's default ephemeral range 32768–60999, how many simultaneous connections can it have to that one server endpoint? Why?

<details>
<summary><b>Solution</b> (click to expand)</summary>

**28232**. Four of the five tuple elements (protocol, client IP, server IP, server port) are fixed, so
only the client port can differ, and there are 60999 − 32768 + 1 = 28232 of them. The server, by
contrast, can hold connections from millions of clients on its one port 443, because their IPs and ports differ.
Proxies that open many connections to one backend hit this limit.

</details>

## 8.2 UDP: the minimal transport

**UDP** (User Datagram Protocol) adds almost nothing to IP: ports and an optional checksum.
Each `send` becomes one **datagram**, delivered whole or not at all, possibly out of order, and
with no retransmission, no connection, and no rate control.

```
byte  0          2          4          6          8
   +----------+----------+----------+----------+------------ ...
   | src port | dst port |  length  | checksum |  data
   +----------+----------+----------+----------+------------ ...
   length = header (8) + data, in bytes
```

Used where speed or simplicity beats reliability, or where the application handles loss
itself: DNS, DHCP, voice and video calls, games, and time synchronisation.
Here is the DNS query from Part 7 inside a UDP datagram (IP header omitted):

In [97]:
namespace frames8 {
// UDP datagram 192.168.1.23:53000 -> 192.168.1.1:53 carrying a DNS query
const std::vector<uint8_t> udp_dns = {
    0xcf, 0x08, 0x00, 0x35, 0x00, 0x29, 0x48, 0x6e, 0x1a, 0x2b, 0x01, 0x00,
    0x00, 0x01, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x03, 0x77, 0x77, 0x77,
    0x07, 0x65, 0x78, 0x61, 0x6d, 0x70, 0x6c, 0x65, 0x03, 0x63, 0x6f, 0x6d,
    0x00, 0x00, 0x01, 0x00, 0x01,
};
}

In [98]:
hexdump(frames8::udp_dns);

0000  cf 08 00 35 00 29 48 6e  1a 2b 01 00 00 01 00 00  |...5.)Hn.+......|
0010  00 00 00 00 03 77 77 77  07 65 78 61 6d 70 6c 65  |.....www.example|
0020  03 63 6f 6d 00 00 01 00  01                       |.com.....|


### ✍️ Exercise 8.2 — Parse a UDP header

`parse_udp(p, n)`: return `std::nullopt` if `n < 8`, or if the length field is `< 8` or `> n`.

In [99]:
namespace ex8_2 {
struct UdpHeader { uint16_t src_port = 0, dst_port = 0, length = 0, checksum = 0; };
std::optional<UdpHeader> parse_udp(const uint8_t* p, size_t n) {
    if (n < 8) return std::nullopt;
    UdpHeader h;
    h.src_port = lib8::read_be16(p);
    h.dst_port = lib8::read_be16(p + 2);
    h.length   = lib8::read_be16(p + 4);
    h.checksum = lib8::read_be16(p + 6);
    if (h.length < 8 || h.length > n) return std::nullopt;   // inconsistent length
    return h;
}
}

**Test for Exercise 8.2** — run the next cell.

In [100]:
{
    const auto& d = frames8::udp_dns;
    auto h = ex8_2::parse_udp(d.data(), d.size()).value_or(ex8_2::UdpHeader{});
    CHECK_EQ(h.src_port, 53000);
    CHECK_EQ(h.dst_port, 53);
    CHECK_EQ(h.length, 41);
    CHECK_EQ(h.checksum, 0x486e);
    CHECK(!ex8_2::parse_udp(d.data(), 7).has_value());
    CHECK(!ex8_2::parse_udp(d.data(), 20).has_value());          // length field says 41 > 20
    std::cout << "payload = " << h.length - 8 << " bytes of DNS\n";
}

✅ h.src_port == 53000
✅ h.dst_port == 53
✅ h.length == 41
✅ h.checksum == 0x486e
✅ !ex8_2::parse_udp(d.data(), 7).has_value()
✅ !ex8_2::parse_udp(d.data(), 20).has_value()
payload = 33 bytes of DNS


<details>
<summary><b>Solution</b> (click to expand)</summary>

UDP's length field is redundant with IP's total length, but the parser must still check that the two are
consistent. A datagram claiming more bytes than were received is truncated or malicious.

```cpp
namespace ex8_2 {
struct UdpHeader { uint16_t src_port = 0, dst_port = 0, length = 0, checksum = 0; };
std::optional<UdpHeader> parse_udp(const uint8_t* p, size_t n) {
    if (n < 8) return std::nullopt;
    UdpHeader h;
    h.src_port = lib8::read_be16(p);
    h.dst_port = lib8::read_be16(p + 2);
    h.length   = lib8::read_be16(p + 4);
    h.checksum = lib8::read_be16(p + 6);
    if (h.length < 8 || h.length > n) return std::nullopt;   // inconsistent length
    return h;
}
}
```

</details>

**Q8.2** *(Compute)* — A UDP header reads `c3 50 00 35 00 25 1f 9a`. Give the source port, destination port,
and payload size. Which application is probably involved?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Source port 0xc350 = **50000** (ephemeral), destination port 0x0035 = **53** → **DNS**; length 0x0025 =
37 → payload = 37 − 8 = **29 bytes**. The checksum is `1f9a`. A destination port in the
well-known range identifies the service, while the source port is the client's ephemeral one.

</details>

## 8.3 The transport checksum and the pseudo-header

UDP and TCP checksums use the same Internet checksum as IPv4 (§6.7), but computed over a
12-byte **pseudo-header** followed by the whole segment (header + data):

```
pseudo-header (never transmitted, only summed)
+-----------------------------------------------+
|               source IP address               |
+-----------------------------------------------+
|             destination IP address            |
+-----------+-----------+-----------------------+
|   zero    | protocol  |  UDP/TCP length       |   protocol: 6 = TCP, 17 = UDP
+-----------+-----------+-----------------------+
```

Including the IP addresses lets the receiver detect a segment delivered to the **wrong host**
(e.g. corrupted addresses). For UDP over IPv4 a checksum of `0` means "none computed", so a
computed value of `0` is sent as `0xFFFF`.

In [101]:
namespace lib8b {

// PROVIDED (you wrote this in §6.7)
uint16_t internet_checksum(const uint8_t* p, size_t n) {
    uint32_t sum = 0;
    for (size_t i = 0; i + 1 < n; i += 2)
        sum += (uint32_t(p[i]) << 8) | p[i + 1];      // big-endian 16-bit words
    if (n % 2) sum += uint32_t(p[n - 1]) << 8;       // odd length: pad with a zero byte
    while (sum >> 16) sum = (sum & 0xFFFF) + (sum >> 16);   // end-around carry
    return uint16_t(~sum);
}
bool ipv4_header_ok(const uint8_t* p, size_t n) {
    if (n < 20) return false;
    size_t hl = size_t(p[0] & 0x0F) * 4;
    if (hl < 20 || hl > n) return false;
    return internet_checksum(p, hl) == 0;             // includes the stored checksum
}

} // namespace lib8b

### ✍️ Exercise 8.3 — Compute and verify a UDP/TCP checksum

`transport_checksum(src_ip, dst_ip, proto, seg, len)`: build pseudo-header + segment in a `std::vector<uint8_t>`,
then return `lib8b::internet_checksum` over it. Verifying a received segment means that this returns `0`.

In [102]:
namespace ex8_3 {
uint16_t transport_checksum(uint32_t src_ip, uint32_t dst_ip, uint8_t proto,
                            const uint8_t* seg, size_t len) {
    std::vector<uint8_t> buf(12);
    lib8::write_be32(&buf[0], src_ip);
    lib8::write_be32(&buf[4], dst_ip);
    buf[8] = 0;
    buf[9] = proto;
    lib8::write_be16(&buf[10], uint16_t(len));
    buf.insert(buf.end(), seg, seg + len);          // then the whole segment
    return lib8b::internet_checksum(buf.data(), buf.size());
}
}

**Test for Exercise 8.3** — run the next cell.

In [103]:
{
    auto ip = [](const char* s) { return lib8::parse_ipv4(s).value_or(0); };
    uint32_t src = ip("192.168.1.23"), dst = ip("192.168.1.1");
    const auto& d = frames8::udp_dns;
    CHECK_EQ(ex8_3::transport_checksum(src, dst, 17, d.data(), d.size()), 0);     // verify: sums to 0

    std::vector<uint8_t> z = d;  z[6] = 0; z[7] = 0;                             // sender's view
    CHECK_EQ(ex8_3::transport_checksum(src, dst, 17, z.data(), z.size()), 0x486e);

    uint32_t wrong_dst = ip("192.168.1.2");                                      // misdelivered
    CHECK(ex8_3::transport_checksum(src, wrong_dst, 17, d.data(), d.size()) != 0);
}

✅ ex8_3::transport_checksum(src, dst, 17, d.data(), d.size()) == 0
✅ ex8_3::transport_checksum(src, dst, 17, z.data(), z.size()) == 0x486e
✅ ex8_3::transport_checksum(src, wrong_dst, 17, d.data(), d.size()) != 0


<details>
<summary><b>Solution</b> (click to expand)</summary>

Reusing the IPv4 checksum function is the point: one algorithm, applied to a different byte range. The last check
shows the pseudo-header's purpose: the payload bytes are intact, but a receiver at the wrong address rejects the
segment. That is also why NAT (§7.2) must patch transport checksums when it rewrites addresses.

```cpp
namespace ex8_3 {
uint16_t transport_checksum(uint32_t src_ip, uint32_t dst_ip, uint8_t proto,
                            const uint8_t* seg, size_t len) {
    std::vector<uint8_t> buf(12);
    lib8::write_be32(&buf[0], src_ip);
    lib8::write_be32(&buf[4], dst_ip);
    buf[8] = 0;
    buf[9] = proto;
    lib8::write_be16(&buf[10], uint16_t(len));
    buf.insert(buf.end(), seg, seg + len);          // then the whole segment
    return lib8b::internet_checksum(buf.data(), buf.size());
}
}
```

</details>

## 8.4 TCP: a reliable byte stream

**TCP** (Transmission Control Protocol) gives two programs a **connection** carrying a reliable,
in-order **byte stream** in each direction. The stream has **no message boundaries**: two
`send()` calls of 5 bytes may arrive as one 10-byte `recv()`, or as 3 + 7.

Every byte of the stream is numbered:
* the **sequence number** in a segment is the number of its **first** data byte;
* each side picks a random **ISN** (initial sequence number); the connection-opening **SYN** and
  the closing **FIN** each consume one number;
* the **acknowledgment number** is the number of the **next byte expected**. It is *cumulative*:
  "I have everything before this".

```
ISN = 1000:   SYN uses 1000 | data bytes 1001..1100 | 1101..1300 | ...
segment:       [seq=1001, 100 bytes] -> receiver answers ack=1101
```

**Q8.3** *(Compute)* — A client's ISN is 1000. After the SYN it sends three data segments of 100, 200 and 50 bytes.
What is the sequence number of each segment, and what ACK number does the server send once all have arrived? If only the second
segment is lost, what ACK does the server send after receiving the first and the third?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Sequence numbers: **1001, 1101, 1301**; final ACK = **1351** (the next byte expected).
With the second segment lost, the server can only acknowledge contiguous data, so both ACKs say **1101**. The
repeated ACK (a *duplicate ACK*) tells the sender that something after byte 1100 is missing.

*Misconceptions:* (1) sequence numbers count **bytes**, not segments; (2) the ACK number is the *next* byte expected,
not the last one received; (3) the SYN consumes a sequence number, so data starts at ISN+1.

</details>

## 8.5 The TCP header

```
byte  0          1          2          3
   +----------+----------+----------+----------+
 0 |     source port     |  destination port   |
   +---------------------+---------------------+
 4 |              sequence number              |
   +-------------------------------------------+
 8 |           acknowledgment number           |
   +-----+----+----------+---------------------+
12 | off |rsv |  flags   |       window        |   off = data offset: header length in 32-bit words (5..15)
   +-----+----+----------+---------------------+
16 |      checksum       |   urgent pointer    |
   +---------------------+---------------------+
20 |      options (if off > 5) ...             |   e.g. MSS: kind 2, length 4, 16-bit value
```

Flags (byte 13, bit 7 → bit 0): `CWR ECE URG ACK PSH RST SYN FIN`.
**SYN** opens a connection; **ACK** means the acknowledgment field is valid (set on every segment after the first);
**FIN**: "I have finished sending"; **RST**: abort; **PSH**: deliver to the application promptly;
URG/ECE/CWR are rarely relevant here. **window**: how many bytes the receiver can accept (§8.10).
**MSS** (maximum segment size): the largest payload the sender of the option wants to receive per segment.

In [104]:
namespace frames8b {
// SYN|ACK from 93.184.216.34:80 to 192.168.1.23:51514, with an MSS option (data offset 6)
const std::vector<uint8_t> tcp_synack = {
    0x00, 0x50, 0xc9, 0x3a, 0x7e, 0x3f, 0x1a, 0x02, 0x00, 0x00, 0xf1, 0xa3,
    0x60, 0x12, 0xfe, 0x88, 0x4e, 0x83, 0x00, 0x00, 0x02, 0x04, 0x05, 0xb4,
};
// PSH|ACK carrying an HTTP request
const std::vector<uint8_t> tcp_data = {
    0xc9, 0x3a, 0x00, 0x50, 0x00, 0x00, 0xf1, 0xa3, 0x7e, 0x3f, 0x1a, 0x03,
    0x50, 0x18, 0x01, 0xf6, 0x84, 0x19, 0x00, 0x00, 0x47, 0x45, 0x54, 0x20,
    0x2f, 0x20, 0x48, 0x54, 0x54, 0x50, 0x2f, 0x31, 0x2e, 0x30, 0x0d, 0x0a,
    0x0d, 0x0a,
};
}

In [105]:
hexdump(frames8b::tcp_synack);

0000  00 50 c9 3a 7e 3f 1a 02  00 00 f1 a3 60 12 fe 88  |.P.:~?......`...|
0010  4e 83 00 00 02 04 05 b4                           |N.......|


In [106]:
namespace lib8c {

// PROVIDED (you wrote this in §1.4)
uint8_t get_field(uint8_t b, int shift, int width) {
    uint8_t mask = uint8_t((1u << width) - 1);   // width ones, e.g. width 3 -> 0b111
    return uint8_t((b >> shift) & mask);
}
uint8_t pack_nibbles(uint8_t hi, uint8_t lo) {
    return uint8_t((hi << 4) | (lo & 0x0F));
}

} // namespace lib8c

### ✍️ Exercise 8.5 — Parse a TCP header and decode the flags

* `parse_tcp(p, n)`: return `std::nullopt` if `n < 20`, `data_offset < 5`, or the header does not fit in `n`.
* `flags_to_string(f)`: flag names joined by `|`, **least-significant bit first** (`FIN`, `SYN`, `RST`, `PSH`,
  `ACK`, `URG`, `ECE`, `CWR`), so that 0x12 → `"SYN|ACK"`, matching Wireshark. Return `"none"` for 0.

In [107]:
namespace ex8_5 {
struct TcpHeader {
    uint16_t src_port = 0, dst_port = 0, window = 0, checksum = 0, urgent = 0;
    uint32_t seq = 0, ack = 0;
    uint8_t data_offset = 0, flags = 0;
    size_t header_len() const { return size_t(data_offset) * 4; }
};
std::optional<TcpHeader> parse_tcp(const uint8_t* p, size_t n) {
    if (n < 20) return std::nullopt;
    TcpHeader h;
    h.src_port    = lib8::read_be16(p);
    h.dst_port    = lib8::read_be16(p + 2);
    h.seq         = lib8::read_be32(p + 4);
    h.ack         = lib8::read_be32(p + 8);
    h.data_offset = lib8c::get_field(p[12], 4, 4);   // high nibble of byte 12
    h.flags       = p[13];
    h.window      = lib8::read_be16(p + 14);
    h.checksum    = lib8::read_be16(p + 16);
    h.urgent      = lib8::read_be16(p + 18);
    if (h.data_offset < 5 || h.header_len() > n) return std::nullopt;
    return h;
}
std::string flags_to_string(uint8_t f) {
    static const char* names[8] = {"FIN", "SYN", "RST", "PSH", "ACK", "URG", "ECE", "CWR"};
    std::string s;
    for (int i = 0; i < 8; ++i) {
        if (!(f & (1u << i))) continue;
        if (!s.empty()) s += "|";
        s += names[i];
    }
    return s.empty() ? "none" : s;
}
}

**Test for Exercise 8.5** — run the next cell.

In [108]:
{
    const auto& a = frames8b::tcp_synack;
    auto h = ex8_5::parse_tcp(a.data(), a.size()).value_or(ex8_5::TcpHeader{});
    CHECK_EQ(h.src_port, 80);
    CHECK_EQ(h.dst_port, 51514);
    CHECK_EQ(h.seq, 0x7e3f1a02u);
    CHECK_EQ(h.ack, 0x0000f1a3u);
    CHECK_EQ(h.header_len(), 24u);                       // data offset 6: 4 bytes of options
    CHECK_EQ(h.window, 65160);
    CHECK_EQ(ex8_5::flags_to_string(h.flags), "SYN|ACK");

    const auto& d = frames8b::tcp_data;
    auto h2 = ex8_5::parse_tcp(d.data(), d.size()).value_or(ex8_5::TcpHeader{});
    CHECK_EQ(ex8_5::flags_to_string(h2.flags), "PSH|ACK");
    CHECK_EQ(d.size() - h2.header_len(), 18u);           // payload = the HTTP request
    CHECK_EQ(ex8_5::flags_to_string(0x11), "FIN|ACK");
    CHECK_EQ(ex8_5::flags_to_string(0x04), "RST");
    CHECK_EQ(ex8_5::flags_to_string(0x00), "none");
    CHECK(!ex8_5::parse_tcp(a.data(), 22).has_value());   // options cut off
}

✅ h.src_port == 80
✅ h.dst_port == 51514
✅ h.seq == 0x7e3f1a02u
✅ h.ack == 0x0000f1a3u
✅ h.header_len() == 24u
✅ h.window == 65160
✅ ex8_5::flags_to_string(h.flags) == "SYN|ACK"
✅ ex8_5::flags_to_string(h2.flags) == "PSH|ACK"
✅ d.size() - h2.header_len() == 18u
✅ ex8_5::flags_to_string(0x11) == "FIN|ACK"
✅ ex8_5::flags_to_string(0x04) == "RST"
✅ ex8_5::flags_to_string(0x00) == "none"
✅ !ex8_5::parse_tcp(a.data(), 22).has_value()


<details>
<summary><b>Solution</b> (click to expand)</summary>

The data offset is exactly the IPv4 IHL pattern: a 4-bit count of 32-bit words. Checking that the full header,
options included, fits in the buffer prevents reading options past the end. Note how the SYN|ACK acknowledges
`0x0000f1a3`, one more than the client's ISN, because the SYN consumed one sequence number.

```cpp
namespace ex8_5 {
struct TcpHeader {
    uint16_t src_port = 0, dst_port = 0, window = 0, checksum = 0, urgent = 0;
    uint32_t seq = 0, ack = 0;
    uint8_t data_offset = 0, flags = 0;
    size_t header_len() const { return size_t(data_offset) * 4; }
};
std::optional<TcpHeader> parse_tcp(const uint8_t* p, size_t n) {
    if (n < 20) return std::nullopt;
    TcpHeader h;
    h.src_port    = lib8::read_be16(p);
    h.dst_port    = lib8::read_be16(p + 2);
    h.seq         = lib8::read_be32(p + 4);
    h.ack         = lib8::read_be32(p + 8);
    h.data_offset = lib8c::get_field(p[12], 4, 4);   // high nibble of byte 12
    h.flags       = p[13];
    h.window      = lib8::read_be16(p + 14);
    h.checksum    = lib8::read_be16(p + 16);
    h.urgent      = lib8::read_be16(p + 18);
    if (h.data_offset < 5 || h.header_len() > n) return std::nullopt;
    return h;
}
std::string flags_to_string(uint8_t f) {
    static const char* names[8] = {"FIN", "SYN", "RST", "PSH", "ACK", "URG", "ECE", "CWR"};
    std::string s;
    for (int i = 0; i < 8; ++i) {
        if (!(f & (1u << i))) continue;
        if (!s.empty()) s += "|";
        s += names[i];
    }
    return s.empty() ? "none" : s;
}
}
```

</details>

**Q8.4** *(Compute)* — Byte 12 of a TCP header is `0x80` and byte 13 is `0x10`. What is the header length, which flags are set,
and what is likely in the options?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Data offset = 0x8 = 8 words = **32 bytes** → 12 bytes of options. Flags 0x10 = **ACK** only, an ordinary segment of
an established connection. On Linux the 12 option bytes are typically NOP, NOP, and a 10-byte *timestamp* option
(used for RTT measurement). *Misconception:* reading `0x80` as 128 bytes or 8 bytes; the high nibble counts **words**.

</details>

## 8.6 The three-way handshake

```
client (CLOSED)                                   server (LISTEN)
   |---- SYN        seq=x ------------------------->|   SYN_SENT        -> SYN_RCVD
   |<--- SYN|ACK    seq=y, ack=x+1 -----------------|
   |---- ACK        seq=x+1, ack=y+1 -------------->|   ESTABLISHED     -> ESTABLISHED
```

Each SYN carries the sender's random ISN and options such as MSS and window scaling (§8.10).
Each ACK proves that the other side's ISN arrived. Data may start with the third segment.
A SYN to a port with no listener is answered with **RST** ("connection refused").

**Q8.5** *(Why?)* — Why three segments? What could go wrong with a two-way handshake (SYN, then SYN|ACK, and the server
immediately considers the connection open)?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Both directions need their ISN **confirmed**: the client's ISN is acknowledged by the SYN|ACK, and the server's ISN
only by the final ACK. With two messages the server never learns whether its ISN arrived. Worse, an **old duplicate
SYN** from a long-dead connection, delayed in the network, would make the server open a bogus connection and allocate
resources. With three messages the client, seeing a SYN|ACK for a connection it never started, answers RST.
ISNs are random so that old segments, and attackers who cannot see the traffic, cannot guess valid sequence numbers.

</details>

## 8.7 Teardown and `TIME_WAIT`

Each direction is closed independently with a FIN, which the other side ACKs (four segments; the
middle two are often combined):

```
A (closes first: "active close")                    B
   |---- FIN seq=u ------------------------------->|   A: FIN_WAIT_1   B: CLOSE_WAIT
   |<--- ACK ack=u+1 ------------------------------|   A: FIN_WAIT_2   (B may keep sending data)
   |<--- FIN seq=v --------------------------------|                   B: LAST_ACK
   |---- ACK ack=v+1 ----------------------------->|   A: TIME_WAIT    B: CLOSED
   |   ... wait 2 x MSL (Linux: 60 s) ...          |
   A: CLOSED
```

**MSL** (maximum segment lifetime) is the longest a segment is assumed to survive in the network.
The side that closes first stays in **TIME_WAIT** for 2 × MSL, for two reasons:
(1) if its final ACK is lost, B retransmits its FIN, and A must still be there to re-ACK it;
(2) delayed segments of this connection must die before the same 5-tuple can be reused.
**RST** instead aborts a connection immediately, with no handshake.

**Q8.6** *(Compute)* — A load-testing client opens and actively closes short connections to one server endpoint as fast as possible.
Each closed connection's port sits in TIME_WAIT for 60 s. With 28232 ephemeral ports, what sustained rate of new connections
can it achieve?

<details>
<summary><b>Solution</b> (click to expand)</summary>

28232 ports / 60 s ≈ **471 connections per second**. Beyond that, `connect()` fails with "address
not available" because every port for that 5-tuple is still in TIME_WAIT. Remedies: keep connections alive and
reuse them (HTTP keep-alive, connection pools), let the *server* close first, or spread load over more
IPs/ports. *Misconception:* "TIME_WAIT is a bug to disable". It protects against stale segments corrupting a new
connection.

</details>

## 8.8 A simplified TCP state machine

TCP's connection life cycle is a finite-state machine. A simplified version (ignoring simultaneous
open and close, and RST) has these transitions; anything else is **invalid** in our model:

```
CLOSED       + APP_LISTEN    -> LISTEN       send (nothing)
CLOSED       + APP_CONNECT   -> SYN_SENT     send SYN
LISTEN       + RCV_SYN       -> SYN_RCVD     send SYN|ACK
SYN_SENT     + RCV_SYN_ACK   -> ESTABLISHED  send ACK
SYN_RCVD     + RCV_ACK       -> ESTABLISHED  send (nothing)
ESTABLISHED  + APP_CLOSE     -> FIN_WAIT_1   send FIN
ESTABLISHED  + RCV_FIN       -> CLOSE_WAIT   send ACK
FIN_WAIT_1   + RCV_ACK       -> FIN_WAIT_2   send (nothing)
FIN_WAIT_2   + RCV_FIN       -> TIME_WAIT    send ACK
CLOSE_WAIT   + APP_CLOSE     -> LAST_ACK     send FIN
LAST_ACK     + RCV_ACK       -> CLOSED       send (nothing)
TIME_WAIT    + TIMEOUT_2MSL  -> CLOSED       send (nothing)
```

### ✍️ Exercise 8.8 — Run the TCP state machine

Implement `step(s, e)` from the table above. For a pair not in the table return `{s, "INVALID"}`: the state is unchanged.
(`enum class` is a scoped enumeration: write `State::LISTEN`; `int(s)` gives its index.)

In [109]:
namespace ex8_8 {
enum class State { CLOSED, LISTEN, SYN_SENT, SYN_RCVD, ESTABLISHED, FIN_WAIT_1, FIN_WAIT_2, CLOSE_WAIT, LAST_ACK, TIME_WAIT };
enum class Event { APP_LISTEN, APP_CONNECT, APP_CLOSE, RCV_SYN, RCV_SYN_ACK, RCV_ACK, RCV_FIN, TIMEOUT_2MSL };
const char* name(State s) { static const char* n[] = {"CLOSED", "LISTEN", "SYN_SENT", "SYN_RCVD", "ESTABLISHED", "FIN_WAIT_1", "FIN_WAIT_2", "CLOSE_WAIT", "LAST_ACK", "TIME_WAIT"}; return n[int(s)]; }
const char* name(Event e) { static const char* n[] = {"APP_LISTEN", "APP_CONNECT", "APP_CLOSE", "RCV_SYN", "RCV_SYN_ACK", "RCV_ACK", "RCV_FIN", "TIMEOUT_2MSL"}; return n[int(e)]; }
struct Transition { State next; std::string send; };   // send: "SYN", "SYN|ACK", "ACK", "FIN", "" or "INVALID"
Transition step(State s, Event e) {
    if (s == State::CLOSED && e == Event::APP_LISTEN) return {State::LISTEN, ""};
    if (s == State::CLOSED && e == Event::APP_CONNECT) return {State::SYN_SENT, "SYN"};
    if (s == State::LISTEN && e == Event::RCV_SYN) return {State::SYN_RCVD, "SYN|ACK"};
    if (s == State::SYN_SENT && e == Event::RCV_SYN_ACK) return {State::ESTABLISHED, "ACK"};
    if (s == State::SYN_RCVD && e == Event::RCV_ACK) return {State::ESTABLISHED, ""};
    if (s == State::ESTABLISHED && e == Event::APP_CLOSE) return {State::FIN_WAIT_1, "FIN"};
    if (s == State::ESTABLISHED && e == Event::RCV_FIN) return {State::CLOSE_WAIT, "ACK"};
    if (s == State::FIN_WAIT_1 && e == Event::RCV_ACK) return {State::FIN_WAIT_2, ""};
    if (s == State::FIN_WAIT_2 && e == Event::RCV_FIN) return {State::TIME_WAIT, "ACK"};
    if (s == State::CLOSE_WAIT && e == Event::APP_CLOSE) return {State::LAST_ACK, "FIN"};
    if (s == State::LAST_ACK && e == Event::RCV_ACK) return {State::CLOSED, ""};
    if (s == State::TIME_WAIT && e == Event::TIMEOUT_2MSL) return {State::CLOSED, ""};
    return {s, "INVALID"};   // not allowed in this state
}
}

**Test for Exercise 8.8** — run the next cell.

In [110]:
{
    using ex8_8::State; using ex8_8::Event;
    struct Run { const char* who; std::vector<Event> events; std::vector<std::string> want; };
    std::vector<Run> runs = {
        {"client", {Event::APP_CONNECT, Event::RCV_SYN_ACK, Event::APP_CLOSE, Event::RCV_ACK, Event::RCV_FIN, Event::TIMEOUT_2MSL}, {"SYN_SENT/SYN", "ESTABLISHED/ACK", "FIN_WAIT_1/FIN", "FIN_WAIT_2/", "TIME_WAIT/ACK", "CLOSED/"}},
        {"server", {Event::APP_LISTEN, Event::RCV_SYN, Event::RCV_ACK, Event::RCV_FIN, Event::APP_CLOSE, Event::RCV_ACK}, {"LISTEN/", "SYN_RCVD/SYN|ACK", "ESTABLISHED/", "CLOSE_WAIT/ACK", "LAST_ACK/FIN", "CLOSED/"}},
        {"bad", {Event::RCV_ACK, Event::APP_CONNECT, Event::RCV_FIN}, {"CLOSED/INVALID", "SYN_SENT/SYN", "SYN_SENT/INVALID"}},
    };
    for (const auto& r : runs) {
        State s = State::CLOSED;
        std::vector<std::string> got;
        std::cout << r.who << ":\n";
        for (Event e : r.events) {
            auto t = ex8_8::step(s, e);
            std::cout << "  " << ex8_8::name(s) << " + " << ex8_8::name(e) << " -> " << ex8_8::name(t.next)
                      << (t.send.empty() ? "" : "   send " + t.send) << "\n";
            got.push_back(std::string(ex8_8::name(t.next)) + "/" + t.send);
            s = t.next;
        }
        CHECK_EQ(got, r.want);
    }
}

client:
  CLOSED + APP_CONNECT -> SYN_SENT   send SYN
  SYN_SENT + RCV_SYN_ACK -> ESTABLISHED   send ACK
  ESTABLISHED + APP_CLOSE -> FIN_WAIT_1   send FIN
  FIN_WAIT_1 + RCV_ACK -> FIN_WAIT_2
  FIN_WAIT_2 + RCV_FIN -> TIME_WAIT   send ACK
  TIME_WAIT + TIMEOUT_2MSL -> CLOSED
✅ got == r.want
server:
  CLOSED + APP_LISTEN -> LISTEN
  LISTEN + RCV_SYN -> SYN_RCVD   send SYN|ACK
  SYN_RCVD + RCV_ACK -> ESTABLISHED
  ESTABLISHED + RCV_FIN -> CLOSE_WAIT   send ACK
  CLOSE_WAIT + APP_CLOSE -> LAST_ACK   send FIN
  LAST_ACK + RCV_ACK -> CLOSED
✅ got == r.want
bad:
  CLOSED + RCV_ACK -> CLOSED   send INVALID
  CLOSED + APP_CONNECT -> SYN_SENT   send SYN
  SYN_SENT + RCV_FIN -> SYN_SENT   send INVALID
✅ got == r.want


<details>
<summary><b>Solution</b> (click to expand)</summary>

A table-driven state machine is easy to audit against a specification; real kernels encode the same diagram (from
RFC 9293, the current TCP specification) with more states (`CLOSING` for simultaneous close) and more events (RST,
timeouts). Note which side ends in `TIME_WAIT`: the one that sent the **first** FIN (here the client), while the
passive closer goes through `CLOSE_WAIT` → `LAST_ACK`. A server with thousands of sockets stuck in `CLOSE_WAIT` has
an application bug: it never called `close()`.

```cpp
namespace ex8_8 {
enum class State { CLOSED, LISTEN, SYN_SENT, SYN_RCVD, ESTABLISHED, FIN_WAIT_1, FIN_WAIT_2, CLOSE_WAIT, LAST_ACK, TIME_WAIT };
enum class Event { APP_LISTEN, APP_CONNECT, APP_CLOSE, RCV_SYN, RCV_SYN_ACK, RCV_ACK, RCV_FIN, TIMEOUT_2MSL };
const char* name(State s) { static const char* n[] = {"CLOSED", "LISTEN", "SYN_SENT", "SYN_RCVD", "ESTABLISHED", "FIN_WAIT_1", "FIN_WAIT_2", "CLOSE_WAIT", "LAST_ACK", "TIME_WAIT"}; return n[int(s)]; }
const char* name(Event e) { static const char* n[] = {"APP_LISTEN", "APP_CONNECT", "APP_CLOSE", "RCV_SYN", "RCV_SYN_ACK", "RCV_ACK", "RCV_FIN", "TIMEOUT_2MSL"}; return n[int(e)]; }
struct Transition { State next; std::string send; };   // send: "SYN", "SYN|ACK", "ACK", "FIN", "" or "INVALID"
Transition step(State s, Event e) {
    if (s == State::CLOSED && e == Event::APP_LISTEN) return {State::LISTEN, ""};
    if (s == State::CLOSED && e == Event::APP_CONNECT) return {State::SYN_SENT, "SYN"};
    if (s == State::LISTEN && e == Event::RCV_SYN) return {State::SYN_RCVD, "SYN|ACK"};
    if (s == State::SYN_SENT && e == Event::RCV_SYN_ACK) return {State::ESTABLISHED, "ACK"};
    if (s == State::SYN_RCVD && e == Event::RCV_ACK) return {State::ESTABLISHED, ""};
    if (s == State::ESTABLISHED && e == Event::APP_CLOSE) return {State::FIN_WAIT_1, "FIN"};
    if (s == State::ESTABLISHED && e == Event::RCV_FIN) return {State::CLOSE_WAIT, "ACK"};
    if (s == State::FIN_WAIT_1 && e == Event::RCV_ACK) return {State::FIN_WAIT_2, ""};
    if (s == State::FIN_WAIT_2 && e == Event::RCV_FIN) return {State::TIME_WAIT, "ACK"};
    if (s == State::CLOSE_WAIT && e == Event::APP_CLOSE) return {State::LAST_ACK, "FIN"};
    if (s == State::LAST_ACK && e == Event::RCV_ACK) return {State::CLOSED, ""};
    if (s == State::TIME_WAIT && e == Event::TIMEOUT_2MSL) return {State::CLOSED, ""};
    return {s, "INVALID"};   // not allowed in this state
}
}
```

</details>

## 8.9 Reliability: from stop-and-wait to sliding windows

Reliability rests on three mechanisms: **acknowledgements**, a **retransmission timer** (resend
if no ACK arrives within about one RTT plus a margin), and **sequence numbers** (so that the
receiver can discard duplicates and reorder).

**Stop-and-wait** sends one segment and waits for its ACK. The link is then busy only a tiny
fraction of the time:
$U = \dfrac{L/R}{\text{RTT} + L/R}$.

A **sliding window** allows up to $W$ unacknowledged segments *in flight*, so $U$ is multiplied by $W$
(capped at 1). To fill the pipe, $W \cdot L$ must reach the **BDP** (§2.7).

```
seq:   0   1   2   3 | 4   5   6   7 | 8   9
      [ acked        ][ in flight, W=4 ][ not yet sent ]
                       ^ base = oldest unacked; slides right as ACKs arrive
```

**Go-Back-N**: the receiver accepts only in-order data and sends cumulative ACKs; after a loss
the sender resends *everything* from the lost segment on. **Selective repeat** buffers
out-of-order data and resends only what is missing. TCP sits in between: cumulative ACKs, out-of-order
buffering, and **fast retransmit** after 3 duplicate ACKs, without waiting for the timer.

**Q8.7** *(Compute)* — 1 Gb/s link, RTT 30 ms, 1500-byte segments. What is the utilisation with stop-and-wait? With a window of
100 segments? How many segments must be in flight to fill the link?

<details>
<summary><b>Solution</b> (click to expand)</summary>

$L/R$ = 12 µs. Stop-and-wait: $U = 12\,\mu s / 30.012\,\text{ms}$ ≈ **0.00040** (0.40 Mb/s of 1000).
Window 100: $U$ ≈ **0.0400** (40 Mb/s). To fill the link: $W \ge (RTT + L/R)/(L/R)$ =
**2501 segments** ≈ BDP / L = 2500. Pipelining is not an optimisation;
without it, long paths are unusable.

</details>

### ✍️ Exercise 8.9 — Simulate Go-Back-N with a scripted loss

Simulate rounds (one RTT each). In every round the sender transmits all segments `base … min(base+W, n)−1`.
A segment in `lose` is lost the **first** time it is sent (log it with an `x` suffix, e.g. `5x`). The receiver
accepts a segment only if it equals `expected` (then `expected++`); anything else is discarded. At the end of
the round the cumulative ACK is `expected`, and `base = expected`. Log each round exactly as
`round <r>: sent <segments> | ack <base>`. Return the log; count rounds and transmissions in the out-parameters.

With `n=10, W=4, lose={5}` the log is:
```
round 1: sent 0 1 2 3 | ack 4
round 2: sent 4 5x 6 7 | ack 5
round 3: sent 5 6 7 8 | ack 9
round 4: sent 9 | ack 10
```
(`std::set<int>` is an ordered set; `s.count(x)` tests membership and `s.insert(x)` adds.)

In [111]:
namespace ex8_9 {
std::vector<std::string> go_back_n(int n, int W, const std::set<int>& lose, int& rounds, int& transmissions) {
    rounds = 0; transmissions = 0;
    std::vector<std::string> log;
    std::set<int> already_lost;
    int base = 0;
    while (base < n) {
        ++rounds;
        int expected = base;
        std::string sent;
        for (int s = base; s < std::min(base + W, n); ++s) {
            ++transmissions;
            if (!sent.empty()) sent += " ";
            if (lose.count(s) && !already_lost.count(s)) {   // lost on its first transmission
                already_lost.insert(s);
                sent += std::to_string(s) + "x";
            } else {
                sent += std::to_string(s);
                if (s == expected) ++expected;               // in order: accept
            }                                                // else: out of order, discarded
        }
        base = expected;                                     // cumulative ACK
        log.push_back("round " + std::to_string(rounds) + ": sent " + sent + " | ack " + std::to_string(base));
    }
    return log;
}
}

**Test for Exercise 8.9** — run the next cell.

In [112]:
{
    int rounds = 0, tx = 0;
    auto log = ex8_9::go_back_n(10, 4, {5}, rounds, tx);
    for (const auto& line : log) std::cout << line << "\n";
    CHECK_EQ(log, (std::vector<std::string>{"round 1: sent 0 1 2 3 | ack 4", "round 2: sent 4 5x 6 7 | ack 5", "round 3: sent 5 6 7 8 | ack 9", "round 4: sent 9 | ack 10"}));
    CHECK_EQ(rounds, 4);
    CHECK_EQ(tx, 13);

    auto clean = ex8_9::go_back_n(10, 4, {}, rounds, tx);
    CHECK_EQ(rounds, 3);
    CHECK_EQ(tx, 10);
}

round 1: sent 0 1 2 3 | ack 4
round 2: sent 4 5x 6 7 | ack 5
round 3: sent 5 6 7 8 | ack 9
round 4: sent 9 | ack 10
✅ log == (std::vector<std::string>{"round 1: sent 0 1 2 3 | ack 4", "round 2: sent 4 5x 6 7 | ack 5", "round 3: sent 5 6 7 8 | ack 9", "round 4: sent 9 | ack 10"})
✅ rounds == 4
✅ tx == 13
✅ rounds == 3
✅ tx == 10


<details>
<summary><b>Solution</b> (click to expand)</summary>

One lost segment cost 3 extra transmissions: segment 5 itself plus 6 and 7, which arrived fine but were
discarded because they were out of order. That waste is exactly what selective repeat and TCP's out-of-order buffering
avoid. The loss also costs a whole extra round (RTT) of delay, which no retransmission scheme can avoid.

```cpp
namespace ex8_9 {
std::vector<std::string> go_back_n(int n, int W, const std::set<int>& lose, int& rounds, int& transmissions) {
    rounds = 0; transmissions = 0;
    std::vector<std::string> log;
    std::set<int> already_lost;
    int base = 0;
    while (base < n) {
        ++rounds;
        int expected = base;
        std::string sent;
        for (int s = base; s < std::min(base + W, n); ++s) {
            ++transmissions;
            if (!sent.empty()) sent += " ";
            if (lose.count(s) && !already_lost.count(s)) {   // lost on its first transmission
                already_lost.insert(s);
                sent += std::to_string(s) + "x";
            } else {
                sent += std::to_string(s);
                if (s == expected) ++expected;               // in order: accept
            }                                                // else: out of order, discarded
        }
        base = expected;                                     // cumulative ACK
        log.push_back("round " + std::to_string(rounds) + ": sent " + sent + " | ack " + std::to_string(base));
    }
    return log;
}
}
```

</details>

## 8.10 Flow control vs congestion control

These two mechanisms are easy to confuse. Both limit how much a sender may have in flight, but they protect different things:

| | **flow control** | **congestion control** |
|---|---|---|
| protects | the **receiver**'s buffer | the **network** (router queues) |
| window | **rwnd**: advertised by the receiver in every segment's *window* field | **cwnd**: computed by the sender, never transmitted |
| signal | "I have N bytes of free buffer space" | loss (or delay/ECN marks) implies queues are overflowing |

The sender may have at most $\min(\text{rwnd}, \text{cwnd})$ unacknowledged bytes in flight.
The window field is 16 bits (max 65 535 bytes), too small for large BDPs, so a **window scale**
option in the SYN multiplies it by $2^k$.

**Q8.8** *(Concept)* — Which mechanism limits throughput in each case? (a) A server streams to an old phone whose app reads
data slowly, over an idle network. (b) A fast desktop downloads over a shared, congested ISP link.

<details>
<summary><b>Solution</b> (click to expand)</summary>

(a) **Flow control**: the phone's receive buffer fills, it advertises a small (even zero) **rwnd**, and the
sender must pause however empty the network is. (b) **Congestion control**: the receiver has plenty of buffer,
but router queues overflow, losses occur, and the sender's **cwnd** shrinks. *Misconception:* "the window in the
TCP header is the congestion window". The header carries only rwnd; cwnd lives inside the sender.

</details>

## 8.11 Congestion control: slow start and AIMD

The classic TCP algorithm (Reno) adjusts **cwnd**, measured here in segments (MSS units), once per RTT:

* **Slow start**: cwnd starts small and **doubles** every RTT (+1 per ACK) until it reaches the
  threshold **ssthresh**. Exponential growth finds the available capacity quickly.
* **Congestion avoidance**: above ssthresh, grow **additively**: +1 segment per RTT.
* **Loss detected by 3 duplicate ACKs** (mild): ssthresh = cwnd/2 and cwnd = ssthresh, a
  **multiplicative decrease** (*fast recovery*).
* **Loss detected by a timeout** (severe, nothing is getting through): ssthresh = cwnd/2 and
  cwnd = 1, back to slow start.

Additive increase plus multiplicative decrease = **AIMD**, which produces TCP's sawtooth.
(Modern Linux defaults to CUBIC, which grows faster at high rates, but the structure is the same.)

### ✍️ Exercise 8.11 — Trace cwnd per RTT

`cwnd_trace(rtts, ssthresh, dupack_rtts, timeout_rtts)`: start with cwnd = 1. For each RTT `r = 0 … rtts−1`:
**record** cwnd, then update it:
* if `r` is in `dupack_rtts`: ssthresh = max(cwnd/2, 2); cwnd = ssthresh;
* else if `r` is in `timeout_rtts`: ssthresh = max(cwnd/2, 2); cwnd = 1;
* else if cwnd < ssthresh: cwnd = min(2·cwnd, ssthresh);
* else cwnd += 1.

(Integer division.) With ssthresh 16, a duplicate-ACK loss in RTT 9 and a timeout in RTT 15, the
expected trace is `1, 2, 4, 8, 16, 17, 18, 19, 20, 21, 10, 11, 12, 13, 14, 15, 1, 2, 4, 7`.

In [113]:
namespace ex8_11 {
std::vector<int> cwnd_trace(int rtts, int ssthresh, const std::set<int>& dupack_rtts,
                            const std::set<int>& timeout_rtts) {
    std::vector<int> out;
    int cwnd = 1;
    for (int r = 0; r < rtts; ++r) {
        out.push_back(cwnd);
        if (dupack_rtts.count(r)) {            // mild loss: halve (multiplicative decrease)
            ssthresh = std::max(cwnd / 2, 2);
            cwnd = ssthresh;
        } else if (timeout_rtts.count(r)) {    // severe loss: restart slow start
            ssthresh = std::max(cwnd / 2, 2);
            cwnd = 1;
        } else if (cwnd < ssthresh) {          // slow start: double
            cwnd = std::min(2 * cwnd, ssthresh);
        } else {                               // congestion avoidance: +1
            cwnd += 1;
        }
    }
    return out;
}
}

**Test for Exercise 8.11** — run the next cell.

In [114]:
{
    auto t = ex8_11::cwnd_trace(20, 16, {9}, {15});
    for (size_t r = 0; r < t.size(); ++r)
        std::cout << "RTT " << std::setw(2) << r << " | " << std::string(size_t(std::max(t[r], 0)), '#')
                  << " " << t[r] << "\n";
    CHECK_EQ(t, (std::vector<int>{1, 2, 4, 8, 16, 17, 18, 19, 20, 21, 10, 11, 12, 13, 14, 15, 1, 2, 4, 7}));
    CHECK_EQ(ex8_11::cwnd_trace(6, 64, {}, {}), (std::vector<int>{1, 2, 4, 8, 16, 32}));   // pure slow start
}

RTT  0 | # 1
RTT  1 | ## 2
RTT  2 | #### 4
RTT  3 | ######## 8
RTT  4 | ################ 16
RTT  5 | ################# 17
RTT  6 | ################## 18
RTT  7 | ################### 19
RTT  8 | #################### 20
RTT  9 | ##################### 21
RTT 10 | ########## 10
RTT 11 | ########### 11
RTT 12 | ############ 12
RTT 13 | ############# 13
RTT 14 | ############## 14
RTT 15 | ############### 15
RTT 16 | # 1
RTT 17 | ## 2
RTT 18 | #### 4
RTT 19 | ####### 7
✅ t == (std::vector<int>{1, 2, 4, 8, 16, 17, 18, 19, 20, 21, 10, 11, 12, 13, 14, 15, 1, 2, 4, 7})
✅ ex8_11::cwnd_trace(6, 64, {}, {}) == (std::vector<int>{1, 2, 4, 8, 16, 32})


<details>
<summary><b>Solution</b> (click to expand)</summary>

The chart shows the three regimes: exponential growth up to ssthresh (RTT 0–4), linear growth, halving at the
duplicate-ACK loss, linear growth again, then the collapse to 1 after the timeout, followed by a fresh slow start up to
the new, lower ssthresh. "Slow" start is actually the *fast* phase; the name contrasts it with the older behaviour of
sending a full window at once.

```cpp
namespace ex8_11 {
std::vector<int> cwnd_trace(int rtts, int ssthresh, const std::set<int>& dupack_rtts,
                            const std::set<int>& timeout_rtts) {
    std::vector<int> out;
    int cwnd = 1;
    for (int r = 0; r < rtts; ++r) {
        out.push_back(cwnd);
        if (dupack_rtts.count(r)) {            // mild loss: halve (multiplicative decrease)
            ssthresh = std::max(cwnd / 2, 2);
            cwnd = ssthresh;
        } else if (timeout_rtts.count(r)) {    // severe loss: restart slow start
            ssthresh = std::max(cwnd / 2, 2);
            cwnd = 1;
        } else if (cwnd < ssthresh) {          // slow start: double
            cwnd = std::min(2 * cwnd, ssthresh);
        } else {                               // congestion avoidance: +1
            cwnd += 1;
        }
    }
    return out;
}
}
```

</details>

**Q8.9** *(Why?)* — Why does TCP increase additively but decrease multiplicatively? What would go wrong with multiplicative increase
in congestion avoidance?

<details>
<summary><b>Solution</b> (click to expand)</summary>

AIMD makes competing flows **converge to a fair share**: when two flows both halve, the larger one loses more in
absolute terms, while additive increase adds the same amount to both; repeated cycles shrink the gap (Chiu & Jain's
result). Multiplicative decrease also backs off fast enough to drain overflowing queues. With multiplicative
*increase* above ssthresh, flows would overshoot capacity by large factors, causing bursts of heavy loss and never
converging to fairness. Slow start is multiplicative only while far below the last known safe rate.

</details>

## 8.12 UDP vs TCP: choosing

| | UDP | TCP |
|---|---|---|
| connection setup | none (the first datagram carries data) | 1 RTT handshake before data |
| delivery | unreliable, unordered, message boundaries kept | reliable, ordered byte stream |
| congestion / flow control | none (the application's job) | built in |
| head-of-line blocking | no | yes |
| header | 8 bytes | 20–60 bytes |

**Head-of-line blocking**: because TCP delivers bytes in order, one lost segment stalls delivery
of all later bytes, even if they already arrived, until the retransmission fills the gap.

**Q8.10** *(Concept)* — Pick UDP or TCP, with a one-line reason each: (a) a DNS lookup, (b) a video call, (c) downloading a
software update, (d) a multiplayer game sending player positions 60 times per second.

<details>
<summary><b>Solution</b> (click to expand)</summary>

(a) **UDP**: one small question and one small answer; a handshake would double the latency; if lost, simply retry.
(b) **UDP**: late audio is useless; a retransmitted frame would arrive after it should have been played, and
head-of-line blocking would freeze the call.
(c) **TCP**: every byte must arrive intact and in order; latency hardly matters.
(d) **UDP**: each update supersedes the previous one; a lost position should be skipped, not retransmitted.
*Misconception:* "UDP is faster". It is not faster on the wire; it simply lets the application skip reliability
it does not need.

</details>

### 🔁 Recap

- Ports multiplex programs; TCP connections are demultiplexed by the full 5-tuple, UDP by (dst IP, dst port).
- UDP header: 8 bytes (ports, length, checksum). UDP/TCP checksums cover a pseudo-header with the IP addresses.
- TCP numbers bytes: seq = first byte, ack = next byte expected (cumulative); SYN and FIN consume one number each.
- Handshake SYN → SYN|ACK → ACK; teardown FIN/ACK per direction; the active closer waits in TIME_WAIT for 2×MSL.
- Reliability = ACK + timeout + retransmit; windows fill the BDP. rwnd (flow control) protects the receiver, cwnd (slow start, AIMD) the network.

**You should now be able to explain…**

- [ ] how a segment finds its socket, and why one server port serves many clients;
- [ ] every field of the TCP header, and how to decode the flags byte;
- [ ] why the handshake has three messages, and why TIME_WAIT exists;
- [ ] the difference between flow control and congestion control;
- [ ] a cwnd trace through slow start, AIMD, and a timeout.

# Part 9 — Sockets in practice (loopback only)

Everything so far has been simulated. Now you use your operating system's real TCP/IP stack.
All traffic stays on **loopback** (`127.0.0.1`): packets never leave the machine, no special
privileges are needed, and nothing depends on an external network.

## 9.1 Sockets, file descriptors, and addresses

The kernel implements UDP, TCP and IP; a program uses them through **system calls** on a
**socket**. A socket is referred to by a **file descriptor** (fd), a small non-negative `int`
handle, like an open file. Failing calls return `-1` and set **`errno`**, a per-thread error code
(`std::strerror(errno)` gives its text).

| call | purpose |
|---|---|
| `socket(AF_INET, SOCK_DGRAM or SOCK_STREAM, 0)` | create a UDP or TCP socket → fd |
| `bind(fd, addr, len)` | attach a local IP and port; **port 0 = "kernel, pick a free one"** |
| `getsockname(fd, addr, &len)` | read back the local address (e.g. the chosen port) |
| `setsockopt(fd, SOL_SOCKET, SO_RCVTIMEO, &tv, sizeof tv)` | make blocking receives give up after `tv` |
| `close(fd)` | release the socket |

An IPv4 address is passed as `sockaddr_in` with `sin_family = AF_INET`, and `sin_port` and
`sin_addr.s_addr` in **network byte order** (`htons`, `htonl`, §1.7). `INADDR_LOOPBACK` is 127.0.0.1
as a host-order value. **`SO_RCVTIMEO`** takes a `timeval {tv_sec, tv_usec}`; a receive that times
out returns `-1` with `errno` = `EAGAIN`/`EWOULDBLOCK`. Every receiving socket in this part gets
a ~2 s timeout, so that no bug can hang the notebook.

The cell below gives two helpers: `Fd`, which closes its descriptor automatically when it goes out
of scope (the C++ idiom called **RAII**, resource acquisition is initialisation), and `sa`, a cast helper.

In [115]:
namespace lib9 {

// GIVEN helper (new here, not an exercise)
// RAII owner of a file descriptor: the destructor closes it on EVERY path out of a scope
// (normal return, early return, exception). Copying is forbidden so there is exactly one owner.
struct Fd {
    int fd = -1;
    explicit Fd(int f) : fd(f) {}
    ~Fd() { if (fd >= 0) ::close(fd); }
    Fd(const Fd&) = delete;
    Fd& operator=(const Fd&) = delete;
};

// The socket API takes a generic sockaddr*; this cast lets us pass our IPv4 sockaddr_in.
sockaddr* sa(sockaddr_in* a) { return reinterpret_cast<sockaddr*>(a); }

} // namespace lib9

### ✍️ Exercise 9.1 — Address, port and timeout helpers

* `make_loopback_addr(port)`: a zero-initialised `sockaddr_in` for 127.0.0.1:`port`.
* `local_port(fd)`: the local port via `getsockname`, or `-1` on error.
* `set_recv_timeout(fd, ms)`: set `SO_RCVTIMEO`; return `true` on success.

In [116]:
namespace ex9_1 {
sockaddr_in make_loopback_addr(uint16_t port) {
    sockaddr_in a{};                               // zero all fields (incl. padding)
    a.sin_family = AF_INET;
    a.sin_port = htons(port);                      // network byte order!
    a.sin_addr.s_addr = htonl(INADDR_LOOPBACK);    // 127.0.0.1, network byte order
    return a;
}
int local_port(int fd) {
    sockaddr_in a{};
    socklen_t len = sizeof a;
    if (getsockname(fd, lib9::sa(&a), &len) != 0) return -1;
    return ntohs(a.sin_port);
}
bool set_recv_timeout(int fd, int ms) {
    timeval tv{};
    tv.tv_sec = ms / 1000;
    tv.tv_usec = (ms % 1000) * 1000;
    return setsockopt(fd, SOL_SOCKET, SO_RCVTIMEO, &tv, sizeof tv) == 0;
}
}

**Test for Exercise 9.1** — run the next cell.

In [117]:
{
    auto a = ex9_1::make_loopback_addr(8080);
    CHECK_EQ(int(a.sin_family), AF_INET);
    CHECK_EQ(a.sin_port, htons(8080));
    CHECK_EQ(ntohl(a.sin_addr.s_addr), 0x7F000001u);

    lib9::Fd s(socket(AF_INET, SOCK_DGRAM, 0));
    auto any = ex9_1::make_loopback_addr(0);
    CHECK_EQ(bind(s.fd, lib9::sa(&any), sizeof any), 0);
    int port = ex9_1::local_port(s.fd);
    std::cout << "the kernel assigned port " << port << "\n";
    CHECK(port > 0);

    CHECK(ex9_1::set_recv_timeout(s.fd, 200));
    timeval tv{};
    socklen_t tl = sizeof tv;
    getsockopt(s.fd, SOL_SOCKET, SO_RCVTIMEO, &tv, &tl);        // read the option back
    long us = long(tv.tv_sec) * 1000000 + long(tv.tv_usec);
    bool timeout_set = (us >= 150000 && us <= 250000);
    CHECK(timeout_set);
    if (timeout_set) {                     // only then is recv() guaranteed not to block forever
        char b[16];
        auto t0 = std::chrono::steady_clock::now();
        ssize_t n = recv(s.fd, b, sizeof b, 0);                 // nobody sends: must time out
        int err = errno;
        double waited = std::chrono::duration<double>(std::chrono::steady_clock::now() - t0).count();
        CHECK(n < 0 && (err == EAGAIN || err == EWOULDBLOCK));
        std::cout << "recv() gave up after " << std::setprecision(2) << waited << std::setprecision(6) << " s: "
                  << std::strerror(err) << "\n";
    }
}

✅ int(a.sin_family) == AF_INET
✅ a.sin_port == htons(8080)
✅ ntohl(a.sin_addr.s_addr) == 0x7F000001u
✅ bind(s.fd, lib9::sa(&any), sizeof any) == 0
the kernel assigned port 56863
✅ port > 0
✅ ex9_1::set_recv_timeout(s.fd, 200)
✅ timeout_set
✅ n < 0 && (err == EAGAIN || err == EWOULDBLOCK)
recv() gave up after 0.21 s: Resource temporarily unavailable


<details>
<summary><b>Solution</b> (click to expand)</summary>

`sockaddr_in a{}` zeroes the structure, including the `sin_zero` padding, which some systems check. Forgetting
`htons` on the port is the classic socket bug: `a.sin_port = 8080` silently binds port 36895 (0x1F90 byte-swapped
is 0x901F) on a little-endian machine. `getsockname` is the only way to learn which port the kernel chose for port 0.

```cpp
namespace ex9_1 {
sockaddr_in make_loopback_addr(uint16_t port) {
    sockaddr_in a{};                               // zero all fields (incl. padding)
    a.sin_family = AF_INET;
    a.sin_port = htons(port);                      // network byte order!
    a.sin_addr.s_addr = htonl(INADDR_LOOPBACK);    // 127.0.0.1, network byte order
    return a;
}
int local_port(int fd) {
    sockaddr_in a{};
    socklen_t len = sizeof a;
    if (getsockname(fd, lib9::sa(&a), &len) != 0) return -1;
    return ntohs(a.sin_port);
}
bool set_recv_timeout(int fd, int ms) {
    timeval tv{};
    tv.tv_sec = ms / 1000;
    tv.tv_usec = (ms % 1000) * 1000;
    return setsockopt(fd, SOL_SOCKET, SO_RCVTIMEO, &tv, sizeof tv) == 0;
}
}
```

</details>

**Q9.1** *(Why?)* — Why do all our servers bind to port 0 instead of a fixed port such as 8080?

<details>
<summary><b>Solution</b> (click to expand)</summary>

A fixed port may already be in use (by another program, another notebook, or a previous run still in `TIME_WAIT`),
and `bind` would then fail with `EADDRINUSE`. Port 0 asks the kernel for **any free port**, which always
works; `getsockname` then tells us which one we got so the client can connect to it. Real servers use fixed,
well-known ports so that clients can find them; tests use port 0 to avoid conflicts.

</details>

## 9.2 UDP echo in one process

```
server                                client
socket(SOCK_DGRAM)                    socket(SOCK_DGRAM)
bind(127.0.0.1:0)
                                      sendto(data, server address)
                                        └─ the kernel auto-binds the client to an EPHEMERAL port
recvfrom(buf, &sender) -> data + the sender's address
sendto(data, sender)                  recv(buf) -> the echo
close()                               close()
```

`sendto(fd, data, len, 0, addr, addrlen)` sends one datagram; `recvfrom(fd, buf, cap, 0, addr, &addrlen)`
receives one and fills in who sent it. Both return the number of bytes, or `-1` on error.

In [118]:
namespace lib9b {

// PROVIDED (you wrote this in §9.1)
sockaddr_in make_loopback_addr(uint16_t port) {
    sockaddr_in a{};                               // zero all fields (incl. padding)
    a.sin_family = AF_INET;
    a.sin_port = htons(port);                      // network byte order!
    a.sin_addr.s_addr = htonl(INADDR_LOOPBACK);    // 127.0.0.1, network byte order
    return a;
}
int local_port(int fd) {
    sockaddr_in a{};
    socklen_t len = sizeof a;
    if (getsockname(fd, lib9::sa(&a), &len) != 0) return -1;
    return ntohs(a.sin_port);
}
bool set_recv_timeout(int fd, int ms) {
    timeval tv{};
    tv.tv_sec = ms / 1000;
    tv.tv_usec = (ms % 1000) * 1000;
    return setsockopt(fd, SOL_SOCKET, SO_RCVTIMEO, &tv, sizeof tv) == 0;
}

} // namespace lib9b

### ✍️ Exercise 9.2 — A UDP echo round trip

Implement `udp_echo_once(msg, port_seen_by_server, client_local_port)` following the diagram, in one thread,
with two `lib9::Fd` sockets. Set a 2000 ms receive timeout on both. Record the client's port as the server saw
it (from `recvfrom`) and as the client sees it (`local_port` after the first `sendto`). Return the echoed
string, or `""` on any failure.

In [119]:
namespace ex9_2 {
std::string udp_echo_once(const std::string& msg, int& port_seen_by_server, int& client_local_port) {
    port_seen_by_server = -1;
    client_local_port = -1;
    lib9::Fd server(socket(AF_INET, SOCK_DGRAM, 0));
    lib9::Fd client(socket(AF_INET, SOCK_DGRAM, 0));
    if (server.fd < 0 || client.fd < 0) return "";

    sockaddr_in addr = lib9b::make_loopback_addr(0);
    if (bind(server.fd, lib9::sa(&addr), sizeof addr) != 0) return "";
    addr = lib9b::make_loopback_addr(uint16_t(lib9b::local_port(server.fd)));   // real server address
    lib9b::set_recv_timeout(server.fd, 2000);
    lib9b::set_recv_timeout(client.fd, 2000);

    if (sendto(client.fd, msg.data(), msg.size(), 0, lib9::sa(&addr), sizeof addr) < 0) return "";
    client_local_port = lib9b::local_port(client.fd);         // assigned by that first sendto

    char buf[1500];
    sockaddr_in peer{};
    socklen_t plen = sizeof peer;
    ssize_t n = recvfrom(server.fd, buf, sizeof buf, 0, lib9::sa(&peer), &plen);
    if (n < 0) return "";
    port_seen_by_server = ntohs(peer.sin_port);
    sendto(server.fd, buf, size_t(n), 0, lib9::sa(&peer), plen);   // echo to whoever sent it

    n = recv(client.fd, buf, sizeof buf, 0);
    if (n < 0) return "";
    return std::string(buf, size_t(n));
}                                                           // both fds closed here
}

**Test for Exercise 9.2** — run the next cell.

In [120]:
{
    int seen = -1, local = -1;
    std::string r = ex9_2::udp_echo_once("ping over UDP", seen, local);
    std::cout << "echo: \"" << r << "\"  client port (server's view) " << seen
              << ", (client's view) " << local << "\n";
    CHECK_EQ(r, "ping over UDP");
    CHECK(seen == local && local >= 1024);        // an ephemeral port, seen identically by both
}

echo: "ping over UDP"  client port (server's view) 47559, (client's view) 47559
✅ r == "ping over UDP"
✅ seen == local && local >= 1024


<details>
<summary><b>Solution</b> (click to expand)</summary>

A single thread suffices because `sendto` never waits for the receiver: the datagram is queued in the server socket's
receive buffer until `recvfrom` collects it. The client never called `bind`, yet it has a port: the kernel assigned
an **ephemeral** one at the first `sendto`, and the server learned it from the datagram's source port, the same
5-tuple mechanism as §8.1.

```cpp
namespace ex9_2 {
std::string udp_echo_once(const std::string& msg, int& port_seen_by_server, int& client_local_port) {
    port_seen_by_server = -1;
    client_local_port = -1;
    lib9::Fd server(socket(AF_INET, SOCK_DGRAM, 0));
    lib9::Fd client(socket(AF_INET, SOCK_DGRAM, 0));
    if (server.fd < 0 || client.fd < 0) return "";

    sockaddr_in addr = lib9b::make_loopback_addr(0);
    if (bind(server.fd, lib9::sa(&addr), sizeof addr) != 0) return "";
    addr = lib9b::make_loopback_addr(uint16_t(lib9b::local_port(server.fd)));   // real server address
    lib9b::set_recv_timeout(server.fd, 2000);
    lib9b::set_recv_timeout(client.fd, 2000);

    if (sendto(client.fd, msg.data(), msg.size(), 0, lib9::sa(&addr), sizeof addr) < 0) return "";
    client_local_port = lib9b::local_port(client.fd);         // assigned by that first sendto

    char buf[1500];
    sockaddr_in peer{};
    socklen_t plen = sizeof peer;
    ssize_t n = recvfrom(server.fd, buf, sizeof buf, 0, lib9::sa(&peer), &plen);
    if (n < 0) return "";
    port_seen_by_server = ntohs(peer.sin_port);
    sendto(server.fd, buf, size_t(n), 0, lib9::sa(&peer), plen);   // echo to whoever sent it

    n = recv(client.fd, buf, sizeof buf, 0);
    if (n < 0) return "";
    return std::string(buf, size_t(n));
}                                                           // both fds closed here
}
```

</details>

**Q9.2** *(Why?)* — What happens if the client's datagram is lost (impossible on loopback, common on Wi-Fi) in the code above?
What would a real UDP application need to add?

<details>
<summary><b>Solution</b> (click to expand)</summary>

`recvfrom` on the server times out after 2 s and the function returns `""`. Without the timeout it would block
**forever**, because UDP never tells you that a datagram was lost. A real application needs its own timeout and
retry logic (DNS resolvers resend after about 1 s), identifiers to match replies with requests (the DNS ID field),
and must tolerate duplicates. *Misconception:* "`sendto` succeeded, so it was delivered". Success only means
the datagram was handed to the local kernel.

</details>

## 9.3 A TCP client and server with `std::thread`

```
server                                        client
socket(SOCK_STREAM)
bind(127.0.0.1:0)
listen(fd, backlog)    <- LISTEN: the kernel now completes handshakes on its own
accept(fd, &peer)  ·························  connect(server address)    <- 3-way handshake
  └─ returns a NEW fd for this one connection
recv / send        <=======================>  send / recv
close()                                       close()                    <- FIN exchange
```

The **listening** socket only produces new connections; each `accept` returns a **connected**
socket for one client. Both sides block, so they must run **concurrently**:
`std::thread t([&] { ... });` runs the lambda in a new thread (`[&]` lets it use the enclosing
function's local variables by reference), and `t.join()` waits for it to finish. A `std::thread`
destroyed without `join()` **terminates the whole program**, so always join before returning.

Send with `send(fd, data, len, MSG_NOSIGNAL)`: without `MSG_NOSIGNAL`, writing to a connection the
peer has closed raises the `SIGPIPE` signal, which kills the process (the setup cell also ignores it).
Setting `SO_RCVTIMEO` on the listening socket makes `accept` give up too.

### ✍️ Exercise 9.3 — TCP echo over loopback

Implement `tcp_echo(msg, port_seen_by_server, client_local_port)`:
1. Create the listening socket, bind to port 0, `listen(fd, 1)`, set a 2000 ms timeout, and compute the server
   address. Do this **before** starting the thread, so that the client can never connect too early.
2. Server thread: `accept` (record the peer's port), set a timeout on the new fd, `recv` once, `send` it back.
3. Main thread (client): `connect`, record `local_port`, `send` the message, then `recv` in a loop until
   `msg.size()` bytes have arrived or `recv` returns ≤ 0.
4. `join` the thread, then return what the client received.

In [121]:
namespace ex9_3 {
std::string tcp_echo(const std::string& msg, int& port_seen_by_server, int& client_local_port) {
    port_seen_by_server = -1;
    client_local_port = -1;
    lib9::Fd listener(socket(AF_INET, SOCK_STREAM, 0));
    if (listener.fd < 0) return "";
    sockaddr_in addr = lib9b::make_loopback_addr(0);
    if (bind(listener.fd, lib9::sa(&addr), sizeof addr) != 0) return "";
    if (listen(listener.fd, 1) != 0) return "";
    lib9b::set_recv_timeout(listener.fd, 2000);                   // accept() gives up after 2 s
    addr = lib9b::make_loopback_addr(uint16_t(lib9b::local_port(listener.fd)));

    std::thread server([&] {
        sockaddr_in peer{};
        socklen_t plen = sizeof peer;
        lib9::Fd conn(accept(listener.fd, lib9::sa(&peer), &plen));   // NEW socket per connection
        if (conn.fd < 0) return;
        port_seen_by_server = ntohs(peer.sin_port);
        lib9b::set_recv_timeout(conn.fd, 2000);
        char buf[256];
        ssize_t n = recv(conn.fd, buf, sizeof buf, 0);
        if (n > 0) send(conn.fd, buf, size_t(n), MSG_NOSIGNAL);      // echo
    });                                                               // conn closed here

    std::string reply;
    {
        lib9::Fd client(socket(AF_INET, SOCK_STREAM, 0));
        lib9b::set_recv_timeout(client.fd, 2000);
        if (client.fd >= 0 && connect(client.fd, lib9::sa(&addr), sizeof addr) == 0) {
            client_local_port = lib9b::local_port(client.fd);
            send(client.fd, msg.data(), msg.size(), MSG_NOSIGNAL);
            char buf[256];
            while (reply.size() < msg.size()) {       // the echo may arrive in pieces
                ssize_t n = recv(client.fd, buf, sizeof buf, 0);
                if (n <= 0) break;                    // closed, error, or timeout
                reply.append(buf, size_t(n));
            }
        }
    }                                                 // client closed here
    server.join();                                    // joined on every path
    return reply;
}
}

**Test for Exercise 9.3** — run the next cell.

In [122]:
{
    int seen = -1, local = -1;
    std::string r = ex9_3::tcp_echo("hello over TCP", seen, local);
    std::cout << "echo: \"" << r << "\"  client port (server's view) " << seen
              << ", (client's view) " << local << "\n";
    CHECK_EQ(r, "hello over TCP");
    CHECK(seen == local && local >= 1024);

    int seen2 = -1, local2 = -1;
    ex9_3::tcp_echo("again", seen2, local2);
    std::cout << "second run used client port " << local2 << "\n";
    CHECK(local2 >= 1024);                        // another ephemeral port (usually a different one)
}

echo: "hello over TCP"  client port (server's view) 50278, (client's view) 50278
✅ r == "hello over TCP"
✅ seen == local && local >= 1024
second run used client port 51784
✅ local2 >= 1024


<details>
<summary><b>Solution</b> (click to expand)</summary>

Creating the listening socket *before* starting the thread removes the race: once `listen` has returned, the kernel
completes the handshake of an incoming `connect` and queues the connection even if `accept` has not been called yet.
The timeouts guarantee termination: if `connect` fails, `accept` gives up after 2 s, the thread ends, and `join`
returns. Each run usually gets a different ephemeral port. The kernel may reuse a port number only for a *different*
5-tuple (here, a different server port), because the old connection is still in `TIME_WAIT` on the side that
closed first (§8.7).

```cpp
namespace ex9_3 {
std::string tcp_echo(const std::string& msg, int& port_seen_by_server, int& client_local_port) {
    port_seen_by_server = -1;
    client_local_port = -1;
    lib9::Fd listener(socket(AF_INET, SOCK_STREAM, 0));
    if (listener.fd < 0) return "";
    sockaddr_in addr = lib9b::make_loopback_addr(0);
    if (bind(listener.fd, lib9::sa(&addr), sizeof addr) != 0) return "";
    if (listen(listener.fd, 1) != 0) return "";
    lib9b::set_recv_timeout(listener.fd, 2000);                   // accept() gives up after 2 s
    addr = lib9b::make_loopback_addr(uint16_t(lib9b::local_port(listener.fd)));

    std::thread server([&] {
        sockaddr_in peer{};
        socklen_t plen = sizeof peer;
        lib9::Fd conn(accept(listener.fd, lib9::sa(&peer), &plen));   // NEW socket per connection
        if (conn.fd < 0) return;
        port_seen_by_server = ntohs(peer.sin_port);
        lib9b::set_recv_timeout(conn.fd, 2000);
        char buf[256];
        ssize_t n = recv(conn.fd, buf, sizeof buf, 0);
        if (n > 0) send(conn.fd, buf, size_t(n), MSG_NOSIGNAL);      // echo
    });                                                               // conn closed here

    std::string reply;
    {
        lib9::Fd client(socket(AF_INET, SOCK_STREAM, 0));
        lib9b::set_recv_timeout(client.fd, 2000);
        if (client.fd >= 0 && connect(client.fd, lib9::sa(&addr), sizeof addr) == 0) {
            client_local_port = lib9b::local_port(client.fd);
            send(client.fd, msg.data(), msg.size(), MSG_NOSIGNAL);
            char buf[256];
            while (reply.size() < msg.size()) {       // the echo may arrive in pieces
                ssize_t n = recv(client.fd, buf, sizeof buf, 0);
                if (n <= 0) break;                    // closed, error, or timeout
                reply.append(buf, size_t(n));
            }
        }
    }                                                 // client closed here
    server.join();                                    // joined on every path
    return reply;
}
}
```

</details>

**Q9.3** *(Concept)* — In `tcp_echo`, how many file descriptors does the server side use, and which 5-tuple does the accepted one
represent? Which socket would a second client's SYN reach?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Two: the **listening** fd (local 127.0.0.1:P, no remote) and the **connected** fd returned by `accept`, representing
(TCP, 127.0.0.1, client port, 127.0.0.1, P). A second client's SYN matches no existing connection's 5-tuple, so it
reaches the **listening** socket and would create a third fd at the next `accept`: exactly the demultiplexing
rules you implemented in §8.1.

</details>

## 9.4 TCP does not preserve message boundaries

TCP delivers a **byte stream** (§8.4). The kernel may merge several `send` calls into one segment,
split one `send` across segments, and `recv` returns **whatever bytes are available**, up to the
buffer size. Message boundaries are simply not part of the service.

**Q9.4** *(Predict the output)* — A client calls `send("hello")`, then `send("world")`. The server sleeps 200 ms, then calls
`recv(fd, buf, 100, 0)` once. What does that `recv` most likely return?

<details>
<summary><b>Solution</b> (click to expand)</summary>

Most likely **all 10 bytes, `helloworld`**, in one `recv`: both writes are already sitting in the server's receive
buffer, and `recv` hands over everything available. It *could* return 5 bytes, or even 3. TCP promises only the
**order** of bytes, never how they are grouped. *Misconception:* "one send = one recv". Code that assumes this
works in testing and breaks under load. Run the demo below.

</details>

In [123]:
namespace demo9_4 {
// GIVEN demo: two send() calls on the client, then the server reads with a 100-byte buffer.
std::vector<std::string> two_sends() {
    std::vector<std::string> pieces;               // what each server recv() returned
    lib9::Fd listener(socket(AF_INET, SOCK_STREAM, 0));
    sockaddr_in addr = lib9b::make_loopback_addr(0);
    if (listener.fd < 0 || bind(listener.fd, lib9::sa(&addr), sizeof addr) != 0 || listen(listener.fd, 1) != 0)
        return pieces;
    lib9b::set_recv_timeout(listener.fd, 2000);
    addr = lib9b::make_loopback_addr(uint16_t(lib9b::local_port(listener.fd)));

    std::thread server([&] {
        lib9::Fd conn(accept(listener.fd, nullptr, nullptr));
        if (conn.fd < 0) return;
        lib9b::set_recv_timeout(conn.fd, 2000);
        std::this_thread::sleep_for(std::chrono::milliseconds(200));   // let both sends arrive
        char buf[100];
        while (true) {
            ssize_t n = recv(conn.fd, buf, sizeof buf, 0);
            if (n <= 0) break;                                         // 0 = client closed
            pieces.emplace_back(buf, size_t(n));
        }
    });
    {
        lib9::Fd client(socket(AF_INET, SOCK_STREAM, 0));
        if (client.fd >= 0 && connect(client.fd, lib9::sa(&addr), sizeof addr) == 0) {
            send(client.fd, "hello", 5, MSG_NOSIGNAL);
            send(client.fd, "world", 5, MSG_NOSIGNAL);
        }
    }                                                                  // close -> server's recv returns 0
    server.join();
    return pieces;
}
}

In [124]:
{
    auto pieces = demo9_4::two_sends();
    std::cout << "the server's recv() calls returned " << pieces.size() << " piece(s):";
    std::string all;
    for (const auto& p : pieces) { std::cout << " \"" << p << "\""; all += p; }
    std::cout << "\n";
    CHECK_EQ(all, "helloworld");     // the bytes and their order are guaranteed; the grouping is not
}

the server's recv() calls returned 1 piece(s): "helloworld"
✅ all == "helloworld"


To carry **messages** over TCP, the application must mark boundaries itself. The standard
technique is **length-prefixed framing**: send each message as a 4-byte big-endian length
followed by the bytes. The receiver accumulates incoming bytes and extracts a message only
once a length *and* that many bytes are available.

```
stream:  [00 00 00 05][h e l l o][00 00 00 00][00 00 00 03][a b c] ...
           len = 5      message    len = 0       len = 3     message
recv() may return any slice of this, e.g. "00 00" | "00 05 h e l" | "l o 00 00 00" | ...
```

In [125]:
namespace lib9c {

// PROVIDED (you wrote this in §1.6)
uint16_t read_be16(const uint8_t* p) {
    return uint16_t((uint16_t(p[0]) << 8) | p[1]);
}
uint32_t read_be32(const uint8_t* p) {
    return (uint32_t(p[0]) << 24) | (uint32_t(p[1]) << 16) |
           (uint32_t(p[2]) << 8)  |  uint32_t(p[3]);
}
void write_be16(uint8_t* p, uint16_t v) {
    p[0] = uint8_t(v >> 8);      // most significant byte first
    p[1] = uint8_t(v & 0xFF);
}
void write_be32(uint8_t* p, uint32_t v) {
    p[0] = uint8_t(v >> 24);
    p[1] = uint8_t(v >> 16);     // the uint8_t cast keeps only the low 8 bits
    p[2] = uint8_t(v >> 8);
    p[3] = uint8_t(v);
}

} // namespace lib9c

### ✍️ Exercise 9.4a — Length-prefixed framing: encoder and decoder

* `frame_message(msg)`: 4-byte big-endian length, then the bytes.
* `FrameDecoder::feed(p, n)`: append bytes to `buf`.
* `FrameDecoder::next()`: if `buf` holds a complete frame, remove it from `buf` and return the message;
  otherwise return `std::nullopt` and leave `buf` untouched.

In [126]:
namespace ex9_4a {
std::vector<uint8_t> frame_message(const std::string& msg) {
    std::vector<uint8_t> out(4);
    lib9c::write_be32(out.data(), uint32_t(msg.size()));
    out.insert(out.end(), msg.begin(), msg.end());
    return out;
}
struct FrameDecoder {
    std::vector<uint8_t> buf;   // received but not yet consumed
    void feed(const uint8_t* p, size_t n) { buf.insert(buf.end(), p, p + n); }
    std::optional<std::string> next() {
        if (buf.size() < 4) return std::nullopt;                  // length not complete yet
        uint32_t len = lib9c::read_be32(buf.data());
        if (buf.size() - 4 < len) return std::nullopt;            // body not complete yet
        std::string msg(buf.begin() + 4, buf.begin() + 4 + len);
        buf.erase(buf.begin(), buf.begin() + 4 + len);            // consume the frame
        return msg;
    }
};
}

**Test for Exercise 9.4a** — run the next cell.

In [127]:
{
    CHECK_EQ(ex9_4a::frame_message("hi"), (std::vector<uint8_t>{0x00, 0x00, 0x00, 0x02, 'h', 'i'}));
    std::vector<std::string> msgs = {"hello", "", "a somewhat longer message", std::string(300, 'z')};
    std::vector<uint8_t> stream;
    for (const auto& m : msgs) {
        auto f = ex9_4a::frame_message(m);
        stream.insert(stream.end(), f.begin(), f.end());
    }
    for (size_t chunk : {size_t(1), size_t(3), size_t(7), size_t(1000)}) {   // every way of slicing
        ex9_4a::FrameDecoder dec;
        std::vector<std::string> got;
        for (size_t off = 0; off < stream.size(); off += chunk) {
            dec.feed(stream.data() + off, std::min(chunk, stream.size() - off));
            for (int guard = 0; guard < 10; ++guard) {           // drain complete frames
                auto m = dec.next();
                if (!m) break;
                got.push_back(*m);
            }
        }
        std::cout << "chunk size " << chunk << ": decoded " << got.size() << " messages\n";
        CHECK(got == msgs);
    }
}

✅ ex9_4a::frame_message("hi") == (std::vector<uint8_t>{0x00, 0x00, 0x00, 0x02, 'h', 'i'})
chunk size 1: decoded 4 messages
✅ got == msgs
chunk size 3: decoded 4 messages
✅ got == msgs
chunk size 7: decoded 4 messages
✅ got == msgs
chunk size 1000: decoded 4 messages
✅ got == msgs


<details>
<summary><b>Solution</b> (click to expand)</summary>

The decoder never assumes a `recv` boundary means anything: it only trusts the length prefix, so every slicing, even
1 byte at a time, yields the same messages, empty ones included. Writing the check as `buf.size() - 4 < len`
avoids overflow in `4 + len`. A production decoder also rejects absurd lengths (e.g. > 16 MiB): otherwise a
malicious peer sends `ff ff ff ff` and makes you wait for, or allocate, 4 GiB.

```cpp
namespace ex9_4a {
std::vector<uint8_t> frame_message(const std::string& msg) {
    std::vector<uint8_t> out(4);
    lib9c::write_be32(out.data(), uint32_t(msg.size()));
    out.insert(out.end(), msg.begin(), msg.end());
    return out;
}
struct FrameDecoder {
    std::vector<uint8_t> buf;   // received but not yet consumed
    void feed(const uint8_t* p, size_t n) { buf.insert(buf.end(), p, p + n); }
    std::optional<std::string> next() {
        if (buf.size() < 4) return std::nullopt;                  // length not complete yet
        uint32_t len = lib9c::read_be32(buf.data());
        if (buf.size() - 4 < len) return std::nullopt;            // body not complete yet
        std::string msg(buf.begin() + 4, buf.begin() + 4 + len);
        buf.erase(buf.begin(), buf.begin() + 4 + len);            // consume the frame
        return msg;
    }
};
}
```

</details>

In [128]:
namespace lib9d {

// PROVIDED (you wrote this in §9.4a)
std::vector<uint8_t> frame_message(const std::string& msg) {
    std::vector<uint8_t> out(4);
    lib9c::write_be32(out.data(), uint32_t(msg.size()));
    out.insert(out.end(), msg.begin(), msg.end());
    return out;
}
struct FrameDecoder {
    std::vector<uint8_t> buf;   // received but not yet consumed
    void feed(const uint8_t* p, size_t n) { buf.insert(buf.end(), p, p + n); }
    std::optional<std::string> next() {
        if (buf.size() < 4) return std::nullopt;                  // length not complete yet
        uint32_t len = lib9c::read_be32(buf.data());
        if (buf.size() - 4 < len) return std::nullopt;            // body not complete yet
        std::string msg(buf.begin() + 4, buf.begin() + 4 + len);
        buf.erase(buf.begin(), buf.begin() + 4 + len);            // consume the frame
        return msg;
    }
};

} // namespace lib9d

### ✍️ Exercise 9.4b — Framing over a real TCP connection

* `send_all(fd, p, n)`: `send` may accept **fewer** bytes than requested; loop until all are sent. Return `false`
  on error.
* `tcp_framed(msgs, recv_calls)`: a server thread accepts one connection and reads with a deliberately tiny
  buffer, **`recv(fd, buf, 3, 0)`**, feeding a `lib9d::FrameDecoder` and collecting messages until it has
  `msgs.size()` of them or `recv` returns ≤ 0. Count the `recv` calls in `recv_calls`. The client sends all
  frames with **one** `send_all`, then closes. Return the messages the server collected. Use the same
  listen-before-thread, timeout, and join pattern as §9.3.

In [129]:
namespace ex9_4b {
bool send_all(int fd, const uint8_t* p, size_t n) {
    while (n > 0) {
        ssize_t k = send(fd, p, n, MSG_NOSIGNAL);
        if (k <= 0) return false;            // error (e.g. peer closed)
        p += k;                              // advance past what was accepted
        n -= size_t(k);
    }
    return true;
}
std::vector<std::string> tcp_framed(const std::vector<std::string>& msgs, int& recv_calls) {
    recv_calls = 0;
    std::vector<std::string> got;
    lib9::Fd listener(socket(AF_INET, SOCK_STREAM, 0));
    sockaddr_in addr = lib9b::make_loopback_addr(0);
    if (listener.fd < 0 || bind(listener.fd, lib9::sa(&addr), sizeof addr) != 0 ||
        listen(listener.fd, 1) != 0)
        return got;
    lib9b::set_recv_timeout(listener.fd, 2000);
    addr = lib9b::make_loopback_addr(uint16_t(lib9b::local_port(listener.fd)));

    std::thread server([&] {
        lib9::Fd conn(accept(listener.fd, nullptr, nullptr));
        if (conn.fd < 0) return;
        lib9b::set_recv_timeout(conn.fd, 2000);
        lib9d::FrameDecoder dec;
        uint8_t buf[3];                                   // tiny on purpose
        while (got.size() < msgs.size()) {
            ssize_t n = recv(conn.fd, buf, sizeof buf, 0);
            if (n <= 0) break;
            ++recv_calls;
            dec.feed(buf, size_t(n));
            while (auto m = dec.next()) got.push_back(*m);
        }
    });

    {
        std::vector<uint8_t> all;                          // every frame, back to back
        for (const auto& m : msgs) {
            auto f = lib9d::frame_message(m);
            all.insert(all.end(), f.begin(), f.end());
        }
        lib9::Fd client(socket(AF_INET, SOCK_STREAM, 0));
        if (client.fd >= 0 && connect(client.fd, lib9::sa(&addr), sizeof addr) == 0)
            send_all(client.fd, all.data(), all.size());
    }
    server.join();
    return got;
}
}

**Test for Exercise 9.4b** — run the next cell.

In [130]:
{
    std::vector<std::string> msgs = {"first", "", "third message, a bit longer", std::string(5000, 'x')};
    int calls = 0;
    auto got = ex9_4b::tcp_framed(msgs, calls);
    std::cout << "server needed " << calls << " recv() calls for " << got.size() << " messages\n";
    CHECK(got == msgs);
    CHECK(calls > int(msgs.size()));                 // many reads per message: boundaries are ours
}

server needed 1683 recv() calls for 4 messages
✅ got == msgs
✅ calls > int(msgs.size())


<details>
<summary><b>Solution</b> (click to expand)</summary>

The sender wrote everything in one call, the receiver read it 3 bytes at a time, and the messages still came out
intact, because the framing, not TCP, defines where each message ends. `send_all` matters in practice: on a
non-blocking socket, or when a signal interrupts the call, `send` can return a short count, and ignoring it
silently truncates the stream. The `while (auto m = dec.next())` idiom declares `m` inside the condition and
loops while it holds a value.

```cpp
namespace ex9_4b {
bool send_all(int fd, const uint8_t* p, size_t n) {
    while (n > 0) {
        ssize_t k = send(fd, p, n, MSG_NOSIGNAL);
        if (k <= 0) return false;            // error (e.g. peer closed)
        p += k;                              // advance past what was accepted
        n -= size_t(k);
    }
    return true;
}
std::vector<std::string> tcp_framed(const std::vector<std::string>& msgs, int& recv_calls) {
    recv_calls = 0;
    std::vector<std::string> got;
    lib9::Fd listener(socket(AF_INET, SOCK_STREAM, 0));
    sockaddr_in addr = lib9b::make_loopback_addr(0);
    if (listener.fd < 0 || bind(listener.fd, lib9::sa(&addr), sizeof addr) != 0 ||
        listen(listener.fd, 1) != 0)
        return got;
    lib9b::set_recv_timeout(listener.fd, 2000);
    addr = lib9b::make_loopback_addr(uint16_t(lib9b::local_port(listener.fd)));

    std::thread server([&] {
        lib9::Fd conn(accept(listener.fd, nullptr, nullptr));
        if (conn.fd < 0) return;
        lib9b::set_recv_timeout(conn.fd, 2000);
        lib9d::FrameDecoder dec;
        uint8_t buf[3];                                   // tiny on purpose
        while (got.size() < msgs.size()) {
            ssize_t n = recv(conn.fd, buf, sizeof buf, 0);
            if (n <= 0) break;
            ++recv_calls;
            dec.feed(buf, size_t(n));
            while (auto m = dec.next()) got.push_back(*m);
        }
    });

    {
        std::vector<uint8_t> all;                          // every frame, back to back
        for (const auto& m : msgs) {
            auto f = lib9d::frame_message(m);
            all.insert(all.end(), f.begin(), f.end());
        }
        lib9::Fd client(socket(AF_INET, SOCK_STREAM, 0));
        if (client.fd >= 0 && connect(client.fd, lib9::sa(&addr), sizeof addr) == 0)
            send_all(client.fd, all.data(), all.size());
    }
    server.join();
    return got;
}
}
```

</details>

## 9.5 Connecting to a closed port: `ECONNREFUSED`

If a SYN arrives for a port where nothing is listening, the kernel answers with a **RST**
segment, and the client's `connect` fails immediately with `errno == ECONNREFUSED`.
To get a port that is certainly closed, bind a socket to port 0, note the port, and close the
socket without ever calling `listen`.

### ✍️ Exercise 9.5 — Observe ECONNREFUSED

`connect_to_closed_port(port_used)`: find a closed port as described, store it in `port_used`, then `connect` a
new TCP socket to it. Return `errno` if `connect` fails, `0` if it (unexpectedly) succeeds, `-1` if the setup fails.

In [131]:
namespace ex9_5 {
int connect_to_closed_port(int& port_used) {
    port_used = -1;
    {
        lib9::Fd probe(socket(AF_INET, SOCK_STREAM, 0));
        sockaddr_in a = lib9b::make_loopback_addr(0);
        if (probe.fd < 0 || bind(probe.fd, lib9::sa(&a), sizeof a) != 0) return -1;
        port_used = lib9b::local_port(probe.fd);
    }                                         // closed without listen(): the port is closed
    lib9::Fd s(socket(AF_INET, SOCK_STREAM, 0));
    if (s.fd < 0) return -1;
    sockaddr_in a = lib9b::make_loopback_addr(uint16_t(port_used));
    if (connect(s.fd, lib9::sa(&a), sizeof a) == 0) return 0;
    return errno;                             // read errno right after the failing call
}
}

**Test for Exercise 9.5** — run the next cell.

In [132]:
{
    int port = -1;
    int err = ex9_5::connect_to_closed_port(port);
    std::cout << "connect to 127.0.0.1:" << port << " -> errno " << err << " ("
              << (err > 0 ? std::strerror(err) : "no error") << ")\n";
    CHECK(port > 0);
    CHECK_EQ(err, ECONNREFUSED);
}

connect to 127.0.0.1:47471 -> errno 111 (Connection refused)
✅ port > 0
✅ err == ECONNREFUSED


<details>
<summary><b>Solution</b> (click to expand)</summary>

`errno` must be read immediately after the failing call, since any later library call may overwrite it. On loopback the
RST comes back within microseconds. This is the fastest possible "no" and tells you the **host is reachable but the
port is closed**.

```cpp
namespace ex9_5 {
int connect_to_closed_port(int& port_used) {
    port_used = -1;
    {
        lib9::Fd probe(socket(AF_INET, SOCK_STREAM, 0));
        sockaddr_in a = lib9b::make_loopback_addr(0);
        if (probe.fd < 0 || bind(probe.fd, lib9::sa(&a), sizeof a) != 0) return -1;
        port_used = lib9b::local_port(probe.fd);
    }                                         // closed without listen(): the port is closed
    lib9::Fd s(socket(AF_INET, SOCK_STREAM, 0));
    if (s.fd < 0) return -1;
    sockaddr_in a = lib9b::make_loopback_addr(uint16_t(port_used));
    if (connect(s.fd, lib9::sa(&a), sizeof a) == 0) return 0;
    return errno;                             // read errno right after the failing call
}
}
```

</details>

**Q9.5** *(Why?)* — `connect` to a remote server fails after about two minutes with `ETIMEDOUT` rather than instantly with
`ECONNREFUSED`. What is different about that server's side?

<details>
<summary><b>Solution</b> (click to expand)</summary>

`ECONNREFUSED` means a **RST** came back: the host is up but nothing listens on the port. `ETIMEDOUT` means **nothing**
came back: the SYNs were dropped silently (typically by a firewall configured to drop, or the host is down or
unreachable), so the client retransmitted the SYN with exponential backoff until it gave up. In debugging,
"refused" points at the service, while "timeout" points at the network path or a firewall.

</details>

### 🔁 Recap

- A socket is a kernel endpoint referenced by an fd; failing calls return −1 and set `errno`.
- `sockaddr_in` needs `htons`/`htonl`; bind to port 0 and use `getsockname` to learn the chosen port; clients get ephemeral ports automatically.
- UDP: `sendto`/`recvfrom`, one datagram per call, boundaries kept, loss is the application's problem.
- TCP: server `socket → bind → listen → accept → recv/send → close`; client `socket → connect → send/recv → close`; `accept` returns a new fd.
- TCP has no message boundaries: frame messages (length prefix), loop on `send`/`recv`; RST → `ECONNREFUSED`, silence → `ETIMEDOUT`.

**You should now be able to explain…**

- [ ] the exact call sequences of a TCP server and client, and which calls block;
- [ ] why a timeout on every receiving socket (and `MSG_NOSIGNAL` on sends) is essential;
- [ ] where the client's port number comes from;
- [ ] why one `send` is not one `recv`, and how length-prefixed framing fixes it;
- [ ] what `ECONNREFUSED` versus `ETIMEDOUT` reveals about the other side.

# Part 10 — Capstone: dissect a real frame

Below is a complete frame as a capture tool would show it (no preamble, no FCS): a laptop opening
a TCP connection to a web server. It is Ethernet + IPv4 + a TCP **SYN** with an MSS option, and
Ethernet padding up to the 60-byte minimum. Every length and checksum in it is valid.

In [133]:
namespace capstone {
const std::vector<uint8_t> syn_frame = {
    0xf0, 0x9f, 0xc2, 0x10, 0x20, 0x30, 0x3c, 0x22, 0xfb, 0x12, 0xab, 0x9e,
    0x08, 0x00, 0x45, 0x00, 0x00, 0x2c, 0x4d, 0x2a, 0x40, 0x00, 0x40, 0x06,
    0xef, 0x9e, 0xc0, 0xa8, 0x01, 0x0a, 0xcb, 0x00, 0x71, 0x50, 0xc9, 0x3a,
    0x00, 0x50, 0x9a, 0x3b, 0x27, 0xc1, 0x00, 0x00, 0x00, 0x00, 0x60, 0x02,
    0xfa, 0xf0, 0x13, 0xab, 0x00, 0x00, 0x02, 0x04, 0x05, 0xb4, 0x00, 0x00,
};
}

In [134]:
hexdump(capstone::syn_frame);

0000  f0 9f c2 10 20 30 3c 22  fb 12 ab 9e 08 00 45 00  |.... 0<"......E.|
0010  00 2c 4d 2a 40 00 40 06  ef 9e c0 a8 01 0a cb 00  |.,M*@.@.........|
0020  71 50 c9 3a 00 50 9a 3b  27 c1 00 00 00 00 60 02  |qP.:.P.;'.....`.|
0030  fa f0 13 ab 00 00 02 04  05 b4 00 00              |............|


```
offset  0 ─ 13   Ethernet header (dst, src, EtherType)
       14 ─ 33   IPv4 header (IHL = 5)
       34 ─ 57   TCP header (data offset = 6: 20 bytes + 4 bytes of options)
       58 ─ 59   Ethernet padding (IP's total length says the packet ends at 57)
```
Every earlier helper you need is provided in one library cell:

In [135]:
namespace lib10 {

// PROVIDED (you wrote this in §1.2)
std::string to_hex8(uint8_t b) {
    std::ostringstream os;
    // unsigned(b): print the number, not the character; setw(2)+setfill('0'): keep leading 0
    os << std::hex << std::setw(2) << std::setfill('0') << unsigned(b);
    return os.str();
}

// PROVIDED (you wrote this in §1.4)
uint8_t get_field(uint8_t b, int shift, int width) {
    uint8_t mask = uint8_t((1u << width) - 1);   // width ones, e.g. width 3 -> 0b111
    return uint8_t((b >> shift) & mask);
}
uint8_t pack_nibbles(uint8_t hi, uint8_t lo) {
    return uint8_t((hi << 4) | (lo & 0x0F));
}

// PROVIDED (you wrote this in §1.6)
uint16_t read_be16(const uint8_t* p) {
    return uint16_t((uint16_t(p[0]) << 8) | p[1]);
}
uint32_t read_be32(const uint8_t* p) {
    return (uint32_t(p[0]) << 24) | (uint32_t(p[1]) << 16) |
           (uint32_t(p[2]) << 8)  |  uint32_t(p[3]);
}
void write_be16(uint8_t* p, uint16_t v) {
    p[0] = uint8_t(v >> 8);      // most significant byte first
    p[1] = uint8_t(v & 0xFF);
}
void write_be32(uint8_t* p, uint32_t v) {
    p[0] = uint8_t(v >> 24);
    p[1] = uint8_t(v >> 16);     // the uint8_t cast keeps only the low 8 bits
    p[2] = uint8_t(v >> 8);
    p[3] = uint8_t(v);
}

// PROVIDED (you wrote this in §5.2)
std::string format_mac(const uint8_t* p) {
    std::string s;
    for (int i = 0; i < 6; ++i) {
        if (i) s += ':';          // separator between bytes, not after the last
        s += to_hex8(p[i]);
    }
    return s;
}

// PROVIDED (you wrote this in §5.4)
struct EthHeader {
    std::array<uint8_t, 6> dst{};
    std::array<uint8_t, 6> src{};
    uint16_t ethertype = 0;
};
std::optional<EthHeader> parse_eth(const uint8_t* p, size_t n) {
    if (n < 14) return std::nullopt;            // too short to hold the header
    EthHeader h;
    std::memcpy(h.dst.data(), p + 0, 6);
    std::memcpy(h.src.data(), p + 6, 6);
    h.ethertype = read_be16(p + 12);      // network byte order
    return h;
}

// PROVIDED (you wrote this in §6.1)
std::optional<uint32_t> parse_ipv4(const std::string& s) {
    uint32_t result = 0, octet = 0;
    int dots = 0, digits = 0;
    for (char c : s) {
        if (c >= '0' && c <= '9') {
            octet = octet * 10 + uint32_t(c - '0');
            if (++digits > 3 || octet > 255) return std::nullopt;
        } else if (c == '.') {
            if (digits == 0 || dots == 3) return std::nullopt;   // empty octet / too many dots
            result = (result << 8) | octet;                     // shift in the finished octet
            ++dots; octet = 0; digits = 0;
        } else {
            return std::nullopt;                                // any other character
        }
    }
    if (dots != 3 || digits == 0) return std::nullopt;
    return (result << 8) | octet;
}
std::string format_ipv4(uint32_t ip) {
    return std::to_string(ip >> 24) + "." + std::to_string((ip >> 16) & 0xFF) + "." +
           std::to_string((ip >> 8) & 0xFF) + "." + std::to_string(ip & 0xFF);
}

// PROVIDED (you wrote this in §6.6)
struct Ipv4Header {
    uint8_t version = 0, ihl = 0, ttl = 0, protocol = 0, flags = 0;
    uint16_t total_len = 0, ident = 0, frag_offset = 0, checksum = 0;
    uint32_t src = 0, dst = 0;
    size_t header_len() const { return size_t(ihl) * 4; }
};
std::optional<Ipv4Header> parse_ipv4_header(const uint8_t* p, size_t n) {
    if (n < 20) return std::nullopt;
    Ipv4Header h;
    h.version     = get_field(p[0], 4, 4);     // high nibble
    h.ihl         = get_field(p[0], 0, 4);     // low nibble, in 32-bit words
    h.total_len   = read_be16(p + 2);
    h.ident       = read_be16(p + 4);
    h.flags       = get_field(p[6], 5, 3);     // top 3 bits of byte 6
    h.frag_offset = read_be16(p + 6) & 0x1FFF;  // low 13 bits
    h.ttl         = p[8];
    h.protocol    = p[9];
    h.checksum    = read_be16(p + 10);
    h.src         = read_be32(p + 12);
    h.dst         = read_be32(p + 16);
    if (h.version != 4 || h.ihl < 5) return std::nullopt;
    if (h.header_len() > n || h.total_len < h.header_len() || h.total_len > n) return std::nullopt;
    return h;
}

// PROVIDED (you wrote this in §6.7)
uint16_t internet_checksum(const uint8_t* p, size_t n) {
    uint32_t sum = 0;
    for (size_t i = 0; i + 1 < n; i += 2)
        sum += (uint32_t(p[i]) << 8) | p[i + 1];      // big-endian 16-bit words
    if (n % 2) sum += uint32_t(p[n - 1]) << 8;       // odd length: pad with a zero byte
    while (sum >> 16) sum = (sum & 0xFFFF) + (sum >> 16);   // end-around carry
    return uint16_t(~sum);
}
bool ipv4_header_ok(const uint8_t* p, size_t n) {
    if (n < 20) return false;
    size_t hl = size_t(p[0] & 0x0F) * 4;
    if (hl < 20 || hl > n) return false;
    return internet_checksum(p, hl) == 0;             // includes the stored checksum
}

// PROVIDED (you wrote this in §8.3)
uint16_t transport_checksum(uint32_t src_ip, uint32_t dst_ip, uint8_t proto,
                            const uint8_t* seg, size_t len) {
    std::vector<uint8_t> buf(12);
    write_be32(&buf[0], src_ip);
    write_be32(&buf[4], dst_ip);
    buf[8] = 0;
    buf[9] = proto;
    write_be16(&buf[10], uint16_t(len));
    buf.insert(buf.end(), seg, seg + len);          // then the whole segment
    return internet_checksum(buf.data(), buf.size());
}

// PROVIDED (you wrote this in §8.5)
struct TcpHeader {
    uint16_t src_port = 0, dst_port = 0, window = 0, checksum = 0, urgent = 0;
    uint32_t seq = 0, ack = 0;
    uint8_t data_offset = 0, flags = 0;
    size_t header_len() const { return size_t(data_offset) * 4; }
};
std::optional<TcpHeader> parse_tcp(const uint8_t* p, size_t n) {
    if (n < 20) return std::nullopt;
    TcpHeader h;
    h.src_port    = read_be16(p);
    h.dst_port    = read_be16(p + 2);
    h.seq         = read_be32(p + 4);
    h.ack         = read_be32(p + 8);
    h.data_offset = get_field(p[12], 4, 4);   // high nibble of byte 12
    h.flags       = p[13];
    h.window      = read_be16(p + 14);
    h.checksum    = read_be16(p + 16);
    h.urgent      = read_be16(p + 18);
    if (h.data_offset < 5 || h.header_len() > n) return std::nullopt;
    return h;
}
std::string flags_to_string(uint8_t f) {
    static const char* names[8] = {"FIN", "SYN", "RST", "PSH", "ACK", "URG", "ECE", "CWR"};
    std::string s;
    for (int i = 0; i < 8; ++i) {
        if (!(f & (1u << i))) continue;
        if (!s.empty()) s += "|";
        s += names[i];
    }
    return s.empty() ? "none" : s;
}

} // namespace lib10

## 10.1 Walking TCP options

TCP options (bytes 20 … header_len − 1 of the TCP header) form a list of **TLV** (type-length-value) items:

```
kind 0            end of option list (1 byte, stop)
kind 1            NOP, no operation (1 byte, used as padding for alignment)
any other kind    [kind][len][len − 2 bytes of data]      len counts kind and len themselves
kind 2 = MSS      [02][04][16-bit MSS, big-endian]
```

### ✍️ Exercise 10.1 — Find the MSS option

`find_mss(opts, n)`: walk the options; return the MSS value, or `-1` if absent or malformed (a length < 2 or one
that runs past `n`).

In [136]:
namespace ex10_1 {
int find_mss(const uint8_t* opts, size_t n) {
    size_t i = 0;
    while (i < n) {
        uint8_t kind = opts[i];
        if (kind == 0) break;                 // end of list
        if (kind == 1) { ++i; continue; }     // NOP
        if (i + 1 >= n) return -1;            // no room for the length byte
        uint8_t len = opts[i + 1];
        if (len < 2 || i + len > n) return -1;   // malformed: would loop forever or overrun
        if (kind == 2 && len == 4) return lib10::read_be16(opts + i + 2);
        i += len;                             // skip any other option
    }
    return -1;
}
}

**Test for Exercise 10.1** — run the next cell.

In [137]:
{
    const uint8_t a[] = {0x02, 0x04, 0x05, 0xb4};
    const uint8_t b[] = {0x01, 0x01, 0x04, 0x02, 0x02, 0x04, 0x02, 0x18};   // NOP NOP SACK-OK MSS 536
    const uint8_t c[] = {0x01, 0x03, 0x03, 0x07, 0x00, 0x00, 0x00, 0x00};   // NOP, window scale 7, end
    const uint8_t bad[] = {0x05, 0x00, 0x02, 0x04};                          // length 0: malformed
    CHECK_EQ(ex10_1::find_mss(a, sizeof a), 1460);
    CHECK_EQ(ex10_1::find_mss(b, sizeof b), 536);
    CHECK_EQ(ex10_1::find_mss(c, sizeof c), -1);
    CHECK_EQ(ex10_1::find_mss(bad, sizeof bad), -1);
}

✅ ex10_1::find_mss(a, sizeof a) == 1460
✅ ex10_1::find_mss(b, sizeof b) == 536
✅ ex10_1::find_mss(c, sizeof c) == -1
✅ ex10_1::find_mss(bad, sizeof bad) == -1


<details>
<summary><b>Solution</b> (click to expand)</summary>

The `len < 2` check is essential: an option with length 0 would make `i += len` loop forever. That is a real
denial-of-service bug class in packet parsers. Every TLV walker needs both checks, "the length makes progress" and
"the length stays inside the buffer".

```cpp
namespace ex10_1 {
int find_mss(const uint8_t* opts, size_t n) {
    size_t i = 0;
    while (i < n) {
        uint8_t kind = opts[i];
        if (kind == 0) break;                 // end of list
        if (kind == 1) { ++i; continue; }     // NOP
        if (i + 1 >= n) return -1;            // no room for the length byte
        uint8_t len = opts[i + 1];
        if (len < 2 || i + len > n) return -1;   // malformed: would loop forever or overrun
        if (kind == 2 && len == 4) return lib10::read_be16(opts + i + 2);
        i += len;                             // skip any other option
    }
    return -1;
}
}
```

</details>

In [138]:
namespace lib10b {

// PROVIDED (you wrote this in §10.1)
int find_mss(const uint8_t* opts, size_t n) {
    size_t i = 0;
    while (i < n) {
        uint8_t kind = opts[i];
        if (kind == 0) break;                 // end of list
        if (kind == 1) { ++i; continue; }     // NOP
        if (i + 1 >= n) return -1;            // no room for the length byte
        uint8_t len = opts[i + 1];
        if (len < 2 || i + len > n) return -1;   // malformed: would loop forever or overrun
        if (kind == 2 && len == 4) return lib10::read_be16(opts + i + 2);
        i += len;                             // skip any other option
    }
    return -1;
}

} // namespace lib10b

### ✍️ Exercise 10.2 — A three-layer dissector

Fill in `dissect(frame)`, layer by layer, and **print** a summary like the one below. Return `std::nullopt` if any
layer fails to parse, or if the EtherType is not IPv4 or the IP protocol is not TCP.

* Ethernet: `lib10::parse_eth`; the IP packet starts at offset 14.
* IPv4: `lib10::parse_ipv4_header`; `ip_checksum_ok` = the header checksums to 0 (`lib10::internet_checksum`).
  The TCP segment spans `total_len − header_len` bytes after the IP header, and everything after the IP packet is
  Ethernet padding.
* TCP: `lib10::parse_tcp`; `tcp_checksum_ok` = `lib10::transport_checksum(...)` over the whole segment is 0; the MSS
  comes from `lib10b::find_mss` on the option bytes.

```
Ethernet  3c:22:fb:12:ab:9e -> f0:9f:c2:10:20:30  type 0x0800
IPv4      192.168.1.10 -> 203.0.113.80  ttl 64  proto 6  hdr 20  total 44  checksum OK
TCP       51514 -> 80  seq 2587568065  flags SYN  hdr 24  mss 1460  checksum OK
padding   2 bytes
```

In [139]:
namespace ex10_2 {
struct Dissection {
    std::string eth_src, eth_dst;
    uint16_t ethertype = 0;
    std::string ip_src, ip_dst;
    int ttl = 0, protocol = 0;
    size_t ip_header_len = 0, ip_total_len = 0;
    bool ip_checksum_ok = false;
    uint16_t src_port = 0, dst_port = 0;
    uint32_t seq = 0;
    std::string flags;
    size_t tcp_header_len = 0;
    int mss = -1;
    bool tcp_checksum_ok = false;
    size_t eth_padding = 0;
};
std::optional<Dissection> dissect(const std::vector<uint8_t>& f) {
    Dissection d;
    // ---- L2: Ethernet ----
    auto eth = lib10::parse_eth(f.data(), f.size());
    if (!eth || eth->ethertype != 0x0800) return std::nullopt;
    d.eth_src = lib10::format_mac(eth->src.data());
    d.eth_dst = lib10::format_mac(eth->dst.data());
    d.ethertype = eth->ethertype;

    // ---- L3: IPv4 ----
    const uint8_t* ipp = f.data() + 14;
    size_t ip_avail = f.size() - 14;
    auto ip = lib10::parse_ipv4_header(ipp, ip_avail);
    if (!ip || ip->protocol != 6) return std::nullopt;
    d.ip_src = lib10::format_ipv4(ip->src);
    d.ip_dst = lib10::format_ipv4(ip->dst);
    d.ttl = ip->ttl;
    d.protocol = ip->protocol;
    d.ip_header_len = ip->header_len();
    d.ip_total_len = ip->total_len;
    d.ip_checksum_ok = lib10::internet_checksum(ipp, d.ip_header_len) == 0;
    d.eth_padding = ip_avail - d.ip_total_len;          // bytes beyond the IP packet

    // ---- L4: TCP ----
    const uint8_t* tp = ipp + d.ip_header_len;
    size_t seg_len = d.ip_total_len - d.ip_header_len;  // trust IP's length, not the frame size
    auto tcp = lib10::parse_tcp(tp, seg_len);
    if (!tcp) return std::nullopt;
    d.src_port = tcp->src_port;
    d.dst_port = tcp->dst_port;
    d.seq = tcp->seq;
    d.flags = lib10::flags_to_string(tcp->flags);
    d.tcp_header_len = tcp->header_len();
    d.mss = lib10b::find_mss(tp + 20, d.tcp_header_len - 20);
    d.tcp_checksum_ok = lib10::transport_checksum(ip->src, ip->dst, 6, tp, seg_len) == 0;

    // ---- print a summary ----
    std::cout << "Ethernet  " << d.eth_src << " -> " << d.eth_dst << "  type 0x" << std::hex
              << std::setw(4) << std::setfill('0') << d.ethertype << std::dec << std::setfill(' ') << "\n"
              << "IPv4      " << d.ip_src << " -> " << d.ip_dst << "  ttl " << d.ttl << "  proto "
              << d.protocol << "  hdr " << d.ip_header_len << "  total " << d.ip_total_len
              << "  checksum " << (d.ip_checksum_ok ? "OK" : "BAD") << "\n"
              << "TCP       " << d.src_port << " -> " << d.dst_port << "  seq " << d.seq << "  flags "
              << d.flags << "  hdr " << d.tcp_header_len << "  mss " << d.mss
              << "  checksum " << (d.tcp_checksum_ok ? "OK" : "BAD") << "\n"
              << "padding   " << d.eth_padding << " bytes\n";
    return d;
}
}

**Test for Exercise 10.2** — run the next cell.

In [140]:
{
    auto r = ex10_2::dissect(capstone::syn_frame);
    CHECK(r.has_value());
    auto d = r.value_or(ex10_2::Dissection{});
    CHECK_EQ(d.eth_src, "3c:22:fb:12:ab:9e");
    CHECK_EQ(d.eth_dst, "f0:9f:c2:10:20:30");
    CHECK_EQ(d.ethertype, 0x0800);
    CHECK_EQ(d.ip_src, "192.168.1.10");
    CHECK_EQ(d.ip_dst, "203.0.113.80");
    CHECK_EQ(d.ttl, 64);
    CHECK_EQ(d.protocol, 6);
    CHECK_EQ(d.ip_header_len, 20u);
    CHECK_EQ(d.ip_total_len, 44u);
    CHECK(d.ip_checksum_ok);
    CHECK_EQ(d.src_port, 51514);
    CHECK_EQ(d.dst_port, 80);
    CHECK_EQ(d.seq, 2587568065u);
    CHECK_EQ(d.flags, "SYN");
    CHECK_EQ(d.tcp_header_len, 24u);
    CHECK_EQ(d.mss, 1460);
    CHECK(d.tcp_checksum_ok);
    CHECK_EQ(d.eth_padding, 2u);

    std::cout << "\n-- the same frame with one TTL bit flipped in transit --\n";
    auto bad = capstone::syn_frame;
    bad[14 + 8] ^= 0x01;
    auto rb = ex10_2::dissect(bad).value_or(ex10_2::Dissection{});
    CHECK(!rb.ip_checksum_ok);
    CHECK(rb.tcp_checksum_ok);     // the TCP checksum does not cover TTL

    std::cout << "\n-- the same frame with the destination port corrupted --\n";
    auto bad2 = capstone::syn_frame;
    bad2[14 + 20 + 3] ^= 0x10;
    auto rb2 = ex10_2::dissect(bad2).value_or(ex10_2::Dissection{});
    CHECK(rb2.ip_checksum_ok);
    CHECK(!rb2.tcp_checksum_ok);
}

Ethernet  3c:22:fb:12:ab:9e -> f0:9f:c2:10:20:30  type 0x0800
IPv4      192.168.1.10 -> 203.0.113.80  ttl 64  proto 6  hdr 20  total 44  checksum OK
TCP       51514 -> 80  seq 2587568065  flags SYN  hdr 24  mss 1460  checksum OK
padding   2 bytes
✅ r.has_value()
✅ d.eth_src == "3c:22:fb:12:ab:9e"
✅ d.eth_dst == "f0:9f:c2:10:20:30"
✅ d.ethertype == 0x0800
✅ d.ip_src == "192.168.1.10"
✅ d.ip_dst == "203.0.113.80"
✅ d.ttl == 64
✅ d.protocol == 6
✅ d.ip_header_len == 20u
✅ d.ip_total_len == 44u
✅ d.ip_checksum_ok
✅ d.src_port == 51514
✅ d.dst_port == 80
✅ d.seq == 2587568065u
✅ d.flags == "SYN"
✅ d.tcp_header_len == 24u
✅ d.mss == 1460
✅ d.tcp_checksum_ok
✅ d.eth_padding == 2u

-- the same frame with one TTL bit flipped in transit --
Ethernet  3c:22:fb:12:ab:9e -> f0:9f:c2:10:20:30  type 0x0800
IPv4      192.168.1.10 -> 203.0.113.80  ttl 65  proto 6  hdr 20  total 44  checksum BAD
TCP       51514 -> 80  seq 2587568065  flags SYN  hdr 24  mss 1460  checksum OK
padding   2 bytes
✅ !rb.ip_che

<details>
<summary><b>Solution</b> (click to expand)</summary>

The key design point is **which length to trust at each layer**. Ethernet delivered 60 bytes, but IP's total length
(44) defines the packet, so the 2 trailing bytes are padding, and the TCP segment length comes from IP, not from the
frame. The corruption tests show each checksum's coverage: TTL is covered by the IP header checksum only; ports are
covered by the TCP checksum only. On a real link the Ethernet CRC would have caught both, but a router's memory
corruption after the CRC check would not have been caught by it.

```cpp
namespace ex10_2 {
struct Dissection {
    std::string eth_src, eth_dst;
    uint16_t ethertype = 0;
    std::string ip_src, ip_dst;
    int ttl = 0, protocol = 0;
    size_t ip_header_len = 0, ip_total_len = 0;
    bool ip_checksum_ok = false;
    uint16_t src_port = 0, dst_port = 0;
    uint32_t seq = 0;
    std::string flags;
    size_t tcp_header_len = 0;
    int mss = -1;
    bool tcp_checksum_ok = false;
    size_t eth_padding = 0;
};
std::optional<Dissection> dissect(const std::vector<uint8_t>& f) {
    Dissection d;
    // ---- L2: Ethernet ----
    auto eth = lib10::parse_eth(f.data(), f.size());
    if (!eth || eth->ethertype != 0x0800) return std::nullopt;
    d.eth_src = lib10::format_mac(eth->src.data());
    d.eth_dst = lib10::format_mac(eth->dst.data());
    d.ethertype = eth->ethertype;

    // ---- L3: IPv4 ----
    const uint8_t* ipp = f.data() + 14;
    size_t ip_avail = f.size() - 14;
    auto ip = lib10::parse_ipv4_header(ipp, ip_avail);
    if (!ip || ip->protocol != 6) return std::nullopt;
    d.ip_src = lib10::format_ipv4(ip->src);
    d.ip_dst = lib10::format_ipv4(ip->dst);
    d.ttl = ip->ttl;
    d.protocol = ip->protocol;
    d.ip_header_len = ip->header_len();
    d.ip_total_len = ip->total_len;
    d.ip_checksum_ok = lib10::internet_checksum(ipp, d.ip_header_len) == 0;
    d.eth_padding = ip_avail - d.ip_total_len;          // bytes beyond the IP packet

    // ---- L4: TCP ----
    const uint8_t* tp = ipp + d.ip_header_len;
    size_t seg_len = d.ip_total_len - d.ip_header_len;  // trust IP's length, not the frame size
    auto tcp = lib10::parse_tcp(tp, seg_len);
    if (!tcp) return std::nullopt;
    d.src_port = tcp->src_port;
    d.dst_port = tcp->dst_port;
    d.seq = tcp->seq;
    d.flags = lib10::flags_to_string(tcp->flags);
    d.tcp_header_len = tcp->header_len();
    d.mss = lib10b::find_mss(tp + 20, d.tcp_header_len - 20);
    d.tcp_checksum_ok = lib10::transport_checksum(ip->src, ip->dst, 6, tp, seg_len) == 0;

    // ---- print a summary ----
    std::cout << "Ethernet  " << d.eth_src << " -> " << d.eth_dst << "  type 0x" << std::hex
              << std::setw(4) << std::setfill('0') << d.ethertype << std::dec << std::setfill(' ') << "\n"
              << "IPv4      " << d.ip_src << " -> " << d.ip_dst << "  ttl " << d.ttl << "  proto "
              << d.protocol << "  hdr " << d.ip_header_len << "  total " << d.ip_total_len
              << "  checksum " << (d.ip_checksum_ok ? "OK" : "BAD") << "\n"
              << "TCP       " << d.src_port << " -> " << d.dst_port << "  seq " << d.seq << "  flags "
              << d.flags << "  hdr " << d.tcp_header_len << "  mss " << d.mss
              << "  checksum " << (d.tcp_checksum_ok ? "OK" : "BAD") << "\n"
              << "padding   " << d.eth_padding << " bytes\n";
    return d;
}
}
```

</details>

In [141]:
namespace lib10c {

// PROVIDED (you wrote this in §10.2)
struct Dissection {
    std::string eth_src, eth_dst;
    uint16_t ethertype = 0;
    std::string ip_src, ip_dst;
    int ttl = 0, protocol = 0;
    size_t ip_header_len = 0, ip_total_len = 0;
    bool ip_checksum_ok = false;
    uint16_t src_port = 0, dst_port = 0;
    uint32_t seq = 0;
    std::string flags;
    size_t tcp_header_len = 0;
    int mss = -1;
    bool tcp_checksum_ok = false;
    size_t eth_padding = 0;
};
std::optional<Dissection> dissect(const std::vector<uint8_t>& f) {
    Dissection d;
    // ---- L2: Ethernet ----
    auto eth = lib10::parse_eth(f.data(), f.size());
    if (!eth || eth->ethertype != 0x0800) return std::nullopt;
    d.eth_src = lib10::format_mac(eth->src.data());
    d.eth_dst = lib10::format_mac(eth->dst.data());
    d.ethertype = eth->ethertype;

    // ---- L3: IPv4 ----
    const uint8_t* ipp = f.data() + 14;
    size_t ip_avail = f.size() - 14;
    auto ip = lib10::parse_ipv4_header(ipp, ip_avail);
    if (!ip || ip->protocol != 6) return std::nullopt;
    d.ip_src = lib10::format_ipv4(ip->src);
    d.ip_dst = lib10::format_ipv4(ip->dst);
    d.ttl = ip->ttl;
    d.protocol = ip->protocol;
    d.ip_header_len = ip->header_len();
    d.ip_total_len = ip->total_len;
    d.ip_checksum_ok = lib10::internet_checksum(ipp, d.ip_header_len) == 0;
    d.eth_padding = ip_avail - d.ip_total_len;          // bytes beyond the IP packet

    // ---- L4: TCP ----
    const uint8_t* tp = ipp + d.ip_header_len;
    size_t seg_len = d.ip_total_len - d.ip_header_len;  // trust IP's length, not the frame size
    auto tcp = lib10::parse_tcp(tp, seg_len);
    if (!tcp) return std::nullopt;
    d.src_port = tcp->src_port;
    d.dst_port = tcp->dst_port;
    d.seq = tcp->seq;
    d.flags = lib10::flags_to_string(tcp->flags);
    d.tcp_header_len = tcp->header_len();
    d.mss = lib10b::find_mss(tp + 20, d.tcp_header_len - 20);
    d.tcp_checksum_ok = lib10::transport_checksum(ip->src, ip->dst, 6, tp, seg_len) == 0;

    // ---- print a summary ----
    std::cout << "Ethernet  " << d.eth_src << " -> " << d.eth_dst << "  type 0x" << std::hex
              << std::setw(4) << std::setfill('0') << d.ethertype << std::dec << std::setfill(' ') << "\n"
              << "IPv4      " << d.ip_src << " -> " << d.ip_dst << "  ttl " << d.ttl << "  proto "
              << d.protocol << "  hdr " << d.ip_header_len << "  total " << d.ip_total_len
              << "  checksum " << (d.ip_checksum_ok ? "OK" : "BAD") << "\n"
              << "TCP       " << d.src_port << " -> " << d.dst_port << "  seq " << d.seq << "  flags "
              << d.flags << "  hdr " << d.tcp_header_len << "  mss " << d.mss
              << "  checksum " << (d.tcp_checksum_ok ? "OK" : "BAD") << "\n"
              << "padding   " << d.eth_padding << " bytes\n";
    return d;
}

} // namespace lib10c

### ✍️ Exercise 10.3 — Build the server's SYN|ACK reply, byte by byte

Now write bytes instead of reading them. `build_syn_ack(syn, server_isn)` returns the reply frame:

* **Ethernet**: swap the MAC addresses; EtherType 0x0800.
* **IPv4**: `0x45`, TOS 0, total length 44, identification 0, flags DF (bytes 6–7 = `0x4000`), TTL 64, protocol 6,
  checksum computed last, source/destination = the SYN's destination/source.
* **TCP**: ports swapped; seq = `server_isn`; ack = SYN's seq + 1; data offset 6; flags SYN|ACK (0x12);
  window 65160; urgent 0; options = MSS 1460 (`02 04 05 b4`); checksum computed over the pseudo-header + segment.
* Pad the frame with zeros to 60 bytes.

Fill every checksum field with 0 before computing it. For `server_isn = 0x1F2E3D4C` the correct frame is:
```
0000  3c 22 fb 12 ab 9e f0 9f  c2 10 20 30 08 00 45 00  |<"........ 0..E.|
0010  00 2c 00 00 40 00 40 06  3c c9 cb 00 71 50 c0 a8  |.,..@.@.<...qP..|
0020  01 0a 00 50 c9 3a 1f 2e  3d 4c 9a 3b 27 c2 60 12  |...P.:..=L.;'.`.|
0030  fe 88 b3 87 00 00 02 04  05 b4 00 00              |............|
```

In [142]:
namespace ex10_3 {
std::vector<uint8_t> build_syn_ack(const std::vector<uint8_t>& syn, uint32_t server_isn) {
    if (syn.size() < 14 + 20 + 20) return {};
    std::vector<uint8_t> f(14 + 20 + 24, 0);                   // Ethernet + IPv4 + TCP with MSS
    // ---- Ethernet: swap addresses ----
    std::memcpy(&f[0], &syn[6], 6);                             // dst = SYN's source
    std::memcpy(&f[6], &syn[0], 6);                             // src = SYN's destination
    lib10::write_be16(&f[12], 0x0800);

    // ---- IPv4 ----
    const uint8_t* sip = &syn[14];
    size_t syn_ihl = size_t(sip[0] & 0x0F) * 4;
    uint8_t* ip = &f[14];
    ip[0] = 0x45;                                               // version 4, IHL 5
    lib10::write_be16(ip + 2, 20 + 24);                         // total length
    lib10::write_be16(ip + 6, 0x4000);                          // DF
    ip[8] = 64;                                                 // TTL
    ip[9] = 6;                                                  // TCP
    std::memcpy(ip + 12, sip + 16, 4);                          // src = SYN's dst
    std::memcpy(ip + 16, sip + 12, 4);                          // dst = SYN's src
    lib10::write_be16(ip + 10, lib10::internet_checksum(ip, 20));

    // ---- TCP ----
    const uint8_t* st = sip + syn_ihl;
    uint8_t* t = ip + 20;
    lib10::write_be16(t + 0, lib10::read_be16(st + 2));         // src port = SYN's dst port
    lib10::write_be16(t + 2, lib10::read_be16(st + 0));         // dst port = SYN's src port
    lib10::write_be32(t + 4, server_isn);
    lib10::write_be32(t + 8, lib10::read_be32(st + 4) + 1);     // ack = client ISN + 1
    t[12] = 6 << 4;                                             // data offset 6
    t[13] = 0x12;                                               // SYN|ACK
    lib10::write_be16(t + 14, 65160);                           // window
    const uint8_t mss[4] = {0x02, 0x04, 0x05, 0xb4};
    std::memcpy(t + 20, mss, 4);
    uint32_t src = lib10::read_be32(ip + 12), dst = lib10::read_be32(ip + 16);
    lib10::write_be16(t + 16, lib10::transport_checksum(src, dst, 6, t, 24));

    f.resize(60, 0);                                            // Ethernet minimum (without FCS)
    return f;
}
}

**Test for Exercise 10.3** — run the next cell.

In [143]:
{
    auto reply = ex10_3::build_syn_ack(capstone::syn_frame, 0x1F2E3D4Cu);
    hexdump(reply);
    CHECK_EQ(reply, (std::vector<uint8_t>{
        0x3c, 0x22, 0xfb, 0x12, 0xab, 0x9e, 0xf0, 0x9f, 0xc2, 0x10, 0x20, 0x30,
        0x08, 0x00, 0x45, 0x00, 0x00, 0x2c, 0x00, 0x00, 0x40, 0x00, 0x40, 0x06,
        0x3c, 0xc9, 0xcb, 0x00, 0x71, 0x50, 0xc0, 0xa8, 0x01, 0x0a, 0x00, 0x50,
        0xc9, 0x3a, 0x1f, 0x2e, 0x3d, 0x4c, 0x9a, 0x3b, 0x27, 0xc2, 0x60, 0x12,
        0xfe, 0x88, 0xb3, 0x87, 0x00, 0x00, 0x02, 0x04, 0x05, 0xb4, 0x00, 0x00,
    }));
    std::cout << "\n-- dissecting your reply --\n";
    auto d = lib10c::dissect(reply).value_or(lib10c::Dissection{});
    CHECK(d.ip_checksum_ok && d.tcp_checksum_ok);
    CHECK_EQ(d.flags, "SYN|ACK");
}

0000  3c 22 fb 12 ab 9e f0 9f  c2 10 20 30 08 00 45 00  |<"........ 0..E.|
0010  00 2c 00 00 40 00 40 06  3c c9 cb 00 71 50 c0 a8  |.,..@.@.<...qP..|
0020  01 0a 00 50 c9 3a 1f 2e  3d 4c 9a 3b 27 c2 60 12  |...P.:..=L.;'.`.|
0030  fe 88 b3 87 00 00 02 04  05 b4 00 00              |............|
✅ reply == (std::vector<uint8_t>{ 0x3c, 0x22, 0xfb, 0x12, 0xab, 0x9e, 0xf0, 0x9f, 0xc2, 0x10, 0x20, 0x30, 0x08, 0x00, 0x45, 0x00, 0x00, 0x2c, 0x00, 0x00, 0x40, 0x00, 0x40, 0x06, 0x3c, 0xc9, 0xcb, 0x00, 0x71, 0x50, 0xc0, 0xa8, 0x01, 0x0a, 0x00, 0x50, 0xc9, 0x3a, 0x1f, 0x2e, 0x3d, 0x4c, 0x9a, 0x3b, 0x27, 0xc2, 0x60, 0x12, 0xfe, 0x88, 0xb3, 0x87, 0x00, 0x00, 0x02, 0x04, 0x05, 0xb4, 0x00, 0x00, })

-- dissecting your reply --
Ethernet  f0:9f:c2:10:20:30 -> 3c:22:fb:12:ab:9e  type 0x0800
IPv4      203.0.113.80 -> 192.168.1.10  ttl 64  proto 6  hdr 20  total 44  checksum OK
TCP       80 -> 51514  seq 523124044  flags SYN|ACK  hdr 24  mss 1460  checksum OK
padding   2 bytes
✅ d.ip_checksum_ok && d.tcp_

<details>
<summary><b>Solution</b> (click to expand)</summary>

Building a packet is parsing in reverse, with one ordering constraint: **checksums last**, each computed while its
own field is still zero. The TCP checksum depends on the IP addresses (pseudo-header), but the IP checksum does not
depend on TCP, so either header may be finished first. Kernels do exactly this, and most NICs can compute the
checksums in hardware ("checksum offload"). That is why captures taken on the sending host sometimes show
"incorrect" checksums.

```cpp
namespace ex10_3 {
std::vector<uint8_t> build_syn_ack(const std::vector<uint8_t>& syn, uint32_t server_isn) {
    if (syn.size() < 14 + 20 + 20) return {};
    std::vector<uint8_t> f(14 + 20 + 24, 0);                   // Ethernet + IPv4 + TCP with MSS
    // ---- Ethernet: swap addresses ----
    std::memcpy(&f[0], &syn[6], 6);                             // dst = SYN's source
    std::memcpy(&f[6], &syn[0], 6);                             // src = SYN's destination
    lib10::write_be16(&f[12], 0x0800);

    // ---- IPv4 ----
    const uint8_t* sip = &syn[14];
    size_t syn_ihl = size_t(sip[0] & 0x0F) * 4;
    uint8_t* ip = &f[14];
    ip[0] = 0x45;                                               // version 4, IHL 5
    lib10::write_be16(ip + 2, 20 + 24);                         // total length
    lib10::write_be16(ip + 6, 0x4000);                          // DF
    ip[8] = 64;                                                 // TTL
    ip[9] = 6;                                                  // TCP
    std::memcpy(ip + 12, sip + 16, 4);                          // src = SYN's dst
    std::memcpy(ip + 16, sip + 12, 4);                          // dst = SYN's src
    lib10::write_be16(ip + 10, lib10::internet_checksum(ip, 20));

    // ---- TCP ----
    const uint8_t* st = sip + syn_ihl;
    uint8_t* t = ip + 20;
    lib10::write_be16(t + 0, lib10::read_be16(st + 2));         // src port = SYN's dst port
    lib10::write_be16(t + 2, lib10::read_be16(st + 0));         // dst port = SYN's src port
    lib10::write_be32(t + 4, server_isn);
    lib10::write_be32(t + 8, lib10::read_be32(st + 4) + 1);     // ack = client ISN + 1
    t[12] = 6 << 4;                                             // data offset 6
    t[13] = 0x12;                                               // SYN|ACK
    lib10::write_be16(t + 14, 65160);                           // window
    const uint8_t mss[4] = {0x02, 0x04, 0x05, 0xb4};
    std::memcpy(t + 20, mss, 4);
    uint32_t src = lib10::read_be32(ip + 12), dst = lib10::read_be32(ip + 16);
    lib10::write_be16(t + 16, lib10::transport_checksum(src, dst, 6, t, 24));

    f.resize(60, 0);                                            // Ethernet minimum (without FCS)
    return f;
}
}
```

</details>

## 10.4 The whole journey

You can now follow a request through every layer and every device.

**Q10.1** *(Concept)* — Trace one HTTP request (`GET /` from a laptop on home Wi-Fi to a web server on the Internet) **down** the 7
layers on the laptop, **across** the network, and **up** the 7 layers on the server. At each step, name the device(s)
that touch that layer.

<details>
<summary><b>Solution</b> (click to expand)</summary>

**Down, on the laptop**
* **L7 Application**: the browser forms `GET / HTTP/1.1` + headers.
* **L6 Presentation**: text encoding (ASCII/UTF-8) and optional compression, done inside the browser.
* **L5 Session**: connection reuse (keep-alive) and cookies, managed by the browser's HTTP library.
* **L4 Transport**: the kernel's TCP puts the bytes into segments (ports 51514 → 80, seq/ack, checksum with
  pseudo-header), after the 3-way handshake.
* **L3 Network**: the kernel's IP adds src 192.168.1.10 and the server's IP as dst, TTL 64, header checksum;
  longest-prefix match picks the default route → next hop = gateway.
* **L2 Data link**: ARP cache → gateway MAC; the NIC driver/NIC builds the 802.11/Ethernet frame (dst = gateway
  MAC) and appends the CRC.
* **L1 Physical**: the NIC's PHY transmits radio symbols.

**Across the network**
* **AP** (L1–L2): radio → Ethernet bridging. **Switch** in the home box (L1–L2): forwards by dst MAC to the router port.
* **Home router** (L1–L3, and L4 for NAT): strips L2, LPM → ISP default route, TTL−1, checksum update,
  **NAT rewrites the source IP and port** (and patches the TCP checksum), then a new L2 header for the WAN link.
* **Modem/ONT** (L1–L2): modulates the frame onto DSL/coax/fibre.
* **ISP and Internet routers** (L1–L3): at each hop, CRC check, LPM, TTL−1, checksum update, new L2 header.
* **Switches in the server's data centre** (L1–L2).

**Up, on the server**
* **L1/L2**: the NIC checks the CRC, filters by MAC, and DMAs the frame into the RX ring.
* **L3**: the kernel checks the IP checksum and that the destination is its own address; protocol 6 → TCP.
* **L4**: TCP verifies the checksum, demultiplexes by 5-tuple to the connected socket, ACKs, and delivers bytes in order.
* **L5–L7**: the web server reads from the socket, frames the HTTP request (by `\r\n\r\n` and `Content-Length`),
  parses it, and produces the response, which travels the same path in reverse (NAT translating the destination
  back to the laptop).

*Key insight:* L2 headers are rebuilt at **every** hop, L3 addresses stay end-to-end (except where NAT rewrites
them), and L4 and above are touched only by the two endpoints (and NAT).

</details>

## 10.5 Final quiz

Ten quick questions spanning the whole notebook. Answer each before expanding its solution.

**Q10.2** *(Concept)* — A switch forwards what PDU, based on which header field?

<details>
<summary><b>Solution</b> (click to expand)</summary>

A **frame** (L2), based on the **destination MAC address** (and it learns from the source MAC).

</details>

**Q10.3** *(Compute)* — How many usable host addresses does a /26 have?

<details>
<summary><b>Solution</b> (click to expand)</summary>

$2^{32-26} - 2$ = **62**: the network and broadcast addresses are not usable.

</details>

**Q10.4** *(Predict the output)* — On x86, what does `std::cout << htons(0x0050);` print (decimal)?

<details>
<summary><b>Solution</b> (click to expand)</summary>

**20480** (0x5000): the two bytes are swapped on a little-endian host. 0x0050 is port 80.

</details>

**Q10.5** *(Concept)* — List everything a router changes in an IPv4 packet it forwards (no NAT).

<details>
<summary><b>Solution</b> (click to expand)</summary>

**TTL** (−1) and therefore the **header checksum**, plus a completely **new L2 header** (source = its own
               outgoing MAC, destination = next hop's MAC, new CRC). IP addresses, ports and payload are unchanged (it might
               fragment if DF is clear and the packet exceeds the MTU).

</details>

**Q10.6** *(Compute)* — Host `10.0.5.20/22` sends to `10.0.6.1`. Direct delivery or via the gateway?

<details>
<summary><b>Solution</b> (click to expand)</summary>

The /22 network is **10.0.4.0–10.0.7.255**, so 10.0.6.1 is
               **inside it: direct delivery**. The host ARPs for 10.0.6.1 itself.

</details>

**Q10.7** *(Compute)* — How long does it take to transmit a 1500-byte frame onto a 100 Mb/s link?

<details>
<summary><b>Solution</b> (click to expand)</summary>

$L/R = 12000 / 10^8$ = **120 µs**. That is the transmission delay; propagation delay is additional.

</details>

**Q10.8** *(Concept)* — Does a router forward an ARP request? Why or why not?

<details>
<summary><b>Solution</b> (click to expand)</summary>

**No.** ARP requests are L2 broadcasts, and broadcasts stay within the LAN; a router works at L3 and does not
               forward L2 broadcasts. ARP only ever resolves addresses on the local link.

</details>

**Q10.9** *(Concept)* — Which TCP window travels in the header, and what does it protect?

<details>
<summary><b>Solution</b> (click to expand)</summary>

**rwnd**, the receive window (flow control). It protects the **receiver's buffer**. cwnd (congestion control)
               is private to the sender and protects the network.

</details>

**Q10.10** *(Concept)* — After a TCP connection closes normally, which side sits in `TIME_WAIT`, and for how long?

<details>
<summary><b>Solution</b> (click to expand)</summary>

The side that sent the **first FIN** (the active closer), for **2 × MSL** (60 s on Linux), so that it can re-ACK a
               retransmitted FIN and so that stale segments die before the 5-tuple is reused.

</details>

**Q10.11** *(Concept)* — `connect()` fails with `ECONNREFUSED`. What arrived from the network?

<details>
<summary><b>Solution</b> (click to expand)</summary>

A TCP **RST** in reply to the SYN: the host is reachable but nothing listens on that port. (A silent drop would
               have given `ETIMEDOUT` instead.)

</details>

## Final score

Run the next cell to see how many checks you have passed. (Re-running a test cell counts its checks again;
restart the kernel and *Run All* for a clean count.)

In [144]:
print_score();

Score so far: 319 passed, 0 failed (re-running a test cell counts its CHECKs again)


### 🔁 Recap

- A real frame is parsed layer by layer, each layer's length field bounding the next.
- Each checksum covers a specific range: the IP header only; TCP = pseudo-header + segment; Ethernet CRC = whole frame.
- TLV option walkers must check both 'makes progress' and 'stays in bounds'.
- Building packets is parsing in reverse, with checksums computed last over zeroed fields.
- L2 is rebuilt per hop, L3 is end-to-end (except NAT), and L4+ belongs to the endpoints.

**You should now be able to explain…**

- [ ] every byte of an Ethernet + IPv4 + TCP SYN frame;
- [ ] which checksum catches which corruption, and why padding is not part of the IP packet;
- [ ] how a server constructs its SYN|ACK from the client's SYN;
- [ ] the full path of an HTTP request through all layers and all devices.